In [1]:
!pip install "bayesian-optimization==3.3.0"
!pip install "hyperopt==0.3.0"
!pip install "ConfigSpace==1.2.2"
!pip install "smac==2.4.0"

import contextlib
import logging
import sys
import time
import numpy as np
import matplotlib.pyplot as plt
from bayes_opt import BayesianOptimization
from bayes_opt.acquisition import ExpectedImprovement
from hyperopt import fmin, tpe, hp, STATUS_OK, Trials
from ConfigSpace import ConfigurationSpace, Float

from smac import HyperparameterOptimizationFacade, Scenario
from smac.callback.callback import Callback

from pathlib import Path
from scipy import stats
from statsmodels.stats.multitest import multipletests
import itertools
import re
import pandas as pd
from IPython.display import FileLink, display
from IPython.display import display
import importlib.metadata as importlib_metadata
from functools import partial
from pprint import pprint

from sklearn.gaussian_process.kernels import Matern

class _DuplicateNoticeOutputFilter:
    """Hide bayes_opt's colored duplicate notices while passing other output through."""

    def __init__(self, stream):
        self._stream = stream
        self._pending = ""

    @staticmethod
    def _is_duplicate_notice(line):
        return (
            "Data point " in line
            and " is not unique." in line
            and "duplicates registered. Continuing" in line
        )

    def write(self, text):
        self._pending += text
        while "\n" in self._pending:
            line, _, self._pending = self._pending.partition("\n")
            if not self._is_duplicate_notice(line):
                self._stream.write(line + "\n")
        return len(text)

    def flush(self):
        if self._pending:
            if not self._is_duplicate_notice(self._pending):
                self._stream.write(self._pending)
            self._pending = ""
        self._stream.flush()

    def __getattr__(self, name):
        return getattr(self._stream, name)


@contextlib.contextmanager
def _suppress_smac_info_logs():
    """Temporarily suppress SMAC info logs while preserving warnings and errors."""
    logger = logging.getLogger("smac")
    previous_level = logger.level
    logger.setLevel(logging.WARNING)
    try:
        yield
    finally:
        logger.setLevel(previous_level)


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [bayesian-optimization]
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Created wheel for pynisher: filename=pynisher-1.0.10-py3-none-any.whl size=28743 sha256=0ba0a7414a8ab4fee1fadbdced2af5ebe035799eb76085bc867deaf435ba735a
  Stored in directory: /root/.cache/pip/wheels/61/19/ad/5b4920e998a0724457a64fabf5d7af20948274ad6c597abb51
Successfully built pynisher
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4/4 [smac]3/4 [smac]jobqueue]


In [2]:
# Here are many different families of patients
baseline_family = {
    'name': 'quick_rise_slow_fall',
    'num_optima': 2,
    'spread_scale': 1.0,
    'function_shape': 'quick_rise_slow_fall',
    'noise': 'normal',
    'unique_parameter_interaction': 'none'
}

symmetric_family = {
    'name': 'symmetric',
    'num_optima': 2,
    'spread_scale': 1.0,
    'function_shape': 'symmetric',
    'noise': 'normal',
    'unique_parameter_interaction': 'none'
}

slow_rise_quick_fall_family = {
    'name': 'slow_rise_quick_fall',
    'num_optima': 2,
    'spread_scale': 1.0,
    'function_shape': 'slow_rise_quick_fall',
    'noise': 'normal',
    'unique_parameter_interaction': 'none'
}

three_optima_family = {
    'name': '3optima',
    'num_optima': 3,
    'spread_scale': 1.0,
    'function_shape': 'quick_rise_slow_fall',
    'noise': 'normal',
    'unique_parameter_interaction': 'none'
}

broad_peaks_family = {
    'name': 'broad_optima',
    'num_optima': 2,
    'spread_scale': 2.0,
    'function_shape': 'quick_rise_slow_fall',
    'noise': 'normal',
    'unique_parameter_interaction': 'none'
}

sharp_peaks_family = {
    'name': 'sharp_optima',
    'num_optima': 2,
    'spread_scale': 0.4,
    'function_shape': 'quick_rise_slow_fall',
    'noise': 'normal',
    'unique_parameter_interaction': 'none'
}

heteroscedastic_noise_family = {
    'name': 'heteroscedastic_noise',
    'num_optima': 2,
    'spread_scale': 1.0,
    'function_shape': 'quick_rise_slow_fall',
    'noise': 'heteroscedastic',
    'unique_parameter_interaction': 'none'
}

heavy_tailed_noise_family = {
    'name': 'heavy_tailed_noise',
    'num_optima': 2,
    'spread_scale': 1.0,
    'function_shape': 'quick_rise_slow_fall',
    'noise': 'heavy_tailed',
    'unique_parameter_interaction': 'none'
}

unique_parameter_interaction_family = {
    'name': 'unique_parameter_interaction',
    'num_optima': 2,
    'spread_scale': 1.0,
    'function_shape': 'quick_rise_slow_fall',
    'noise': 'normal',
    'unique_parameter_interaction': 'pulsewidth-amplitude'
}


def asymmetric_hill_peak(
    value,
    optimum,
    left_scale,
    right_scale,
    hill_n=4.0,
):
    if left_scale <= 0 or right_scale <= 0 or hill_n <= 0:
        raise ValueError("Scales and hill_n must be positive.")

    value = np.asarray(value, dtype=float)
    scale = np.where(value < optimum, left_scale, right_scale)
    distance = np.abs(value - optimum) / scale

    return 1.0 / (1.0 + distance**hill_n)


def generate_patient_profile(patient_seed: int, family: dict):
    rng = np.random.default_rng(patient_seed)

    base_optima = [(10.0 + rng.uniform(-2.0, 2.0), 500.0 + rng.uniform(-25.0, 25.0)),
                   (25.0 + rng.uniform(-2.0, 2.0), 100.0 + rng.uniform(-25.0, 25.0))] #may need to change this later on.
    num_optima = family['num_optima']
    optima = []
    for numOfOptima in range(num_optima):
        if numOfOptima < len(base_optima):
            opt_f, opt_w = base_optima[numOfOptima]
        else:
            opt_f = rng.uniform(5.0, 50.0)
            opt_w = rng.uniform(150.0, 450.0)
        optima.append((opt_f, opt_w))

    spread_scale = family['spread_scale']

    if family['function_shape'] == 'quick_rise_slow_fall':
        return {
            'patient_seed': patient_seed,
            'surface_family': family['name'],
            'ideal_amp_pup': rng.normal(4.4, 0.15),
            'ideal_amp_hrv': rng.normal(4.0, 0.15),
            # Asymmetric spreads: faster rise toward threshold, slower decline afterwards
            'amp_rise_pup': rng.uniform(0.25, 0.35) * spread_scale,
            'amp_fall_pup': rng.uniform(0.55, 0.75) * spread_scale,
            'amp_rise_hrv': rng.uniform(0.20, 0.30) * spread_scale,
            'amp_fall_hrv': rng.uniform(0.45, 0.65) * spread_scale,

            'freq_rise': rng.uniform(1.0, 2.0) * spread_scale,
            'freq_fall': rng.uniform(5.0, 6.0) * spread_scale,
            'pulse_width_rise': rng.uniform(20.0, 50.0) * spread_scale,
            'pulse_width_fall': rng.uniform(125.0, 175.0) * spread_scale,

            'optima': optima,
        }
    elif family['function_shape'] == 'symmetric':
        return {
            'patient_seed': patient_seed,
            'surface_family': family['name'],
            'ideal_amp_pup': rng.normal(4.4, 0.15),
            'ideal_amp_hrv': rng.normal(4.0, 0.15),
            # Symmetric spreads: same rise and fall
            'amp_rise_pup': 0.25,
            'amp_fall_pup': 0.25,
            'amp_rise_hrv': 0.25,
            'amp_fall_hrv': 0.25,
            'freq_rise': 1.5,
            'freq_fall': 1.5,
            'pulse_width_rise': 75.0,
            'pulse_width_fall': 75.0,

            'optima': optima,
        }
    elif family['function_shape'] == 'slow_rise_quick_fall':
        return {
            'patient_seed': patient_seed,
            'surface_family': family['name'],
            'ideal_amp_pup': rng.normal(4.4, 0.15),
            'ideal_amp_hrv': rng.normal(4.0, 0.15),
            # Asymmetric spreads: slow rise toward threshold, faster fall afterwards
            'amp_rise_pup': rng.uniform(0.55, 0.75) * spread_scale,
            'amp_fall_pup': rng.uniform(0.25, 0.35) * spread_scale,
            'amp_rise_hrv': rng.uniform(0.45, 0.65) * spread_scale,
            'amp_fall_hrv': rng.uniform(0.20, 0.30) * spread_scale,

            'freq_rise': rng.uniform(5.0, 6.0) * spread_scale,
            'freq_fall': rng.uniform(1.0, 2.0) * spread_scale,
            'pulse_width_rise': rng.uniform(125.0, 175.0) * spread_scale,
            'pulse_width_fall': rng.uniform(20.0, 50.0) * spread_scale,

            'optima': optima,
        }


# Convergence is based on the parameters each method selects from noisy observations.
# The oracle curve is reported separately and is never returned to an optimizer.
INCUMBENT_PARAMETER_COLUMNS = {
    "amplitude": "Selected Incumbent Amplitude (mA)",
    "frequency": "Selected Incumbent Frequency (Hz)",
    "pulse_width": "Selected Incumbent Pulse Width (us)",
}
SELECTED_INCUMBENT_REGRET_COLUMN = "Selected Incumbent Normalized Regret (%)"
ORACLE_BEST_EVALUATED_REGRET_COLUMN = "Oracle Best-Evaluated Normalized Regret (%)"
INCUMBENT_POLICY_COLUMN = "Incumbent Selection Policy"
OBSERVED_INCUMBENT_POLICY = "highest observed score"
SMAC_INCUMBENT_POLICY = "SMAC intensifier"
SMAC_METHOD = "SMAC Optimization"


def _physical_parameters(params):
    return {name: float(params[name]) for name in INCUMBENT_PARAMETER_COLUMNS}


def _normalize_parameters(params, bounds):
    return {
        name: (float(params[name]) - lower) / (upper - lower)
        for name, (lower, upper) in bounds.items()
    }


def _denormalize_parameters(params, bounds):
    return {
        name: lower + float(params[name]) * (upper - lower)
        for name, (lower, upper) in bounds.items()
    }


def _update_observed_incumbent(best_score, best_params, score, params):
    # Strict comparison retains the first point on a tie, as these methods do.
    if score > best_score:
        return score, _physical_parameters(params)
    return best_score, best_params


def _set_incumbent_parameters(row, params):
    if params is None:
        raise RuntimeError("No incumbent was selected from the observed data.")
    for name, column in INCUMBENT_PARAMETER_COLUMNS.items():
        row[column] = float(params[name])


def _normalized_regret_percent(best_possible_score, score):
    if best_possible_score <= 0:
        return np.nan
    return 100 * (best_possible_score - score) / best_possible_score


class SMACIncumbentRecorder(Callback):
    """Record SMAC's own incumbent after its intensifier has processed a trial."""

    def __init__(self):
        super().__init__()
        self.records = []

    def on_tell_end(self, smbo, info, value):
        incumbent = smbo.intensifier.get_incumbent()
        if incumbent is None:
            raise RuntimeError("SMAC did not select an incumbent after a completed trial.")
        # Only observed costs and parameter values are handled by this callback.
        self.records.append({
            "trial_params": _physical_parameters(info.config),
            "incumbent_params": _physical_parameters(incumbent),
            "cost": float(value.cost),
        })


def _attach_smac_incumbents(rows, records, finished_trials):
    """Align each completed SMAC trial with its actual post-tell incumbent."""
    smac_rows = [row for row in rows if row["Method"] == SMAC_METHOD]
    if len(smac_rows) != len(records) or finished_trials != len(smac_rows):
        raise RuntimeError(
            "Cannot reconstruct the complete SMAC incumbent trajectory: trial, "
            "callback, and evaluation counts differ. Execute the optimizer "
            "configuration cell before running the comparison."
        )
    for query, (row, record) in enumerate(zip(smac_rows, records), start=1):
        row_params = (row["Amplitude"], row["Frequency"], row["Pulse Width"])
        trial_params = tuple(record["trial_params"][name] for name in INCUMBENT_PARAMETER_COLUMNS)
        if (
            row["Query"] != query
            or not np.allclose(row_params, trial_params, rtol=0, atol=1e-12)
            or not np.isclose(row["Observed Score"], -record["cost"], rtol=1e-12, atol=1e-12)
        ):
            raise RuntimeError("SMAC trial records do not match the recorded evaluations.")
        _set_incumbent_parameters(row, record["incumbent_params"])


def _finalize_convergence_metrics(rows, best_possible_score):
    """Calculate selected and oracle regrets from diagnostic scores, after selection."""
    evaluated_scores = {}
    for row in rows:
        evaluated_params = (row["Amplitude"], row["Frequency"], row["Pulse Width"])
        evaluated_scores[(row["Method"], evaluated_params)] = row["Noise-Free Score"]
        selected_params = tuple(row[column] for column in INCUMBENT_PARAMETER_COLUMNS.values())
        selected_key = (row["Method"], selected_params)
        if selected_key not in evaluated_scores:
            raise RuntimeError("A selected incumbent was not evaluated by that query.")
        incumbent_score = evaluated_scores[selected_key]
        row["Current Incumbent Noise-Free Score"] = incumbent_score
        selected_regret = _normalized_regret_percent(best_possible_score, incumbent_score)
        row[SELECTED_INCUMBENT_REGRET_COLUMN] = selected_regret
        # Backward-compatible name now explicitly aliases the selected-incumbent metric.
        row["Normalized Regret So Far (%)"] = selected_regret
        row[ORACLE_BEST_EVALUATED_REGRET_COLUMN] = _normalized_regret_percent(
            best_possible_score, row["Best Noise-Free Score So Far"]
        )


def _validate_final_convergence(results_df, convergence_df):
    """Require the final trajectory point to describe the reported recommendation."""
    final_rows = convergence_df.sort_values("Query").groupby("Method", sort=False).tail(1)
    final_rows = final_rows.set_index("Method", verify_integrity=True)
    for _, result in results_df.iterrows():
        method = result["Method"]
        if method not in final_rows.index:
            raise RuntimeError(f"No final convergence row for {method}.")
        final = final_rows.loc[method]
        for convergence_column, result_column in (
            ("Selected Incumbent Amplitude (mA)", "Selected Amplitude (mA)"),
            ("Selected Incumbent Frequency (Hz)", "Selected Frequency (Hz)"),
            ("Selected Incumbent Pulse Width (us)", "Selected Pulse Width (us)"),
            ("Current Incumbent Noise-Free Score", "Selected Score (Noise-Free)"),
            (SELECTED_INCUMBENT_REGRET_COLUMN, "Normalized Regret (%)"),
            (SELECTED_INCUMBENT_REGRET_COLUMN, SELECTED_INCUMBENT_REGRET_COLUMN),
            (ORACLE_BEST_EVALUATED_REGRET_COLUMN, ORACLE_BEST_EVALUATED_REGRET_COLUMN),
        ):
            if not np.isclose(final[convergence_column], result[result_column],
                              rtol=1e-12, atol=1e-12, equal_nan=True):
                raise RuntimeError(f"Final convergence does not match {result_column} for {method}.")


SIMULATED_GAUSSIAN_NOISE_STD = 2.0
SIMULATED_HEAVY_TAILED_DF = 3
SIMULATED_HEAVY_TAILED_SCALE = 2.0


def run_optimization_comparison(num_simulation_trials: int, amount_of_patients: int, patient_and_machine_startSeed : int, patient_type: dict):

    search_boundaries = {
        'amplitude': (0.5, 5.0),       # mA
        'frequency': (1.0, 50.0),      # Hz
        'pulse_width': (50.0, 600.0),  # microseconds
    }

    bo_normalized_bounds = {
        name: (0.0, 1.0) for name in search_boundaries
    }

    all_patients_results = [] 
    all_convergence_results = []

    for i in range(amount_of_patients):

        patient_profile = generate_patient_profile(patient_and_machine_startSeed, patient_type)
        ideal_amp_pup = patient_profile['ideal_amp_pup']
        amp_rise_pup = patient_profile['amp_rise_pup']
        amp_fall_pup = patient_profile['amp_fall_pup']
        ideal_amp_hrv = patient_profile['ideal_amp_hrv']
        amp_rise_hrv = patient_profile['amp_rise_hrv']
        amp_fall_hrv = patient_profile['amp_fall_hrv']
        freq_rise = patient_profile['freq_rise']
        freq_fall = patient_profile['freq_fall']
        pulse_width_rise = patient_profile['pulse_width_rise']
        pulse_width_fall = patient_profile['pulse_width_fall']
        optima = patient_profile['optima']

        def realistic_patient_curve(amplitude, frequency, pulse_width, AddNoise: bool, patient_and_machine_startSeed: int):

            amp_comp_pup = asymmetric_hill_peak(
                amplitude,
                ideal_amp_pup,
                amp_rise_pup,
                amp_fall_pup,
            )
            amp_comp_hrv = asymmetric_hill_peak(
                amplitude,
                ideal_amp_hrv,
                amp_rise_hrv,
                amp_fall_hrv,
            )

            pup_score = 100 * amp_comp_pup
            hrv_score = 100 * amp_comp_hrv
            w_pup, w_hrv = 0.5, 0.5
            amplitude_score = w_pup * pup_score + w_hrv * hrv_score

            def get_interdependence_score(f, w, Amplitude_Scale_Factor = 1.0):
                scores = []
                for (opt_f, opt_w) in optima:
                    freq_comp = asymmetric_hill_peak(
                        f,
                        opt_f,
                        freq_rise,
                        freq_fall,
                    )
                    
                    width_comp = asymmetric_hill_peak(
                        w,
                        opt_w,
                        pulse_width_rise,
                        pulse_width_fall,
                    )
                    dist = ((freq_comp) * (width_comp)) * Amplitude_Scale_Factor
                    scores.append(dist)

                return np.max(np.asarray(scores), axis=0)

            def get_interdependence_score_for_special_interaction(f, w, a):
                scores = []
                for (opt_f, opt_w) in optima:
                    if opt_f >= 25:
                        freq_comp = asymmetric_hill_peak(
                            f,
                            opt_f - a,
                            freq_rise,
                            freq_fall,
                        )
                    else:
                        freq_comp = asymmetric_hill_peak(
                            f,
                            opt_f + a,
                            freq_rise,
                            freq_fall,
                        )

                    if opt_w >= 350: 
                        width_comp = asymmetric_hill_peak(
                            w,
                            opt_w - a * 20,
                            pulse_width_rise,
                            pulse_width_fall,
                        )
                    else:
                        width_comp = asymmetric_hill_peak(
                            w,
                            opt_w + a * 20,
                            pulse_width_rise,
                            pulse_width_fall,
                        )
                    dist = ((freq_comp) * (width_comp)) 
                    scores.append(dist)
            
                return np.max(np.asarray(scores), axis=0)
            
            if patient_type.get('unique_parameter_interaction') == 'pulsewidth-amplitude':
                inter_score = get_interdependence_score_for_special_interaction(frequency, pulse_width, amplitude)
            else:
                inter_score = get_interdependence_score(frequency, pulse_width)

            total_score = inter_score * amplitude_score

            if not AddNoise:
                return total_score

            rng = np.random.default_rng(patient_and_machine_startSeed)
            noise_type = patient_type.get('noise', 'normal')
            if noise_type == 'normal':
                noise = rng.normal(0, SIMULATED_GAUSSIAN_NOISE_STD)
            elif noise_type == 'heteroscedastic':
                    # Noise scales with signal magnitude
                noise = rng.normal(0, amplitude)
            elif noise_type == 'heavy_tailed':
                    # Student's t-distribution
                noise = rng.standard_t(df=SIMULATED_HEAVY_TAILED_DF) * SIMULATED_HEAVY_TAILED_SCALE
            else:
                noise = 0.0

            total_score += noise

            return total_score

        #Find the optimal score and parameters
        if patient_type.get('unique_parameter_interaction') != 'pulsewidth-amplitude':
            amplitude_grid = np.linspace(search_boundaries['amplitude'][0], search_boundaries['amplitude'][1], 10000)
            max_possible_score = -np.inf
            for (opt_f, opt_w) in optima:
                # Vectorized evaluation over the amplitude grid at the peak frequency & pulse width
                curve_scores = realistic_patient_curve(amplitude_grid, opt_f, opt_w, False, 0)
                best_idx = np.argmax(curve_scores)
                peak_score = curve_scores[best_idx]

                if peak_score > max_possible_score:
                    max_possible_score = peak_score
        else: 
            amplitude_grid = np.linspace(
                search_boundaries['amplitude'][0],
                search_boundaries['amplitude'][1],
                10000,
            )
            max_possible_score = -np.inf

            for opt_f, opt_w in optima:
                if opt_f >= 25:
                    shifted_frequency = opt_f - amplitude_grid
                else:
                    shifted_frequency = opt_f + amplitude_grid

                if opt_w >= 350:
                    shifted_pulse_width = opt_w - 20 * amplitude_grid
                else:
                    shifted_pulse_width = opt_w + 20 * amplitude_grid

                curve_scores = realistic_patient_curve(
                    amplitude_grid,
                    shifted_frequency,
                    shifted_pulse_width,
                    False,
                    0,
                )
                peak_score = np.max(curve_scores)

                if peak_score > max_possible_score:
                    max_possible_score = peak_score
        

        method_names = (
            "Bayesian Optimization",
            "TPE Optimization",
            "SMAC Optimization",
            "Random Search",
        )

        def make_evaluation_seed(profile_seed, query_index):
            sequence = np.random.SeedSequence(
                [int(profile_seed), int(query_index)]
            )
            return int(sequence.generate_state(1, dtype=np.uint32)[0])

        profile_seed = patient_and_machine_startSeed
        convergence_rows = []

        query_counts = {
            method: 0
            for method in method_names
        }

        best_observed = {
            method: -np.inf
            for method in method_names
        }

        best_noise_free = {
            method: -np.inf
            for method in method_names
        }

        observed_incumbent_params = {
            method: None
            for method in method_names
        }

        def evaluate_and_record(method, params, evaluation_seed):
            query_index = query_counts[method] + 1
            query_counts[method] = query_index

            observed_score = realistic_patient_curve(
                params["amplitude"],
                params["frequency"],
                params["pulse_width"],
                True,
                evaluation_seed,
            )

            noise_free_score = realistic_patient_curve(
                params["amplitude"],
                params["frequency"],
                params["pulse_width"],
                False,
                evaluation_seed,
            )

            best_observed[method], observed_incumbent_params[method] = (
                _update_observed_incumbent(
                    best_observed[method],
                    observed_incumbent_params[method],
                    observed_score,
                    params,
                )
            )

            best_noise_free[method] = max(
                best_noise_free[method],
                noise_free_score,
            )

            row = {
                "Profile ID": (
                    f"{patient_profile['surface_family']}::{profile_seed}"
                ),
                "Surface Family": patient_profile["surface_family"],
                "Profile Seed": profile_seed,
                "Iteration Budget": num_simulation_trials,
                "Method": method,
                "Query": query_index,
                "Evaluation Seed": evaluation_seed,
                "Amplitude": params["amplitude"],
                "Frequency": params["frequency"],
                "Pulse Width": params["pulse_width"],
                "Observed Score": observed_score,
                "Noise-Free Score": noise_free_score,
                "Cumulative Best Observed Score": best_observed[method],
                "Best Noise-Free Score So Far": best_noise_free[method],
                INCUMBENT_POLICY_COLUMN: (
                    SMAC_INCUMBENT_POLICY if method == SMAC_METHOD
                    else OBSERVED_INCUMBENT_POLICY
                ),
            }
            if method != SMAC_METHOD:
                _set_incumbent_parameters(row, observed_incumbent_params[method])
            convergence_rows.append(row)

            # The optimizer receives the noisy observation only.
            return observed_score

        def objective_bo(amplitude, frequency, pulse_width):
            query_index = (
                query_counts["Bayesian Optimization"] + 1
            )
            physical_params = _denormalize_parameters(
                {
                    "amplitude": amplitude,
                    "frequency": frequency,
                    "pulse_width": pulse_width,
                },
                search_boundaries,
            )
            evaluation_seed = make_evaluation_seed(
                profile_seed,
                query_index,
            )

            return evaluate_and_record(
                "Bayesian Optimization",
                physical_params,
                evaluation_seed,
            )

        optimizer_multi = BayesianOptimization(
            f=objective_bo,
            pbounds=bo_normalized_bounds,
            gp_noise_type=patient_type.get('noise', 'normal'),
            gp_physical_bounds=search_boundaries,
            acquisition_function=ExpectedImprovement(xi=0.01),
            random_state=patient_and_machine_startSeed,
            verbose=0
        )
        print(f"\n--- Starting Bayesian Optimization ({num_simulation_trials} iterations) ---")
        bo_start_time = time.perf_counter()
        duplicate_notice_filter = _DuplicateNoticeOutputFilter(sys.stdout)
        try:
            with contextlib.redirect_stdout(duplicate_notice_filter):
                optimizer_multi.maximize(
                    init_points=BOGP_DEFAULTS["init_points"],
                    n_iter=num_simulation_trials - BOGP_DEFAULTS["init_points"],
                )
        finally:
            duplicate_notice_filter.flush()

        bo_selected_params = _denormalize_parameters(
            optimizer_multi.max["params"], search_boundaries
        )

        bo_elapsed_seconds = time.perf_counter() - bo_start_time
        duplicate_count = optimizer_multi.space.n_duplicate_points
        repeat_summary = f" ({duplicate_count} repeated evaluations)" if duplicate_count else ""
        print(
            f"Bayesian Optimization completed in {bo_elapsed_seconds:.3f} seconds"
            f"{repeat_summary}."
        )

        # --- TPE Optimization ---
        tpe_start_time = time.perf_counter()
        def objective_tpe(params):
            query_index = query_counts["TPE Optimization"] + 1
            evaluation_seed = make_evaluation_seed(
                profile_seed,
                query_index,
            )

            score = evaluate_and_record(
                "TPE Optimization",
                {
                    "amplitude": params["amplitude"],
                    "frequency": params["frequency"],
                    "pulse_width": params["pulse_width"],
                },
                evaluation_seed,
            )

            return {
                "loss": -score,
                "status": STATUS_OK,
            }
        space = {
            'amplitude': hp.uniform('amplitude', search_boundaries['amplitude'][0], search_boundaries['amplitude'][1]),
            'frequency': hp.uniform('frequency', search_boundaries['frequency'][0], search_boundaries['frequency'][1]),
            'pulse_width': hp.uniform('pulse_width', search_boundaries['pulse_width'][0], search_boundaries['pulse_width'][1])
        }
        trials = Trials()
        print(f"\n--- Starting TPE Optimization ({num_simulation_trials} iterations) ---")
        fmin(fn=objective_tpe, space=space, algo=tpe.suggest, max_evals=num_simulation_trials, trials=trials, rstate=np.random.default_rng(patient_and_machine_startSeed))
        tpe_elapsed_seconds = time.perf_counter() - tpe_start_time
        print(f"TPE completed in {tpe_elapsed_seconds:.3f} seconds.")

        # --- SMAC Optimization ---
        smac_start_time = time.perf_counter()
        configspace = ConfigurationSpace(seed=patient_and_machine_startSeed)
        configspace.add([
            Float('amplitude', bounds=search_boundaries['amplitude']),
            Float('frequency', bounds=search_boundaries['frequency']),
            Float('pulse_width', bounds=search_boundaries['pulse_width']),
        ])

        def objective_smac(config, seed=None):
            query_index = query_counts["SMAC Optimization"] + 1
            evaluation_seed = make_evaluation_seed(
                profile_seed,
                query_index,
            )
            score = evaluate_and_record(
                "SMAC Optimization",
                {
                    "amplitude": float(config["amplitude"]),
                    "frequency": float(config["frequency"]),
                    "pulse_width": float(config["pulse_width"]),
                },
                evaluation_seed,
            )

            return -score

        print(f"\n--- Starting SMAC Optimization ({num_simulation_trials} iterations) ---")
        with _suppress_smac_info_logs():
            scenario = Scenario(
                configspace,
                deterministic=SMAC_DEFAULTS["scenario"]["deterministic"],
                objectives=SMAC_DEFAULTS["scenario"]["objectives"],
                use_default_config=SMAC_DEFAULTS["scenario"]["use_default_config"],
                n_workers=SMAC_DEFAULTS["scenario"]["n_workers"],
                n_trials=num_simulation_trials,
                seed=patient_and_machine_startSeed,
            )
            smac_incumbents = SMACIncumbentRecorder()
            smac = HyperparameterOptimizationFacade(
                scenario, objective_smac, callbacks=[smac_incumbents]
            )
            incumbent = smac.optimize()
            smac_score = -smac.runhistory.get_cost(incumbent)

        smac_elapsed_seconds = time.perf_counter() - smac_start_time
        _attach_smac_incumbents(
            convergence_rows, smac_incumbents.records, smac.runhistory.finished
        )
        print(f"SMAC completed in {smac_elapsed_seconds:.3f} seconds.")

        # --- Random Search ---
        rs_start_time = time.perf_counter()
        random_search_rng = np.random.default_rng(patient_and_machine_startSeed)
        best_random_score = -np.inf
        best_random_params = {}
        print(f"\n--- Starting Random Search Optimization ({num_simulation_trials} iterations) ---")
        for _ in range(num_simulation_trials):
            a, f, w = [
                random_search_rng.uniform(
                    search_boundaries[key][0],
                    search_boundaries[key][1],
                )
                for key in [
                    "amplitude",
                    "frequency",
                    "pulse_width",
                ]
            ]

            query_index = query_counts["Random Search"] + 1
            evaluation_seed = make_evaluation_seed(
                profile_seed,
                query_index,
            )

            current_score = evaluate_and_record(
                "Random Search",
                {
                    "amplitude": a,
                    "frequency": f,
                    "pulse_width": w,
                },
                evaluation_seed,
            )

            if current_score > best_random_score:
                best_random_score = current_score
                best_random_params = {
                    "amplitude": a,
                    "frequency": f,
                    "pulse_width": w,
                }

        rs_elapsed_seconds = time.perf_counter() - rs_start_time
        print(f"Random Search completed in {rs_elapsed_seconds:.3f} seconds.")

        incomplete_query_counts = {
            method: query_counts[method]
            for method in method_names
            if query_counts[method] != num_simulation_trials
        }
        if incomplete_query_counts:
            raise RuntimeError(
                f"Optimizer objective evaluation counts differ from the requested "
                f"budget of {num_simulation_trials}: {incomplete_query_counts}."
            )
        if smac.runhistory.finished != num_simulation_trials:
            raise RuntimeError(
                f"SMAC completed {smac.runhistory.finished} runhistory evaluations; "
                f"expected {num_simulation_trials}."
            )

        _finalize_convergence_metrics(convergence_rows, max_possible_score)

        selected_scores_noise_free = [
            realistic_patient_curve(
                bo_selected_params["amplitude"],
                bo_selected_params["frequency"],
                bo_selected_params["pulse_width"],
                False,
                patient_and_machine_startSeed,
            ),
            realistic_patient_curve(
                trials.argmin['amplitude'],
                trials.argmin['frequency'],
                trials.argmin['pulse_width'],
                False,
                patient_and_machine_startSeed,
            ),
            realistic_patient_curve(
                float(incumbent['amplitude']),
                float(incumbent['frequency']),
                float(incumbent['pulse_width']),
                False,
                patient_and_machine_startSeed,
            ),
            realistic_patient_curve(
                best_random_params['amplitude'],
                best_random_params['frequency'],
                best_random_params['pulse_width'],
                False,
                patient_and_machine_startSeed,
            ),
        ]

        simple_regrets = [
            max_possible_score - score
            for score in selected_scores_noise_free
        ]

        normalized_regrets = [
            regret / max_possible_score
            if max_possible_score > 0
            else np.nan
            for regret in simple_regrets
        ]

        comparison_data = {
            'Method': ['Bayesian Optimization', 'TPE Optimization', 'SMAC Optimization', 'Random Search'],
            'Surface Family': [patient_profile['surface_family']] * 4,
            'Seed': [patient_and_machine_startSeed] * 4,
            'Best Possible Score': [max_possible_score] * 4,
            'Best Computed Score': [optimizer_multi.max['target'], -trials.best_trial['result']['loss'], smac_score, best_random_score],
            'Selected Amplitude (mA)': [bo_selected_params["amplitude"], trials.argmin['amplitude'], float(incumbent['amplitude']), best_random_params['amplitude']],
            'Selected Frequency (Hz)': [bo_selected_params["frequency"], trials.argmin['frequency'], float(incumbent['frequency']), best_random_params['frequency']],
            'Selected Pulse Width (us)': [bo_selected_params["pulse_width"], trials.argmin['pulse_width'], float(incumbent['pulse_width']), best_random_params['pulse_width']],

            'Selected Score (Noise-Free)': selected_scores_noise_free,
            'Simple Regret': simple_regrets,
            #Gives percentage regret just in case
            'Normalized Regret (%)': [
                                100 * regret
                                for regret in normalized_regrets
                            ],
            'Sec To Compute': [bo_elapsed_seconds, tpe_elapsed_seconds, smac_elapsed_seconds, rs_elapsed_seconds]
        }

        # Explicit names make the primary recommendation and oracle diagnostic distinct.
        comparison_data[SELECTED_INCUMBENT_REGRET_COLUMN] = comparison_data["Normalized Regret (%)"].copy()
        comparison_data[ORACLE_BEST_EVALUATED_REGRET_COLUMN] = [
            _normalized_regret_percent(max_possible_score, best_noise_free[method])
            for method in comparison_data["Method"]
        ]
        patient_results = pd.DataFrame(comparison_data)
        patient_convergence = pd.DataFrame(convergence_rows)
        _validate_final_convergence(patient_results, patient_convergence)
        all_patients_results.append(patient_results)
        all_convergence_results.append(patient_convergence)

        patient_and_machine_startSeed += 1

    return all_patients_results, all_convergence_results

In [3]:
OPTIMIZER_STARTUP_POINTS = 4

BOGP_DEFAULTS = {
    "acquisition_function": "ExpectedImprovement",
    "xi": 0.01,
    "exploration_decay": None,
    "exploration_decay_delay": None,
    "acquisition_random_state": None,
    "gp_input_bounds": "[0.0, 1.0] independently for all three inputs",
    "gp_length_scale": 1.0,
    "gp_kernel": "Matern(length_scale=1.0, nu=2.5) on normalized inputs",
    "gp_alpha_original": 1e-6,
    "gp_alpha_policy": "per-observation variance / std(y)^2 + numerical jitter",
    "gp_numerical_jitter": 1e-6,
    "gp_normalize_y": True,
    "gp_noise_variances": {
        "normal": "4.0 score^2 (Gaussian standard deviation 2.0)",
        "heteroscedastic": "physical amplitude_i^2 score^2 per observation",
        "heavy_tailed": "12.0 score^2 (Student-t df=3, scale=2; moment matched)",
    },
    "gp_heavy_tailed_limitation": "Gaussian likelihood does not model Student-t tails",
    "gp_n_restarts_optimizer": 5,
    "gp_optimizer": "fmin_l_bfgs_b",
    "init_points": OPTIMIZER_STARTUP_POINTS,
    "n_iter_rule": "num_simulation_trials - init_points",
    "random_state": "patient_and_machine_startSeed",
    "verbose": 0,
    "allow_duplicate_points": True,
}


def _gp_observation_noise_variances(noise_type, physical_amplitudes):
    amplitudes = np.asarray(physical_amplitudes, dtype=float)
    if noise_type == "normal":
        return np.full_like(amplitudes, SIMULATED_GAUSSIAN_NOISE_STD ** 2, dtype=float)
    if noise_type == "heteroscedastic":
        return np.square(amplitudes)
    if noise_type == "heavy_tailed":
        variance = (
            SIMULATED_HEAVY_TAILED_SCALE ** 2
            * SIMULATED_HEAVY_TAILED_DF
            / (SIMULATED_HEAVY_TAILED_DF - 2)
        )
        return np.full_like(amplitudes, variance, dtype=float)
    return np.zeros_like(amplitudes, dtype=float)


def _normalized_gp_alpha(noise_variances, targets, normalize_y):
    """Express observation variance in the target units used by normalize_y."""
    noise_variances = np.asarray(noise_variances, dtype=float)
    if not normalize_y:
        return noise_variances
    target_scale = float(np.std(np.asarray(targets, dtype=float)))
    if target_scale < 10 * np.finfo(float).eps:
        target_scale = 1.0
    return noise_variances / target_scale ** 2


TPE_DEFAULTS = {
    "prior_weight": 1.0,
    "n_startup_jobs": OPTIMIZER_STARTUP_POINTS,
    "n_EI_candidates": 24,
    "gamma": 0.25,
    "suggest_verbose": False,
    "fmin_verbose": False,
    "show_progressbar": False,
    "max_queue_len": 1,
    "return_argmin": True,
    "allow_trials_fmin": True,
    "rstate": "np.random.default_rng(patient_and_machine_startSeed)",
}

SMAC_DEFAULTS = {
    "scenario": {
        "deterministic": False,
        "objectives": "cost",
        "use_default_config": False,
        "n_workers": 1,
        "n_trials": "num_simulation_trials",
        "seed": "patient_and_machine_startSeed",
    },
    "model": {
        "type": "RandomForest",
        "n_trees": 10,
        "ratio_features": 1.0,
        "min_samples_split": 2,
        "min_samples_leaf": 1,
        "max_depth": 1048576,
        "bootstrapping": True,
    },
    "acquisition_function": {
        "type": "Expected Improvement",
        "xi": 0.0,
    },
    "initial_design": {
        "type": "SobolInitialDesign",
        "n_configs": OPTIMIZER_STARTUP_POINTS,
        "max_ratio": 1.0,
        "additional_configs": None,
    },
    "random_design": {
        "type": "ProbabilityRandomDesign",
        "probability": 0.2,
    },
    "intensifier": {
        "type": "Intensifier",
        "max_config_calls": 3,
        "max_incumbents": 10,
    },
    "config_selector": {
        "retrain_after": 8,
        "retries": 16,
    },
}

RANDOM_SEARCH_DEFAULTS = {
    "distribution": "independent Uniform(low, high) per parameter",
    "bounds": {
        "amplitude": (0.5, 5.0),
        "frequency": (1.0, 50.0),
        "pulse_width": (50.0, 600.0),
    },
    "sampling": "one independent draw for each parameter at every query",
    "replacement": True,
    "generator": "np.random.default_rng",
    "seed": "patient_and_machine_startSeed",
    "query_budget": "num_simulation_trials",
}

OPTIMIZER_PACKAGE_VERSIONS = {
    package_name: importlib_metadata.version(package_name)
    for package_name in (
        "bayesian-optimization",
        "hyperopt",
        "ConfigSpace",
        "smac",
    )
}


if "_original_bayesian_optimization" not in globals():
    _original_bayesian_optimization = BayesianOptimization


class ConfiguredBayesianOptimization(_original_bayesian_optimization):
    def __init__(self, *args, **kwargs):
        self._gp_noise_type = kwargs.pop("gp_noise_type", "normal")
        self._gp_physical_bounds = kwargs.pop("gp_physical_bounds", None)
        if self._gp_noise_type == "heteroscedastic" and (
            self._gp_physical_bounds is None
            or "amplitude" not in self._gp_physical_bounds
        ):
            raise ValueError("Heteroscedastic GP noise requires physical amplitude bounds.")
        kwargs.setdefault(
            "allow_duplicate_points",
            BOGP_DEFAULTS["allow_duplicate_points"],
        )
        super().__init__(*args, **kwargs)
        self.set_gp_params(
            kernel=Matern(length_scale=BOGP_DEFAULTS["gp_length_scale"], nu=2.5),
            alpha=BOGP_DEFAULTS["gp_numerical_jitter"],
            normalize_y=BOGP_DEFAULTS["gp_normalize_y"],
            n_restarts_optimizer=BOGP_DEFAULTS["gp_n_restarts_optimizer"],
            optimizer=BOGP_DEFAULTS["gp_optimizer"],
        )

    def _update_observation_noise_alpha(self):
        targets = np.asarray(self._space.target, dtype=float)
        if targets.size == 0:
            return
        if self._gp_noise_type == "heteroscedastic":
            amplitude_index = self._space.keys.index("amplitude")
            normalized_amplitudes = self._space.params[:, amplitude_index]
            lower, upper = self._gp_physical_bounds["amplitude"]
            physical_amplitudes = lower + normalized_amplitudes * (upper - lower)
        else:
            physical_amplitudes = np.zeros_like(targets)
        noise_variances = _gp_observation_noise_variances(
            self._gp_noise_type, physical_amplitudes
        )
        alpha = _normalized_gp_alpha(
            noise_variances, targets, BOGP_DEFAULTS["gp_normalize_y"]
        )
        self._gp.set_params(
            alpha=alpha + BOGP_DEFAULTS["gp_numerical_jitter"]
        )

    def suggest(self):
        self._update_observation_noise_alpha()
        return super().suggest()


BayesianOptimization = ConfiguredBayesianOptimization


if "_original_tpe_suggest" not in globals():
    _original_tpe_suggest = tpe.suggest

tpe.suggest = partial(
    _original_tpe_suggest,
    prior_weight=TPE_DEFAULTS["prior_weight"],
    n_startup_jobs=TPE_DEFAULTS["n_startup_jobs"],
    n_EI_candidates=TPE_DEFAULTS["n_EI_candidates"],
    gamma=TPE_DEFAULTS["gamma"],
    verbose=TPE_DEFAULTS["suggest_verbose"],
)


if "_original_hyperopt_fmin" not in globals():
    _original_hyperopt_fmin = fmin


def configured_hyperopt_fmin(*args, **kwargs):
    kwargs.setdefault("verbose", TPE_DEFAULTS["fmin_verbose"])
    kwargs.setdefault("show_progressbar", TPE_DEFAULTS["show_progressbar"])
    kwargs.setdefault("max_queue_len", TPE_DEFAULTS["max_queue_len"])
    kwargs.setdefault("return_argmin", TPE_DEFAULTS["return_argmin"])
    kwargs.setdefault("allow_trials_fmin", TPE_DEFAULTS["allow_trials_fmin"])
    return _original_hyperopt_fmin(*args, **kwargs)


fmin = configured_hyperopt_fmin


if "_original_smac_facade" not in globals():
    _original_smac_facade = HyperparameterOptimizationFacade


class ConfiguredSMACFacade(_original_smac_facade):
    def __init__(self, scenario, target_function=None, **kwargs):
        kwargs.setdefault("overwrite", True)
        kwargs.setdefault(
            "model",
            _original_smac_facade.get_model(
                scenario,
                n_trees=SMAC_DEFAULTS["model"]["n_trees"],
                ratio_features=SMAC_DEFAULTS["model"]["ratio_features"],
                min_samples_split=SMAC_DEFAULTS["model"]["min_samples_split"],
                min_samples_leaf=SMAC_DEFAULTS["model"]["min_samples_leaf"],
                max_depth=SMAC_DEFAULTS["model"]["max_depth"],
                bootstrapping=SMAC_DEFAULTS["model"]["bootstrapping"],
            ),
        )
        kwargs.setdefault(
            "acquisition_function",
            _original_smac_facade.get_acquisition_function(
                scenario,
                xi=SMAC_DEFAULTS["acquisition_function"]["xi"],
            ),
        )
        kwargs.setdefault(
            "initial_design",
            _original_smac_facade.get_initial_design(
                scenario,
                n_configs=SMAC_DEFAULTS["initial_design"]["n_configs"],
                max_ratio=SMAC_DEFAULTS["initial_design"]["max_ratio"],
                additional_configs=SMAC_DEFAULTS["initial_design"][
                    "additional_configs"
                ],
            ),
        )
        kwargs.setdefault(
            "random_design",
            _original_smac_facade.get_random_design(
                scenario,
                probability=SMAC_DEFAULTS["random_design"]["probability"],
            ),
        )
        kwargs.setdefault(
            "intensifier",
            _original_smac_facade.get_intensifier(
                scenario,
                max_config_calls=SMAC_DEFAULTS["intensifier"]["max_config_calls"],
                max_incumbents=SMAC_DEFAULTS["intensifier"]["max_incumbents"],
            ),
        )
        kwargs.setdefault(
            "config_selector",
            _original_smac_facade.get_config_selector(
                scenario,
                retrain_after=SMAC_DEFAULTS["config_selector"]["retrain_after"],
                retries=SMAC_DEFAULTS["config_selector"]["retries"],
            ),
        )
        super().__init__(scenario, target_function, **kwargs)


HyperparameterOptimizationFacade = ConfiguredSMACFacade


print("Explicit optimizer configuration applied.")
print("Installed optimizer package versions:")
pprint(OPTIMIZER_PACKAGE_VERSIONS)
print("\nBOGP defaults:")
pprint(BOGP_DEFAULTS)
print("\nTPE defaults:")
pprint(TPE_DEFAULTS)
print("\nSMAC defaults:")
pprint(SMAC_DEFAULTS)
print("\nRandom Search defaults:")
pprint(RANDOM_SEARCH_DEFAULTS)

Explicit optimizer configuration applied.
Installed optimizer package versions:
{'ConfigSpace': '1.2.2',
 'bayesian-optimization': '3.3.0',
 'hyperopt': '0.3.0',
 'smac': '2.4.0'}

BOGP defaults:
{'acquisition_function': 'ExpectedImprovement',
 'acquisition_random_state': None,
 'allow_duplicate_points': True,
 'exploration_decay': None,
 'exploration_decay_delay': None,
 'gp_alpha_original': 1e-06,
 'gp_alpha_policy': 'per-observation variance / std(y)^2 + numerical jitter',
 'gp_heavy_tailed_limitation': 'Gaussian likelihood does not model Student-t '
                               'tails',
 'gp_input_bounds': '[0.0, 1.0] independently for all three inputs',
 'gp_kernel': 'Matern(length_scale=1.0, nu=2.5) on normalized inputs',
 'gp_length_scale': 1.0,
 'gp_n_restarts_optimizer': 5,
 'gp_noise_variances': {'heavy_tailed': '12.0 score^2 (Student-t df=3, '
                                        'scale=2; moment matched)',
                        'heteroscedastic': 'physical amplitude_

In [4]:
def export_family_results(
    family,
    family_test,
    family_convergence,
    iteration_budget=40,
    output_dir=".",
    file_prefix=None,
    n_permutations=20000,
    n_bootstrap=10000,
    random_seed=10001,
    display_links=True,
):
    """Analyze one family's optimizer results and export its CSV outputs."""
    if isinstance(family, dict):
        family_name = family.get("name")
    else:
        family_name = str(family)

    if not isinstance(family_name, str) or not family_name.strip():
        raise ValueError("family must be a non-empty name or a dict with a 'name'.")
    if not isinstance(family_test, (list, tuple)) or not family_test:
        raise ValueError("family_test must be a non-empty list of per-profile DataFrames.")
    if not isinstance(family_convergence, (list, tuple)):
        raise ValueError("family_convergence must be a list of per-profile DataFrames.")
    if len(family_test) != len(family_convergence):
        raise ValueError("family_test and family_convergence must have the same length.")
    if not all(isinstance(frame, pd.DataFrame) for frame in family_test):
        raise TypeError("Every item in family_test must be a pandas DataFrame.")
    if not all(isinstance(frame, pd.DataFrame) for frame in family_convergence):
        raise TypeError("Every item in family_convergence must be a pandas DataFrame.")

    required_columns = {
        "Surface Family",
        "Seed",
        "Method",
        "Best Possible Score",
        "Selected Score (Noise-Free)",
    }
    missing_columns = set().union(
        *(required_columns.difference(frame.columns) for frame in family_test)
    )
    if missing_columns:
        raise ValueError(f"Missing family result columns: {sorted(missing_columns)}")

    family_results_df = pd.concat(
        [
            result.assign(
                **{
                    "Profile Index": profile_index,
                    "Profile ID": result["Surface Family"].astype(str)
                    + "::"
                    + result["Seed"].astype(str),
                    "Iteration Budget": iteration_budget,
                }
            )
            for profile_index, result in enumerate(family_test, start=1)
        ],
        ignore_index=True,
    )

    observed_families = set(family_results_df["Surface Family"].astype(str).unique())
    if observed_families != {family_name}:
        raise ValueError(
            f"family name {family_name!r} does not match result families {sorted(observed_families)}."
        )

    family_convergence_df = pd.concat(
        [
            convergence.assign(**{"Profile Index": profile_index})
            for profile_index, convergence in enumerate(family_convergence, start=1)
        ],
        ignore_index=True,
    )

    analysis_df = family_results_df.copy()
    analysis_df["Normalized Regret"] = (
        analysis_df["Best Possible Score"]
        - analysis_df["Selected Score (Noise-Free)"]
    ) / analysis_df["Best Possible Score"]

    method_order = [
        "Bayesian Optimization",
        "TPE Optimization",
        "SMAC Optimization",
        "Random Search",
    ]
    methods = [
        method
        for method in method_order
        if method in analysis_df["Method"].unique()
    ]
    if len(methods) != len(method_order):
        raise ValueError(f"Expected all four methods; found {methods}.")

    duplicate_keys = analysis_df.duplicated(
        subset=["Profile ID", "Method"],
        keep=False,
    )
    if duplicate_keys.any():
        raise ValueError(
            "Each family profile-method combination must have exactly one result."
        )

    family_wide = analysis_df.pivot(
        index="Profile ID",
        columns="Method",
        values="Normalized Regret",
    ).reindex(columns=methods).dropna()

    if len(family_wide) < 2:
        raise ValueError("At least two complete paired profiles are required.")

    def omnibus_permutation_test(wide, seed):
        values = wide.to_numpy(dtype=float)
        observed_statistic = float(np.ptp(values.mean(axis=0)))
        rng = np.random.default_rng(seed)
        permuted_statistics = np.empty(n_permutations)

        for permutation_index in range(n_permutations):
            permuted_values = np.array(
                [rng.permutation(profile_values) for profile_values in values]
            )
            permuted_statistics[permutation_index] = np.ptp(
                permuted_values.mean(axis=0)
            )

        p_value = (
            1
            + np.count_nonzero(permuted_statistics >= observed_statistic)
        ) / (n_permutations + 1)
        return observed_statistic, p_value

    def paired_sign_flip_test(first, second, seed):
        differences = np.asarray(first, dtype=float) - np.asarray(second, dtype=float)
        observed_difference = float(np.mean(differences))
        rng = np.random.default_rng(seed)
        signs = rng.choice(
            np.array([-1.0, 1.0]),
            size=(n_permutations, len(differences)),
        )
        permuted_means = np.mean(signs * differences, axis=1)
        p_value = (
            1
            + np.count_nonzero(
                np.abs(permuted_means) >= abs(observed_difference)
            )
        ) / (n_permutations + 1)
        return observed_difference, p_value

    def paired_bootstrap_ci(differences, seed):
        differences = np.asarray(differences, dtype=float)
        rng = np.random.default_rng(seed)
        indices = rng.integers(
            0,
            len(differences),
            size=(n_bootstrap, len(differences)),
        )
        bootstrap_means = differences[indices].mean(axis=1)
        return tuple(np.quantile(bootstrap_means, [0.025, 0.975]))

    def holm_adjust(p_values):
        p_values = np.asarray(p_values, dtype=float)
        order = np.argsort(p_values)
        sorted_p_values = p_values[order]
        adjusted_sorted = np.maximum.accumulate(
            (len(sorted_p_values) - np.arange(len(sorted_p_values)))
            * sorted_p_values
        )
        adjusted = np.empty_like(adjusted_sorted)
        adjusted[order] = np.minimum(adjusted_sorted, 1.0)
        return adjusted

    friedman_statistic, friedman_p_value = stats.friedmanchisquare(
        *[family_wide[method].to_numpy() for method in methods]
    )
    omnibus_statistic, omnibus_p_value = omnibus_permutation_test(
        family_wide,
        seed=random_seed + iteration_budget,
    )
    kendalls_w = friedman_statistic / (
        len(family_wide) * (len(methods) - 1)
    )

    omnibus_df = pd.DataFrame(
        [
            {
                "Surface Family": family_name,
                "Iteration Budget": iteration_budget,
                "N Paired Profiles": len(family_wide),
                "Endpoint": "Noise-free normalized regret",
                "Friedman Statistic": friedman_statistic,
                "Friedman P Value": friedman_p_value,
                "Kendalls W": kendalls_w,
                "Permutation Range Statistic": omnibus_statistic,
                "Paired Permutation P Value": omnibus_p_value,
            }
        ]
    )

    pairwise_rows = []
    for pair_index, (method_a, method_b) in enumerate(
        itertools.combinations(methods, 2)
    ):
        differences = (
            family_wide[method_a].to_numpy()
            - family_wide[method_b].to_numpy()
        )
        mean_difference, p_value = paired_sign_flip_test(
            family_wide[method_a],
            family_wide[method_b],
            seed=random_seed + 100000 + pair_index,
        )
        ci_low, ci_high = paired_bootstrap_ci(
            differences,
            seed=random_seed + 200000 + pair_index,
        )
        difference_sd = np.std(differences, ddof=1)
        pairwise_rows.append(
            {
                "Surface Family": family_name,
                "Iteration Budget": iteration_budget,
                "Method A": method_a,
                "Method B": method_b,
                "N Paired Profiles": len(differences),
                "Mean Regret Difference (A - B)": mean_difference,
                "95% Bootstrap CI Low": ci_low,
                "95% Bootstrap CI High": ci_high,
                "Paired Effect Size dz": (
                    mean_difference / difference_sd
                    if difference_sd > 0
                    else np.nan
                ),
                "P Method A Has Lower Regret": np.mean(differences < 0),
                "P Value": p_value,
            }
        )

    pairwise_df = pd.DataFrame(pairwise_rows)
    pairwise_df["Holm Adjusted P Value"] = holm_adjust(
        pairwise_df["P Value"]
    )

    summary_df = (
        analysis_df.groupby("Method")["Normalized Regret"]
        .agg(
            N_Profiles="count",
            Mean_Regret="mean",
            SD_Regret="std",
            Median_Regret="median",
            Q25_Regret=lambda values: values.quantile(0.25),
            Q75_Regret=lambda values: values.quantile(0.75),
        )
        .reindex(methods)
        .reset_index()
    )

    safe_prefix = re.sub(
        r"[^A-Za-z0-9_-]+",
        "_",
        str(file_prefix or family_name).strip(),
    ).strip("_")
    if not safe_prefix:
        raise ValueError("file_prefix must contain at least one usable character.")

    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)
    family_title = safe_prefix.replace("_", " ").title()
    download_files = {
        f"{family_title} profile results": (
            output_dir / f"{safe_prefix}_family_results.csv",
            family_results_df,
        ),
        f"{family_title} convergence results": (
            output_dir / f"{safe_prefix}_family_convergence.csv",
            family_convergence_df,
        ),
        f"{family_title} descriptive statistics": (
            output_dir / f"{safe_prefix}_summary_statistics.csv",
            summary_df,
        ),
        f"{family_title} omnibus tests": (
            output_dir / f"{safe_prefix}_omnibus_tests.csv",
            omnibus_df,
        ),
        f"{family_title} paired post-hoc tests": (
            output_dir / f"{safe_prefix}_pairwise_tests.csv",
            pairwise_df,
        ),
    }

    for path, frame in download_files.values():
        frame.to_csv(path, index=False)

    print(
        f"Exported {family_name!r}: {len(family_results_df):,} profile-method rows, "
        f"{len(family_convergence_df):,} query rows, and "
        f"{len(download_files)} CSV files."
    )
    if display_links:
        for label, (path, _) in download_files.items():
            display(FileLink(str(path), result_html_prefix=f"{label}: "))

    return {
        "family_name": family_name,
        "results_df": family_results_df,
        "convergence_df": family_convergence_df,
        "analysis_df": analysis_df,
        "wide": family_wide,
        "summary_df": summary_df,
        "omnibus_df": omnibus_df,
        "pairwise_df": pairwise_df,
        "files": {label: str(path) for label, (path, _) in download_files.items()},
    }


def run_and_export_family(
    family,
    num_simulation_trials=40,
    amount_of_patients=40,
    patient_and_machine_startSeed=10000,
    iteration_budget=None,
    output_dir=".",
    file_prefix=None,
    **export_kwargs,
):
    """Run one family through all optimizers, analyze it, and export its CSVs."""
    family_test, family_convergence = run_optimization_comparison(
        num_simulation_trials=num_simulation_trials,
        amount_of_patients=amount_of_patients,
        patient_and_machine_startSeed=patient_and_machine_startSeed,
        patient_type=family,
    )
    outputs = export_family_results(
        family=family,
        family_test=family_test,
        family_convergence=family_convergence,
        iteration_budget=(
            num_simulation_trials
            if iteration_budget is None
            else iteration_budget
        ),
        output_dir=output_dir,
        file_prefix=file_prefix,
        **export_kwargs,
    )
    outputs["family_test"] = family_test
    outputs["family_convergence"] = family_convergence
    return outputs

In [5]:
METHOD_ORDER = [
    "Bayesian Optimization",
    "TPE Optimization",
    "SMAC Optimization",
    "Random Search",
]

# Queries each optimizer spends on its initial design before model-based search begins.
# Random Search never switches to a model, so it has no boundary.
INITIAL_DESIGN_QUERIES = {
    "Bayesian Optimization": 4,
    "TPE Optimization": 20,
    "SMAC Optimization": 10,
    "Random Search": None,
}

INCUMBENT_SCORE_COLUMN = "Current Incumbent Noise-Free Score"
INCUMBENT_REGRET_COLUMN = SELECTED_INCUMBENT_REGRET_COLUMN
ORACLE_REGRET_COLUMN = ORACLE_BEST_EVALUATED_REGRET_COLUMN


def _build_common_random_number_summary(common_random_number_df):
    summary_columns = [
        "Iteration Budget",
        "N Tested Groups",
        "N Passed Groups",
        "N Failed Groups",
        "Overall Status",
    ]
    if common_random_number_df.empty:
        return pd.DataFrame(columns=summary_columns)

    summary_df = (
        common_random_number_df.groupby("Iteration Budget")[
            "Uses Common Random Number"
        ]
        .agg(
            **{
                "N Tested Groups": "size",
                "N Passed Groups": "sum",
                "N Failed Groups": lambda values: (~values).sum(),
            }
        )
        .reset_index()
    )
    summary_df["N Passed Groups"] = summary_df["N Passed Groups"].astype(int)
    summary_df["N Failed Groups"] = summary_df["N Failed Groups"].astype(int)
    summary_df["Overall Status"] = np.where(
        summary_df["N Failed Groups"] == 0,
        "PASS",
        "FAIL",
    )
    return summary_df


def _add_incumbent_regret(convergence_df, best_possible_by_profile):
    """Validate the recommendation trajectory; never substitute an oracle curve."""
    if convergence_df.empty:
        return None

    required = {
        "Profile ID", "Method", INCUMBENT_SCORE_COLUMN,
        INCUMBENT_REGRET_COLUMN, INCUMBENT_POLICY_COLUMN,
        *INCUMBENT_PARAMETER_COLUMNS.values(),
    }
    missing = required.difference(convergence_df.columns)
    if missing:
        raise ValueError(
            "Selected-incumbent convergence data are incomplete or from an older "
            "notebook. Earlier logs did not track SMAC's actual per-query "
            "incumbent; regenerate convergence data before analyzing trajectories. "
            f"Missing columns: {sorted(missing)}. Final results can still be "
            "analyzed without convergence data."
        )

    expected_policy = convergence_df["Method"].map({
        "Bayesian Optimization": OBSERVED_INCUMBENT_POLICY,
        "TPE Optimization": OBSERVED_INCUMBENT_POLICY,
        "SMAC Optimization": SMAC_INCUMBENT_POLICY,
        "Random Search": OBSERVED_INCUMBENT_POLICY,
    })
    if expected_policy.isna().any() or not expected_policy.eq(
        convergence_df[INCUMBENT_POLICY_COLUMN]
    ).all():
        raise ValueError("Convergence rows have an unknown or incorrect incumbent selection policy.")

    best_possible = pd.to_numeric(
        convergence_df["Profile ID"].map(best_possible_by_profile), errors="coerce"
    )
    incumbent_score = pd.to_numeric(convergence_df[INCUMBENT_SCORE_COLUMN], errors="coerce")
    recorded_regret = pd.to_numeric(convergence_df[INCUMBENT_REGRET_COLUMN], errors="coerce")
    usable = (
        np.isfinite(best_possible) & (best_possible > 0)
        & np.isfinite(incumbent_score) & np.isfinite(recorded_regret)
    )
    if not usable.all():
        raise ValueError("Selected-incumbent convergence contains missing or invalid values.")
    expected_regret = 100 * (best_possible - incumbent_score) / best_possible
    if not np.allclose(recorded_regret, expected_regret, rtol=1e-10, atol=1e-10):
        raise ValueError("Recorded selected-incumbent regret does not match the incumbent score.")
    if "Normalized Regret So Far (%)" in convergence_df and not np.allclose(
        pd.to_numeric(convergence_df["Normalized Regret So Far (%)"], errors="coerce"),
        recorded_regret, rtol=1e-10, atol=1e-10,
    ):
        raise ValueError("The legacy-named regret alias does not match selected-incumbent regret.")

    convergence_df["Best Possible Score"] = best_possible
    return INCUMBENT_REGRET_COLUMN


def _build_convergence_summary(trajectory_summary_df):
    summary_columns = [
        "Surface Family",
        "Method",
        "Iteration Budget",
        "N Profiles",
        "N Queries",
        "First Query",
        "Final Query",
        "First Median Normalized Regret (%)",
        "Final Median Normalized Regret (%)",
        "Final Q25 (%)",
        "Final Q75 (%)",
        "Median Improvement (%)",
    ]
    if trajectory_summary_df.empty:
        return pd.DataFrame(columns=summary_columns)

    group_columns = ["Surface Family", "Method", "Iteration Budget"]
    summary_rows = []
    for group_values, group in trajectory_summary_df.groupby(group_columns, sort=False):
        ordered_group = group.sort_values("Query")
        first_row = ordered_group.iloc[0]
        final_row = ordered_group.iloc[-1]
        summary_rows.append(
            {
                **dict(zip(group_columns, group_values)),
                "N Profiles": (
                    int(final_row["N Profiles"])
                    if "N Profiles" in ordered_group
                    else np.nan
                ),
                "N Queries": len(ordered_group),
                "First Query": first_row["Query"],
                "Final Query": final_row["Query"],
                "First Median Normalized Regret (%)": first_row["Median"],
                "Final Median Normalized Regret (%)": final_row["Median"],
                "Final Q25 (%)": final_row["Q25"],
                "Final Q75 (%)": final_row["Q75"],
                "Median Improvement (%)": first_row["Median"] - final_row["Median"],
            }
        )

    return pd.DataFrame(summary_rows, columns=summary_columns).sort_values(
        group_columns
    ).reset_index(drop=True)


def analyze_family_results(
    family_data,
    family_convergence=None,
    *,
    family_name=None,
    iteration_budget=40,
    score_column=None,
    regret_tolerance=0.05,
    n_permutations=20000,
    n_bootstrap=10000,
    random_seed=10001,
    display_results=True,
    plot_results=True,
):
    """Analyze one family using normalized noise-free regret as the primary endpoint."""
    if n_permutations < 1 or n_bootstrap < 1:
        raise ValueError("n_permutations and n_bootstrap must both be positive.")

    if isinstance(family_data, dict):
        result_data = family_data.get("results_df", family_data.get("family_test"))
        if result_data is None:
            raise ValueError("family_data must contain 'results_df' or 'family_test'.")
        if family_convergence is None:
            family_convergence = family_data.get(
                "convergence_df", family_data.get("family_convergence")
            )
        if family_name is None:
            family_name = family_data.get("family_name")
    else:
        result_data = family_data

    if isinstance(result_data, pd.DataFrame):
        analysis_df = result_data.copy()
    elif isinstance(result_data, (list, tuple)) and result_data:
        result_frames = []
        for profile_index, result_frame in enumerate(result_data, start=1):
            if not isinstance(result_frame, pd.DataFrame):
                raise TypeError("Each family result must be a pandas DataFrame.")
            result_frame = result_frame.copy()
            if "Profile Index" not in result_frame:
                result_frame["Profile Index"] = profile_index
            if "Iteration Budget" not in result_frame:
                result_frame["Iteration Budget"] = iteration_budget
            result_frames.append(result_frame)
        analysis_df = pd.concat(result_frames, ignore_index=True)
    else:
        raise TypeError(
            "family_data must be a results DataFrame, a non-empty list of DataFrames, "
            "or the dictionary returned by run_and_export_family."
        )

    required_columns = {"Surface Family", "Seed", "Method", "Best Possible Score"}
    missing_columns = required_columns.difference(analysis_df.columns)
    if missing_columns:
        raise ValueError(f"Missing required result columns: {sorted(missing_columns)}")

    score_candidates = ["Selected Score (Noise-Free)", "Independent Reevaluation Score"]
    if score_column is None:
        score_column = next(
            (
                candidate
                for candidate in score_candidates
                if candidate in analysis_df and not analysis_df[candidate].isna().all()
            ),
            None,
        )
    if score_column is None or score_column not in analysis_df:
        raise ValueError(
            "A selected noise-free or independent reevaluation score is required. "
            "The noisy maximum in 'Best Computed Score' is not an acceptable primary endpoint."
        )
    if score_column == "Best Computed Score":
        raise ValueError("Best Computed Score is a noisy maximum and cannot be primary.")

    analysis_df["Surface Family"] = analysis_df["Surface Family"].astype(str)
    observed_families = set(analysis_df["Surface Family"].unique())
    if family_name is None:
        if len(observed_families) != 1:
            raise ValueError("family_name is required when results contain more than one family.")
        family_name = next(iter(observed_families))
    family_name = str(family_name)
    if observed_families != {family_name}:
        raise ValueError(
            f"family_name {family_name!r} does not match result families "
            f"{sorted(observed_families)}."
        )

    if "Iteration Budget" not in analysis_df:
        analysis_df["Iteration Budget"] = iteration_budget
    analysis_df["Iteration Budget"] = pd.to_numeric(
        analysis_df["Iteration Budget"], errors="coerce"
    )
    if analysis_df["Iteration Budget"].isna().any():
        raise ValueError("Iteration Budget must be numeric for every result row.")
    analysis_df["Iteration Budget"] = analysis_df["Iteration Budget"].astype(int)

    if "Profile ID" not in analysis_df:
        analysis_df["Profile ID"] = (
            analysis_df["Surface Family"] + "::" + analysis_df["Seed"].astype(str)
        )
    else:
        analysis_df["Profile ID"] = analysis_df["Profile ID"].astype(str)

    numeric_columns = ["Best Possible Score", score_column]
    if "Sec To Compute" in analysis_df:
        numeric_columns.append("Sec To Compute")
    for column in numeric_columns:
        analysis_df[column] = pd.to_numeric(analysis_df[column], errors="coerce")
    if analysis_df[numeric_columns].isna().any().any() or not np.isfinite(
        analysis_df[numeric_columns].to_numpy(dtype=float)
    ).all():
        raise ValueError("Required numeric result columns contain invalid values.")
    if (analysis_df["Best Possible Score"] <= 0).any():
        raise ValueError("Best Possible Score must be positive to calculate regret.")

    analysis_df["Simple Regret"] = (
        analysis_df["Best Possible Score"] - analysis_df[score_column]
    )
    analysis_df["Normalized Regret"] = (
        analysis_df["Simple Regret"] / analysis_df["Best Possible Score"]
    )
    analysis_df["Normalized Regret (%)"] = 100 * analysis_df["Normalized Regret"]

    methods = [method for method in METHOD_ORDER if method in analysis_df["Method"].unique()]
    missing_methods = [method for method in METHOD_ORDER if method not in methods]
    if missing_methods:
        raise ValueError(f"Expected all four methods; missing {missing_methods}.")

    def paired_wide(frame, budget):
        budget_frame = frame[frame["Iteration Budget"] == budget]
        if budget_frame.duplicated(["Profile ID", "Method"], keep=False).any():
            raise ValueError("Each profile-method-budget combination must have one result.")
        return budget_frame.pivot(
            index="Profile ID",
            columns="Method",
            values="Normalized Regret",
        ).reindex(columns=methods).dropna()

    def paired_sign_flip_test(first, second, seed):
        differences = np.asarray(first, dtype=float) - np.asarray(second, dtype=float)
        observed_difference = float(np.mean(differences))
        rng = np.random.default_rng(seed)
        signs = rng.choice(
            np.array([-1.0, 1.0]), size=(n_permutations, len(differences))
        )
        permuted_means = np.mean(signs * differences, axis=1)
        p_value = (
            1 + np.count_nonzero(np.abs(permuted_means) >= abs(observed_difference))
        ) / (n_permutations + 1)
        return differences, observed_difference, p_value

    def paired_bootstrap_ci(differences, seed):
        differences = np.asarray(differences, dtype=float)
        rng = np.random.default_rng(seed)
        indices = rng.integers(0, len(differences), size=(n_bootstrap, len(differences)))
        return tuple(np.quantile(differences[indices].mean(axis=1), [0.025, 0.975]))

    def omnibus_permutation_test(wide, seed):
        values = wide.to_numpy(dtype=float)
        observed_statistic = float(np.ptp(values.mean(axis=0)))
        rng = np.random.default_rng(seed)
        permuted_statistics = np.empty(n_permutations)
        for permutation_index in range(n_permutations):
            permuted_values = np.array(
                [rng.permutation(profile_values) for profile_values in values]
            )
            permuted_statistics[permutation_index] = np.ptp(
                permuted_values.mean(axis=0)
            )
        p_value = (
            1 + np.count_nonzero(permuted_statistics >= observed_statistic)
        ) / (n_permutations + 1)
        return observed_statistic, p_value

    paired_primary_metric = {}
    summary_rows = []
    omnibus_rows = []
    pairwise_rows = []
    skipped_budget_rows = []
    budgets = sorted(analysis_df["Iteration Budget"].unique())

    for budget in budgets:
        wide = paired_wide(analysis_df, budget)
        if len(wide) < 2:
            skipped_budget_rows.append(
                {
                    "Iteration Budget": budget,
                    "N Complete Paired Profiles": len(wide),
                    "Reason": "At least two complete paired profiles are required.",
                }
            )
            continue

        paired_primary_metric[budget] = wide
        permutation_statistic, permutation_p_value = omnibus_permutation_test(
            wide, random_seed + int(budget)
        )
        friedman_statistic, friedman_p_value = stats.friedmanchisquare(
            *[wide[method].to_numpy() for method in methods]
        )
        omnibus_rows.append(
            {
                "Iteration Budget": budget,
                "N Paired Profiles": len(wide),
                "Endpoint": "Normalized noise-free regret (%)",
                "Paired Permutation Range Statistic": 100 * permutation_statistic,
                "Paired Permutation P Value": permutation_p_value,
                "Friedman Statistic": friedman_statistic,
                "Friedman P Value": friedman_p_value,
            }
        )

        budget_frame = analysis_df[
            (analysis_df["Iteration Budget"] == budget)
            & analysis_df["Profile ID"].isin(wide.index)
        ]
        for method in methods:
            method_values = wide[method]
            method_rows = budget_frame[budget_frame["Method"] == method]
            selected_values = method_rows[score_column]
            summary_rows.append(
                {
                    "Iteration Budget": budget,
                    "Method": method,
                    "N Profiles": len(method_values),
                    "Mean Normalized Regret (%)": 100 * method_values.mean(),
                    "SD Normalized Regret (%)": 100 * method_values.std(ddof=1),
                    "Median Normalized Regret (%)": 100 * method_values.median(),
                    "IQR Normalized Regret (%)": 100 * (
                        method_values.quantile(0.75) - method_values.quantile(0.25)
                    ),
                    "Mean Noise-Free Score": selected_values.mean(),
                    "SD Noise-Free Score": selected_values.std(ddof=1),
                    "Median Noise-Free Score": selected_values.median(),
                    "Mean Simple Regret": method_rows["Simple Regret"].mean(),
                    "Median Simple Regret": method_rows["Simple Regret"].median(),
                    "P Regret <= Tolerance": (
                        method_rows["Normalized Regret"] <= regret_tolerance
                    ).mean(),
                }
            )

        for pair_index, (method_a, method_b) in enumerate(itertools.combinations(methods, 2)):
            differences, mean_difference, p_value = paired_sign_flip_test(
                wide[method_a],
                wide[method_b],
                random_seed + 100000 + int(budget) * 100 + pair_index,
            )
            ci_low, ci_high = paired_bootstrap_ci(
                differences,
                random_seed + 200000 + int(budget) * 100 + pair_index,
            )
            difference_sd = np.std(differences, ddof=1)
            pairwise_rows.append(
                {
                    "Iteration Budget": budget,
                    "Method A": method_a,
                    "Method B": method_b,
                    "N Paired Profiles": len(differences),
                    "Mean Normalized Regret Difference (A - B) (%)": 100 * mean_difference,
                    "95% Bootstrap CI Low (%)": 100 * ci_low,
                    "95% Bootstrap CI High (%)": 100 * ci_high,
                    "Paired Effect Size dz": (
                        mean_difference / difference_sd if difference_sd > 0 else np.nan
                    ),
                    "P A Lower Regret Than B": np.mean(differences < 0),
                    "Paired Sign-Flip P Value": p_value,
                }
            )

    if not paired_primary_metric:
        raise ValueError("No iteration budget has at least two complete paired profiles.")

    summary_df = pd.DataFrame(summary_rows)
    omnibus_df = pd.DataFrame(omnibus_rows)
    pairwise_df = pd.DataFrame(pairwise_rows)
    skipped_budgets_df = pd.DataFrame(skipped_budget_rows)

    if not pairwise_df.empty:
        pairwise_df["Holm Adjusted P Value"] = np.nan
        for budget in pairwise_df["Iteration Budget"].unique():
            budget_mask = pairwise_df["Iteration Budget"] == budget
            pairwise_df.loc[budget_mask, "Holm Adjusted P Value"] = multipletests(
                pairwise_df.loc[budget_mask, "Paired Sign-Flip P Value"], method="holm"
            )[1]

    if "Sec To Compute" in analysis_df:
        cost_df = (
            analysis_df.groupby(["Iteration Budget", "Method"])["Sec To Compute"]
            .agg(
                N_Profiles="count",
                Mean_Seconds="mean",
                SD_Seconds="std",
                Median_Seconds="median",
            )
            .reindex(
                pd.MultiIndex.from_product(
                    [budgets, methods], names=["Iteration Budget", "Method"]
                )
            )
            .reset_index()
        )
    else:
        cost_df = pd.DataFrame()

    if family_convergence is None:
        convergence_df = pd.DataFrame()
    elif isinstance(family_convergence, pd.DataFrame):
        convergence_df = family_convergence.copy()
    elif isinstance(family_convergence, (list, tuple)):
        if not all(isinstance(frame, pd.DataFrame) for frame in family_convergence):
            raise TypeError("Each convergence result must be a pandas DataFrame.")
        convergence_df = pd.concat(family_convergence, ignore_index=True)
    else:
        raise TypeError("family_convergence must be a DataFrame, a list of DataFrames, or None.")

    trajectory_summary_df = pd.DataFrame()
    common_random_number_df = pd.DataFrame()
    incomplete_trajectory_df = pd.DataFrame()
    trajectory_score_column = None
    if not convergence_df.empty:
        if "Surface Family" not in convergence_df:
            convergence_df["Surface Family"] = family_name
        if "Profile ID" not in convergence_df:
            profile_seed_column = "Profile Seed" if "Profile Seed" in convergence_df else "Seed"
            if profile_seed_column in convergence_df:
                convergence_df["Profile ID"] = (
                    convergence_df["Surface Family"].astype(str)
                    + "::"
                    + convergence_df[profile_seed_column].astype(str)
                )
        else:
            convergence_df["Profile ID"] = convergence_df["Profile ID"].astype(str)
        if "Iteration Budget" not in convergence_df:
            convergence_df["Iteration Budget"] = iteration_budget

        best_possible_by_profile = (
            analysis_df.drop_duplicates("Profile ID")
            .set_index("Profile ID")["Best Possible Score"]
        )
        trajectory_score_column = _add_incumbent_regret(
            convergence_df, best_possible_by_profile
        )

        if trajectory_score_column is not None:
            trajectory_summary_df = (
                convergence_df.groupby(
                    ["Surface Family", "Method", "Iteration Budget", "Query"]
                )[trajectory_score_column]
                .agg(
                    Median="median",
                    Q25=lambda values: values.quantile(0.25),
                    Q75=lambda values: values.quantile(0.75),
                )
                .reset_index()
            )
            profile_counts = (
                convergence_df.groupby(
                    ["Surface Family", "Method", "Iteration Budget", "Query"]
                )["Profile ID"]
                .nunique()
                .reset_index(name="N Profiles")
            )
            trajectory_summary_df = trajectory_summary_df.merge(
                profile_counts,
                on=["Surface Family", "Method", "Iteration Budget", "Query"],
                how="left",
            )
            # A query summarised over fewer profiles than the family total is a biased subset.
            expected_profiles = convergence_df["Profile ID"].nunique()
            trajectory_summary_df["Complete Coverage"] = (
                trajectory_summary_df["N Profiles"] == expected_profiles
            )
            incomplete_trajectory_df = trajectory_summary_df[
                ~trajectory_summary_df["Complete Coverage"]
            ][["Surface Family", "Method", "Iteration Budget", "Query", "N Profiles"]]
            incomplete_trajectory_df = incomplete_trajectory_df.assign(
                **{"N Profiles Expected": expected_profiles}
            ).reset_index(drop=True)

        crn_columns = {"Profile ID", "Iteration Budget", "Query", "Method", "Evaluation Seed"}
        if crn_columns.issubset(convergence_df.columns):
            common_random_number_df = (
                convergence_df.groupby(["Profile ID", "Iteration Budget", "Query"])
                .agg(
                    N_Methods=("Method", "nunique"),
                    N_Evaluation_Seeds=("Evaluation Seed", "nunique"),
                )
                .reset_index()
            )
            common_random_number_df["Uses Common Random Number"] = (
                (common_random_number_df["N_Methods"] == len(methods))
                & (common_random_number_df["N_Evaluation_Seeds"] == 1)
            )

    figures = {}
    if plot_results:
        primary_figure, primary_axes = plt.subplots(
            1, len(paired_primary_metric), figsize=(6 * len(paired_primary_metric), 5), squeeze=False
        )
        for axis, budget in zip(primary_axes.ravel(), sorted(paired_primary_metric)):
            wide = paired_primary_metric[budget]
            axis.boxplot(
                [100 * wide[method].to_numpy() for method in methods],
                labels=methods,
                patch_artist=True,
            )
            axis.set_title(f"{family_name}: normalized regret, {budget} queries")
            axis.set_ylabel("Normalized noise-free regret (%)")
            axis.tick_params(axis="x", labelrotation=25)
            axis.grid(axis="y", alpha=0.25)
        primary_figure.tight_layout()
        figures["normalized_regret_distribution"] = primary_figure
        plt.show()

        if not cost_df.empty:
            cost_figure, cost_axis = plt.subplots(figsize=(8, 5))
            cost_df.pivot(index="Method", columns="Iteration Budget", values="Mean_Seconds").reindex(
                methods
            ).plot(
                kind="bar",
                yerr=cost_df.pivot(
                    index="Method", columns="Iteration Budget", values="SD_Seconds"
                ).reindex(methods),
                ax=cost_axis,
                capsize=4,
            )
            cost_axis.set_title(f"{family_name}: computational cost")
            cost_axis.set_ylabel("Seconds per profile")
            cost_axis.grid(axis="y", alpha=0.25)
            cost_figure.tight_layout()
            figures["computational_cost"] = cost_figure
            plt.show()

        if not trajectory_summary_df.empty:
            trajectory_budgets = sorted(trajectory_summary_df["Iteration Budget"].unique())
            trajectory_figure, trajectory_axes = plt.subplots(
                1, len(trajectory_budgets), figsize=(7 * len(trajectory_budgets), 5), squeeze=False
            )
            for axis, budget in zip(trajectory_axes.ravel(), trajectory_budgets):
                budget_trajectory = trajectory_summary_df[
                    (trajectory_summary_df["Iteration Budget"] == budget)
                    & trajectory_summary_df["Complete Coverage"]
                ]
                for method in methods:
                    method_data = budget_trajectory[
                        budget_trajectory["Method"] == method
                    ].sort_values("Query")
                    if method_data.empty:
                        continue
                    line = axis.plot(
                        method_data["Query"], method_data["Median"], label=method
                    )[0]
                    axis.fill_between(
                        method_data["Query"],
                        method_data["Q25"],
                        method_data["Q75"],
                        alpha=0.15,
                        color=line.get_color(),
                    )
                    boundary = INITIAL_DESIGN_QUERIES.get(method)
                    if boundary is not None:
                        axis.axvline(
                            boundary,
                            color=line.get_color(),
                            linestyle=":",
                            alpha=0.7,
                            linewidth=1.2,
                        )
                axis.set_title(f"{family_name}: regret convergence, {budget} queries")
                axis.set_xlabel("Query")
                axis.set_ylabel("Normalized noise-free regret (%)")
                axis.grid(alpha=0.25)
            trajectory_figure.legend(loc="lower center", ncol=2)
            trajectory_figure.suptitle(
                "Regret of the parameters each optimizer would recommend after each query. "
                "Dotted lines mark the end of each initial design "
                "(BO 4, SMAC 10, TPE 20 queries; Random Search never switches to a model).",
                fontsize=9,
                y=0.99,
            )
            trajectory_figure.tight_layout(rect=(0, 0.08, 1, 0.95))
            figures["convergence"] = trajectory_figure
            plt.show()

    common_random_number_summary_df = _build_common_random_number_summary(
        common_random_number_df
    )
    trajectory_summary_compact_df = _build_convergence_summary(trajectory_summary_df)
    if display_results:
        print(
            "Primary endpoint: normalized noise-free regret (%); lower is better. "
            "Selected Noise-Free Score is retained as a secondary absolute metric."
        )
        if trajectory_score_column == INCUMBENT_REGRET_COLUMN:
            print(
                "Convergence endpoint: noise-free regret of the incumbent, i.e. the "
                "parameters the optimizer would recommend if stopped at that query."
            )
        else:
            print("No selected-incumbent convergence data were supplied.")
        print("\n--- Primary normalized regret and secondary score summaries ---")
        display(summary_df)
        print("\n--- Paired omnibus tests on normalized regret ---")
        display(omnibus_df)
        print("\n--- Paired sign-flip tests with Holm correction ---")
        display(pairwise_df)
        print("\n--- Computational cost ---")
        display(cost_df)
        if not common_random_number_df.empty:
            print("\n--- Common-random-number audit ---")
            display(common_random_number_summary_df)
        if not trajectory_summary_df.empty:
            print("\n--- Normalized regret convergence summary ---")
            display(trajectory_summary_compact_df)
        if not incomplete_trajectory_df.empty:
            print(
                "\n--- Queries excluded from the convergence figure (incomplete profile coverage) ---"
            )
            display(incomplete_trajectory_df)
        if not skipped_budgets_df.empty:
            print("\n--- Budgets omitted from inferential tests ---")
            display(skipped_budgets_df)

    return {
        "family_name": family_name,
        "score_column": "Normalized Regret",
        "primary_metric_column": "Normalized Regret",
        "selected_score_column": score_column,
        "regret_tolerance": regret_tolerance,
        "analysis_df": analysis_df,
        "paired_scores": paired_primary_metric,
        "paired_primary_metric": paired_primary_metric,
        "summary_df": summary_df,
        "omnibus_df": omnibus_df,
        "pairwise_df": pairwise_df,
        "cost_df": cost_df,
        "convergence_df": convergence_df,
        "trajectory_score_column": trajectory_score_column,
        "trajectory_summary_df": trajectory_summary_df,
        "trajectory_summary_compact_df": trajectory_summary_compact_df,
        "incomplete_trajectory_df": incomplete_trajectory_df,
        "common_random_number_df": common_random_number_df,
        "common_random_number_summary_df": common_random_number_summary_df,
        "skipped_budgets_df": skipped_budgets_df,
        "figures": figures,
    }


## **CLINICALLY ANCHORED EXPERIMENTS**
Run the shared definitions cell, then execute the Badran and Atanackov experiment cells separately.

In [6]:
# This section contains a standalone clinical-data-anchored response function.
# Default bounds for the original Badran model; pass measured bounds for other datasets.
CLINICAL_MAP_FREQUENCY_BOUNDS_HZ = (1.0, 25.0)
CLINICAL_MAP_PULSE_WIDTH_BOUNDS_US = (100.0, 500.0)

# Independent of the iteration and repeat settings used by other comparisons.
CLINICAL_OPTIMIZER_EVALUATIONS_PER_RUN = 40
CLINICAL_OPTIMIZER_REPEATS = 40


def _validate_clinical_map_bounds(bounds, name):
    try:
        values = np.asarray(bounds, dtype=float)
    except (TypeError, ValueError) as error:
        raise ValueError(f"{name} must contain two finite numeric bounds.") from error
    if values.shape != (2,) or not np.isfinite(values).all() or values[0] >= values[1]:
        raise ValueError(f"{name} must be a finite (lower, upper) pair with lower < upper.")
    return float(values[0]), float(values[1])


def build_clinical_response_function(
    anchor_points,
    frequency_bounds_hz=None,
    pulse_width_bounds_us=None,
):
    """Build a linear 2D response interpolator from 1-10 measured anchors.

    Each anchor is ``(frequency_hz, pulse_width_us, response_value)``. The
    response is returned in the same units as the supplied measurements.
    Complete rectangular grids use bilinear interpolation; other non-collinear
    point sets use piecewise-linear interpolation within their convex hull.
    A single anchor produces a constant map, and collinear anchors are linearly
    interpolated along their line with constant perpendicular extension and
    endpoint clamping.
    Queries outside the supplied bounds or interpolation domain are rejected
    rather than extrapolated.

    The returned callable accepts scalars or broadcastable NumPy arrays. To
    create a surface, evaluate it on ``numpy.meshgrid`` arrays.
    """
    from scipy.interpolate import LinearNDInterpolator, RegularGridInterpolator
    from scipy.spatial import QhullError

    try:
        anchors = np.asarray(anchor_points, dtype=float)
    except (TypeError, ValueError) as error:
        raise ValueError("anchor_points must be numeric (frequency, pulse width, response) rows.") from error

    if anchors.ndim == 1:
        anchors = anchors.reshape(1, -1)
    if anchors.ndim != 2 or anchors.shape[1] != 3:
        raise ValueError("anchor_points must have shape (n, 3): frequency_hz, pulse_width_us, response_value.")
    if not 1 <= anchors.shape[0] <= 10:
        raise ValueError("Provide between 1 and 10 clinical anchor points.")
    if not np.isfinite(anchors).all():
        raise ValueError("anchor_points must contain only finite numeric values.")

    if frequency_bounds_hz is None:
        frequency_bounds_hz = CLINICAL_MAP_FREQUENCY_BOUNDS_HZ
    if pulse_width_bounds_us is None:
        pulse_width_bounds_us = CLINICAL_MAP_PULSE_WIDTH_BOUNDS_US
    frequency_lower, frequency_upper = _validate_clinical_map_bounds(
        frequency_bounds_hz, "frequency_bounds_hz"
    )
    pulse_width_lower, pulse_width_upper = _validate_clinical_map_bounds(
        pulse_width_bounds_us, "pulse_width_bounds_us"
    )

    if (
        np.any(anchors[:, 0] < frequency_lower)
        or np.any(anchors[:, 0] > frequency_upper)
        or np.any(anchors[:, 1] < pulse_width_lower)
        or np.any(anchors[:, 1] > pulse_width_upper)
    ):
        raise ValueError("Every anchor must lie within the supplied frequency and pulse-width bounds.")

    anchor_coordinates = np.column_stack((
        (anchors[:, 0] - frequency_lower) / (frequency_upper - frequency_lower),
        (anchors[:, 1] - pulse_width_lower) / (pulse_width_upper - pulse_width_lower),
    ))
    if np.unique(anchor_coordinates, axis=0).shape[0] != len(anchor_coordinates):
        raise ValueError("Anchor frequency/pulse-width coordinates must be unique.")
    anchor_responses = anchors[:, 2].copy()

    unique_frequencies = np.unique(anchors[:, 0])
    unique_pulse_widths = np.unique(anchors[:, 1])
    is_complete_grid = (
        len(unique_frequencies) >= 2
        and len(unique_pulse_widths) >= 2
        and len(unique_frequencies) * len(unique_pulse_widths) == len(anchors)
    )

    if is_complete_grid:
        response_grid = np.empty(
            (len(unique_frequencies), len(unique_pulse_widths)),
            dtype=float,
        )
        for frequency_index, frequency in enumerate(unique_frequencies):
            for pulse_width_index, pulse_width in enumerate(unique_pulse_widths):
                matches = np.flatnonzero(
                    (anchors[:, 0] == frequency) & (anchors[:, 1] == pulse_width)
                )
                response_grid[frequency_index, pulse_width_index] = anchor_responses[
                    matches[0]
                ]
        grid_frequency_coordinates = (
            unique_frequencies - frequency_lower
        ) / (frequency_upper - frequency_lower)
        grid_pulse_width_coordinates = (
            unique_pulse_widths - pulse_width_lower
        ) / (pulse_width_upper - pulse_width_lower)
        grid_interpolator = RegularGridInterpolator(
            (grid_frequency_coordinates, grid_pulse_width_coordinates),
            response_grid,
            method="linear",
            bounds_error=False,
            fill_value=np.nan,
        )
        interpolation_mode = "grid"
    elif len(anchors) == 1:
        interpolation_mode = "constant"
    else:
        centered_coordinates = anchor_coordinates - anchor_coordinates.mean(axis=0)
        if np.linalg.matrix_rank(centered_coordinates) < 2:
            _, _, right_vectors = np.linalg.svd(
                centered_coordinates,
                full_matrices=False,
            )
            line_direction = right_vectors[0]
            line_positions = (
                anchor_coordinates - anchor_coordinates[0]
            ) @ line_direction
            sort_order = np.argsort(line_positions)
            line_positions = line_positions[sort_order]
            line_responses = anchor_responses[sort_order]
            interpolation_mode = "line"
        else:
            try:
                scattered_interpolator = LinearNDInterpolator(
                    anchor_coordinates,
                    anchor_responses,
                    fill_value=np.nan,
                )
            except QhullError as error:
                raise ValueError(
                    "Clinical anchors must define a valid 2D interpolation surface."
                ) from error
            interpolation_mode = "scattered"

    def response_function(frequency_hz, pulse_width_us):
        try:
            frequencies, pulse_widths = np.broadcast_arrays(
                np.asarray(frequency_hz, dtype=float),
                np.asarray(pulse_width_us, dtype=float),
            )
        except (TypeError, ValueError) as error:
            raise ValueError("Frequency and pulse-width queries must be numeric and broadcastable.") from error
        if not np.isfinite(frequencies).all() or not np.isfinite(pulse_widths).all():
            raise ValueError("Frequency and pulse-width queries must be finite.")
        if (
            np.any(frequencies < frequency_lower)
            or np.any(frequencies > frequency_upper)
            or np.any(pulse_widths < pulse_width_lower)
            or np.any(pulse_widths > pulse_width_upper)
        ):
            raise ValueError("Queries must lie within the supplied frequency and pulse-width bounds.")

        query_coordinates = np.column_stack((
            ((frequencies - frequency_lower) / (frequency_upper - frequency_lower)).ravel(),
            ((pulse_widths - pulse_width_lower) / (pulse_width_upper - pulse_width_lower)).ravel(),
        ))
        if interpolation_mode == "grid":
            interpolated = grid_interpolator(query_coordinates)
        elif interpolation_mode == "constant":
            interpolated = np.full(len(query_coordinates), anchor_responses[0])
        elif interpolation_mode == "line":
            query_positions = (
                query_coordinates - anchor_coordinates[0]
            ) @ line_direction
            interpolated = np.interp(
                query_positions,
                line_positions,
                line_responses,
            )
        else:
            interpolated = scattered_interpolator(query_coordinates)

        interpolated = np.asarray(interpolated, dtype=float).reshape(-1)
        if not np.isfinite(interpolated).all():
            raise ValueError(
                "Queries must lie within the interpolation domain of the anchors; "
                "extrapolation is not supported."
            )
        for anchor_index, anchor_coordinate in enumerate(anchor_coordinates):
            is_exact_anchor = np.all(
                np.isclose(
                    query_coordinates,
                    anchor_coordinate,
                    rtol=0.0,
                    atol=1e-12,
                ),
                axis=1,
            )
            interpolated[is_exact_anchor] = anchor_responses[anchor_index]

        response_values = interpolated.reshape(frequencies.shape)
        if response_values.ndim == 0:
            return float(response_values.item())
        return response_values

    return response_function


# The response map is enabled only after every entered response value is supplied.

def build_clinical_pchip_response_function(anchor_points):
    """Build PCHIP pulse-width curves with linear interpolation across frequency."""
    from scipy.interpolate import PchipInterpolator

    try:
        anchors = np.asarray(anchor_points, dtype=float)
    except (TypeError, ValueError) as error:
        raise ValueError(
            "anchor_points must be numeric (frequency, pulse width, response) rows."
        ) from error

    if anchors.ndim == 1:
        anchors = anchors.reshape(1, -1)
    if anchors.ndim != 2 or anchors.shape[1] != 3 or anchors.shape[0] < 4:
        raise ValueError(
            "PCHIP interpolation requires a rectangular grid of at least "
            "two frequencies by two pulse widths."
        )
    if not np.isfinite(anchors).all():
        raise ValueError("anchor_points must contain only finite numeric values.")

    unique_frequencies = np.unique(anchors[:, 0])
    unique_pulse_widths = np.unique(anchors[:, 1])
    if len(unique_frequencies) < 2 or len(unique_pulse_widths) < 2:
        raise ValueError(
            "PCHIP interpolation requires at least two frequencies and two pulse widths."
        )
    if np.unique(anchors[:, :2], axis=0).shape[0] != len(anchors):
        raise ValueError("Each frequency/pulse-width combination must appear exactly once.")
    if len(anchors) != len(unique_frequencies) * len(unique_pulse_widths):
        raise ValueError(
            "PCHIP interpolation requires a complete rectangular "
            "frequency/pulse-width grid."
        )

    response_grid = np.empty(
        (len(unique_frequencies), len(unique_pulse_widths)),
        dtype=float,
    )
    for frequency_index, frequency in enumerate(unique_frequencies):
        for pulse_width_index, pulse_width in enumerate(unique_pulse_widths):
            matches = np.flatnonzero(
                (anchors[:, 0] == frequency) & (anchors[:, 1] == pulse_width)
            )
            if matches.size != 1:
                raise ValueError(
                    "Each frequency/pulse-width combination must appear exactly once."
                )
            response_grid[frequency_index, pulse_width_index] = anchors[matches[0], 2]

    pulse_width_interpolators = [
        PchipInterpolator(
            unique_pulse_widths,
            response_grid[frequency_index],
            extrapolate=False,
        )
        for frequency_index in range(len(unique_frequencies))
    ]
    frequency_lower = unique_frequencies[0]
    frequency_upper = unique_frequencies[-1]
    pulse_width_lower = unique_pulse_widths[0]
    pulse_width_upper = unique_pulse_widths[-1]

    def response_function(frequency_hz, pulse_width_us):
        try:
            frequencies, pulse_widths = np.broadcast_arrays(
                np.asarray(frequency_hz, dtype=float),
                np.asarray(pulse_width_us, dtype=float),
            )
        except (TypeError, ValueError) as error:
            raise ValueError(
                "Frequency and pulse-width queries must be numeric and broadcastable."
            ) from error
        if not np.isfinite(frequencies).all() or not np.isfinite(pulse_widths).all():
            raise ValueError("Frequency and pulse-width queries must be finite.")
        if (
            np.any(frequencies < frequency_lower)
            or np.any(frequencies > frequency_upper)
            or np.any(pulse_widths < pulse_width_lower)
            or np.any(pulse_widths > pulse_width_upper)
        ):
            raise ValueError(
                "Queries must lie within the measured frequency and pulse-width ranges; "
                "extrapolation is not supported."
            )

        flat_frequencies = frequencies.ravel()
        flat_pulse_widths = pulse_widths.ravel()
        upper_frequency_indices = np.searchsorted(
            unique_frequencies, flat_frequencies, side="right"
        )
        upper_frequency_indices = np.clip(
            upper_frequency_indices, 1, len(unique_frequencies) - 1
        )
        lower_frequency_indices = upper_frequency_indices - 1
        frequency_fractions = (
            (flat_frequencies - unique_frequencies[lower_frequency_indices])
            / (
                unique_frequencies[upper_frequency_indices]
                - unique_frequencies[lower_frequency_indices]
            )
        )
        interpolated = np.empty(flat_frequencies.shape, dtype=float)
        for lower_index in range(len(unique_frequencies) - 1):
            in_frequency_interval = lower_frequency_indices == lower_index
            if not np.any(in_frequency_interval):
                continue
            lower_responses = pulse_width_interpolators[lower_index](
                flat_pulse_widths[in_frequency_interval]
            )
            upper_responses = pulse_width_interpolators[lower_index + 1](
                flat_pulse_widths[in_frequency_interval]
            )
            frequency_fraction = frequency_fractions[in_frequency_interval]
            interpolated[in_frequency_interval] = lower_responses + frequency_fraction * (
                upper_responses - lower_responses
            )
        interpolated = np.asarray(interpolated, dtype=float).reshape(-1)

        for frequency_index, frequency in enumerate(unique_frequencies):
            for pulse_width_index, pulse_width in enumerate(unique_pulse_widths):
                is_exact_anchor = (flat_frequencies == frequency) & (
                    flat_pulse_widths == pulse_width
                )
                interpolated[is_exact_anchor] = response_grid[
                    frequency_index, pulse_width_index
                ]

        response_values = interpolated.reshape(frequencies.shape)
        if response_values.ndim == 0:
            return float(response_values.item())
        return response_values

    return response_function

def run_clinical_interpolation_optimizer_comparison(
    *,
    anchor_points,
    response_functions,
    dataset_label,
    frequency_bounds_hz=None,
    pulse_width_bounds_us=None,
    smac_output_directory="smac3_output",
    num_simulation_trials=CLINICAL_OPTIMIZER_EVALUATIONS_PER_RUN,
    amount_of_seed_repeats=CLINICAL_OPTIMIZER_REPEATS,
    patient_and_machine_startSeed=10000,
):
    """Compare all four optimizers on supplied Linear and PCHIP clinical maps."""
    required_optimizer_names = (
        "OPTIMIZER_STARTUP_POINTS",
        "BOGP_DEFAULTS",
        "TPE_DEFAULTS",
        "SMAC_DEFAULTS",
        "RANDOM_SEARCH_DEFAULTS",
        "BayesianOptimization",
        "ExpectedImprovement",
        "tpe",
        "fmin",
        "hp",
        "Trials",
        "STATUS_OK",
        "ConfigurationSpace",
        "Float",
        "Scenario",
        "HyperparameterOptimizationFacade",
        "_normalized_regret_percent",
        "OBSERVED_INCUMBENT_POLICY",
        "SMAC_INCUMBENT_POLICY",
        "_suppress_smac_info_logs",
        "_DuplicateNoticeOutputFilter",
        "contextlib",
        "sys",
        "np",
        "pd",
        "Path",
        "FileLink",
        "display",
        "time",
    )
    missing_optimizer_names = [
        name for name in required_optimizer_names if name not in globals()
    ]
    if missing_optimizer_names:
        raise RuntimeError(
            "Run the notebook's optimizer-configuration cells before calling this "
            "comparison. Missing: " + ", ".join(missing_optimizer_names)
        )
    if not isinstance(dataset_label, str) or not dataset_label.strip():
        raise ValueError("dataset_label must be a non-empty string.")
    dataset_slug = "".join(
        character.lower() if character.isalnum() else "_"
        for character in dataset_label.strip()
    ).strip("_")
    if not dataset_slug:
        raise ValueError("dataset_label must contain at least one alphanumeric character.")
    smac_output_directory = Path(smac_output_directory)
    from ConfigSpace import Configuration
    from smac.callback.callback import Callback

    class _ClinicalSMACIncumbentRecorder(Callback):
        """Record SMAC's post-query incumbent in the clinical parameter space."""

        def __init__(self):
            super().__init__()
            self.records = []

        def on_tell_end(self, smbo, info, value):
            incumbent = smbo.intensifier.get_incumbent()
            if incumbent is None:
                raise RuntimeError("SMAC did not select an incumbent after a completed trial.")
            self.records.append(
                {
                    "trial_params": {
                        "frequency": float(info.config["frequency"]),
                        "pulse_width": float(info.config["pulse_width"]),
                    },
                    "incumbent_params": {
                        "frequency": float(incumbent["frequency"]),
                        "pulse_width": float(incumbent["pulse_width"]),
                    },
                    "cost": float(value.cost),
                }
            )

    if isinstance(num_simulation_trials, (bool, np.bool_)) or not isinstance(
        num_simulation_trials, (int, np.integer)
    ):
        raise TypeError("num_simulation_trials must be an integer.")
    if isinstance(amount_of_seed_repeats, (bool, np.bool_)) or not isinstance(
        amount_of_seed_repeats, (int, np.integer)
    ):
        raise TypeError("amount_of_seed_repeats must be an integer.")
    if isinstance(patient_and_machine_startSeed, (bool, np.bool_)) or not isinstance(
        patient_and_machine_startSeed, (int, np.integer)
    ):
        raise TypeError("patient_and_machine_startSeed must be an integer.")
    if amount_of_seed_repeats <= 0:
        raise ValueError("amount_of_seed_repeats must be positive.")
    if patient_and_machine_startSeed < 0:
        raise ValueError("patient_and_machine_startSeed must be non-negative.")

    startup_points = int(OPTIMIZER_STARTUP_POINTS)
    if num_simulation_trials < startup_points:
        raise ValueError(
            f"num_simulation_trials must be at least {startup_points} "
            "to include the configured startup design."
        )
    if num_simulation_trials < 20:
        raise ValueError(
            "num_simulation_trials must be at least 20 to report the Query-20 checkpoint."
        )
    configured_startup_points = {
        "Bayesian Optimization": int(BOGP_DEFAULTS["init_points"]),
        "TPE Optimization": int(TPE_DEFAULTS["n_startup_jobs"]),
        "SMAC Optimization": int(SMAC_DEFAULTS["initial_design"]["n_configs"]),
        "Random Search": startup_points,
    }
    inconsistent_startup_points = {
        method: count
        for method, count in configured_startup_points.items()
        if count != startup_points
    }
    if inconsistent_startup_points:
        raise RuntimeError(
            "Optimizer startup settings do not match OPTIMIZER_STARTUP_POINTS: "
            f"{inconsistent_startup_points}."
        )

    try:
        anchor_values = np.asarray(anchor_points, dtype=float)
    except (TypeError, ValueError) as error:
        raise ValueError(f"{dataset_label} response anchors must be finite numeric values.") from error
    if (
        anchor_values.ndim != 2
        or anchor_values.shape[1] != 3
        or not 1 <= anchor_values.shape[0] <= 10
        or not np.isfinite(anchor_values).all()
    ):
        raise ValueError(
            f"{dataset_label} must have 1 to 10 finite (frequency, pulse width, response) anchors."
        )

    if not isinstance(response_functions, dict):
        raise TypeError("response_functions must map 'Linear' and 'PCHIP' to callables.")
    response_models = (
        ("Linear", response_functions.get("Linear")),
        ("PCHIP", response_functions.get("PCHIP")),
    )
    disabled_models = [
        model_name
        for model_name, response_function in response_models
        if not callable(response_function)
    ]
    if disabled_models:
        raise RuntimeError(
            "Clinical response models are disabled or unavailable: "
            + ", ".join(disabled_models)
        )
    for model_name, response_function in response_models:
        for frequency, pulse_width, expected_response in anchor_values:
            actual_response = float(response_function(frequency, pulse_width))
            if actual_response != float(expected_response):
                raise RuntimeError(
                    f"{model_name} interpolation does not reproduce every "
                    "clinical anchor exactly."
                )

    # Shape-preserving pulse-width interpolation and linear frequency interpolation
    # make the maximum anchor response the maximum response of both clinical maps.
    best_possible_responses = {
        model_name: float(np.max(anchor_values[:, 2]))
        for model_name, _ in response_models
    }
    if any(
        not np.isfinite(score) or score <= 0.0
        for score in best_possible_responses.values()
    ):
        raise ValueError("Normalized clinical regret requires a finite positive best possible response.")

    measured_frequency_bounds = (
        float(np.min(anchor_values[:, 0])),
        float(np.max(anchor_values[:, 0])),
    )
    measured_pulse_width_bounds = (
        float(np.min(anchor_values[:, 1])),
        float(np.max(anchor_values[:, 1])),
    )
    frequency_bounds = (
        measured_frequency_bounds
        if frequency_bounds_hz is None
        else _validate_clinical_map_bounds(frequency_bounds_hz, "frequency_bounds_hz")
    )
    pulse_width_bounds = (
        measured_pulse_width_bounds
        if pulse_width_bounds_us is None
        else _validate_clinical_map_bounds(pulse_width_bounds_us, "pulse_width_bounds_us")
    )
    if (
        not np.allclose(frequency_bounds, measured_frequency_bounds, rtol=0.0, atol=1e-12)
        or not np.allclose(
            pulse_width_bounds, measured_pulse_width_bounds, rtol=0.0, atol=1e-12
        )
    ):
        raise ValueError(
            f"{dataset_label} optimizer bounds must match the measured anchor ranges."
        )
    if frequency_bounds[0] >= frequency_bounds[1] or pulse_width_bounds[0] >= pulse_width_bounds[1]:
        raise ValueError(f"{dataset_label} anchors must span both measured parameters.")
    search_boundaries = {
        "frequency": frequency_bounds,
        "pulse_width": pulse_width_bounds,
    }
    normalized_boundaries = {
        name: (0.0, 1.0) for name in search_boundaries
    }
    method_names = (
        "Bayesian Optimization",
        "TPE Optimization",
        "SMAC Optimization",
        "Random Search",
    )
    model_names = tuple(model_name for model_name, _ in response_models)
    result_rows = []
    convergence_rows = []
    smac_initial_designs_by_seed = {}
    smac_initial_design_queries = {}

    for model_name, response_function in response_models:
        for repeat_index in range(amount_of_seed_repeats):
            run_seed = int(patient_and_machine_startSeed + repeat_index)
            query_counts = {method: 0 for method in method_names}
            best_responses = {method: -np.inf for method in method_names}
            best_parameters = {method: None for method in method_names}

            def evaluate_and_record(method, frequency, pulse_width):
                frequency = float(frequency)
                pulse_width = float(pulse_width)
                response = float(response_function(frequency, pulse_width))
                if not np.isfinite(response):
                    raise ValueError("Clinical response evaluation was not finite.")
                query_index = query_counts[method] + 1
                query_counts[method] = query_index
                if response > best_responses[method]:
                    best_responses[method] = response
                    best_parameters[method] = {
                        "frequency": frequency,
                        "pulse_width": pulse_width,
                    }
                selected_parameters = best_parameters[method]
                convergence_rows.append(
                    {
                        "Interpolation Model": model_name,
                        "Method": method,
                        "Seed": run_seed,
                        "Query": query_index,
                        "Frequency (Hz)": frequency,
                        "Pulse Width (us)": pulse_width,
                        "Response": response,
                        "Cumulative Best Response": best_responses[method],
                        "Incumbent Selection Policy": (
                            SMAC_INCUMBENT_POLICY
                            if method == "SMAC Optimization"
                            else OBSERVED_INCUMBENT_POLICY
                        ),
                        "Selected Incumbent Frequency (Hz)": selected_parameters[
                            "frequency"
                        ],
                        "Selected Incumbent Pulse Width (us)": selected_parameters[
                            "pulse_width"
                        ],
                        "Selected Incumbent Response": best_responses[method],
                    }
                )
                return response

            def append_result(method, best_response, best_parameters, elapsed_seconds):
                result_rows.append(
                    {
                        "Interpolation Model": model_name,
                        "Method": method,
                        "Seed": run_seed,
                        "Iteration Budget": num_simulation_trials,
                        "Best Response": float(best_response),
                        "Selected Frequency (Hz)": float(best_parameters["frequency"]),
                        "Selected Pulse Width (us)": float(best_parameters["pulse_width"]),
                        "Sec To Compute": float(elapsed_seconds),
                    }
                )

            def print_optimizer_progress(method):
                print(
                    f"Running {model_name} | {method} | "
                    f"repeat {repeat_index + 1}/{amount_of_seed_repeats} | "
                    f"seed={run_seed} | evaluations={num_simulation_trials}"
                )

            method = "Bayesian Optimization"
            print_optimizer_progress(method)
            bo_started = time.perf_counter()

            def objective_bo(frequency, pulse_width):
                physical_frequency = frequency_bounds[0] + float(frequency) * (
                    frequency_bounds[1] - frequency_bounds[0]
                )
                physical_pulse_width = pulse_width_bounds[0] + float(pulse_width) * (
                    pulse_width_bounds[1] - pulse_width_bounds[0]
                )
                return evaluate_and_record(
                    method, physical_frequency, physical_pulse_width
                )

            optimizer_bo = BayesianOptimization(
                f=objective_bo,
                pbounds=normalized_boundaries,
                gp_noise_type="normal",
                gp_physical_bounds=search_boundaries,
                acquisition_function=ExpectedImprovement(xi=BOGP_DEFAULTS["xi"]),
                random_state=run_seed,
                verbose=BOGP_DEFAULTS["verbose"],
            )
            duplicate_notice_filter = _DuplicateNoticeOutputFilter(sys.stdout)
            try:
                with contextlib.redirect_stdout(duplicate_notice_filter):
                    optimizer_bo.maximize(
                        init_points=startup_points,
                        n_iter=num_simulation_trials - startup_points,
                    )
            finally:
                duplicate_notice_filter.flush()
            bo_elapsed_seconds = time.perf_counter() - bo_started
            bo_best_parameters = {
                "frequency": frequency_bounds[0]
                + float(optimizer_bo.max["params"]["frequency"])
                * (frequency_bounds[1] - frequency_bounds[0]),
                "pulse_width": pulse_width_bounds[0]
                + float(optimizer_bo.max["params"]["pulse_width"])
                * (pulse_width_bounds[1] - pulse_width_bounds[0]),
            }
            append_result(
                method,
                optimizer_bo.max["target"],
                bo_best_parameters,
                bo_elapsed_seconds,
            )

            method = "TPE Optimization"
            print_optimizer_progress(method)
            tpe_started = time.perf_counter()
            trials = Trials()

            def objective_tpe(parameters):
                response = evaluate_and_record(
                    method,
                    parameters["frequency"],
                    parameters["pulse_width"],
                )
                return {"loss": -response, "status": STATUS_OK}

            fmin(
                fn=objective_tpe,
                space={
                    "frequency": hp.uniform(
                        "frequency", frequency_bounds[0], frequency_bounds[1]
                    ),
                    "pulse_width": hp.uniform(
                        "pulse_width", pulse_width_bounds[0], pulse_width_bounds[1]
                    ),
                },
                algo=tpe.suggest,
                max_evals=num_simulation_trials,
                trials=trials,
                rstate=np.random.default_rng(run_seed),
            )
            tpe_elapsed_seconds = time.perf_counter() - tpe_started
            tpe_best_parameters = {
                "frequency": float(trials.argmin["frequency"]),
                "pulse_width": float(trials.argmin["pulse_width"]),
            }
            append_result(
                method,
                -float(trials.best_trial["result"]["loss"]),
                tpe_best_parameters,
                tpe_elapsed_seconds,
            )

            method = "SMAC Optimization"
            print_optimizer_progress(method)
            smac_started = time.perf_counter()
            configspace = ConfigurationSpace(seed=run_seed)
            configspace.add(
                [
                    Float("frequency", bounds=frequency_bounds),
                    Float("pulse_width", bounds=pulse_width_bounds),
                ]
            )

            def objective_smac(configuration, seed=None):
                return -evaluate_and_record(
                    method,
                    float(configuration["frequency"]),
                    float(configuration["pulse_width"]),
                )

            scenario_defaults = SMAC_DEFAULTS["scenario"]
            scenario_output_directory = (
                smac_output_directory
                / dataset_slug
                / model_name.lower().replace(" ", "_")
                / f"seed_{run_seed}"
            )
            scenario = Scenario(
                configspace,
                output_directory=scenario_output_directory,
                deterministic=scenario_defaults["deterministic"],
                objectives=scenario_defaults["objectives"],
                use_default_config=scenario_defaults["use_default_config"],
                n_workers=scenario_defaults["n_workers"],
                n_trials=num_simulation_trials,
                seed=run_seed,
            )
            if run_seed not in smac_initial_designs_by_seed:
                with _suppress_smac_info_logs():
                    generated_initial_design = HyperparameterOptimizationFacade.get_initial_design(
                        scenario,
                        n_configs=SMAC_DEFAULTS["initial_design"]["n_configs"],
                        max_ratio=SMAC_DEFAULTS["initial_design"]["max_ratio"],
                        additional_configs=SMAC_DEFAULTS["initial_design"][
                            "additional_configs"
                        ],
                    )
                    generated_startup_configurations = (
                        generated_initial_design.select_configurations()
                    )
                if len(generated_startup_configurations) != configured_startup_points[
                    "SMAC Optimization"
                ]:
                    raise RuntimeError(
                        f"SMAC generated {len(generated_startup_configurations)} "
                        "startup configurations instead of the configured "
                        f"{configured_startup_points['SMAC Optimization']}."
                    )
                smac_initial_designs_by_seed[run_seed] = [
                    dict(configuration)
                    for configuration in generated_startup_configurations
                ]
            smac_startup_configurations = [
                Configuration(configspace, values=configuration_values)
                for configuration_values in smac_initial_designs_by_seed[run_seed]
            ]
            smac_initial_design_queries[(model_name, run_seed)] = np.asarray(
                [
                    [configuration["frequency"], configuration["pulse_width"]]
                    for configuration in smac_startup_configurations
                ],
                dtype=float,
            )
            with _suppress_smac_info_logs():
                smac_initial_design = HyperparameterOptimizationFacade.get_initial_design(
                    scenario,
                    n_configs=0,
                    max_ratio=SMAC_DEFAULTS["initial_design"]["max_ratio"],
                    additional_configs=smac_startup_configurations,
                )
            smac_model = HyperparameterOptimizationFacade.get_model(
                scenario,
                n_trees=SMAC_DEFAULTS["model"]["n_trees"],
                ratio_features=SMAC_DEFAULTS["model"]["ratio_features"],
                min_samples_split=SMAC_DEFAULTS["model"]["min_samples_split"],
                min_samples_leaf=SMAC_DEFAULTS["model"]["min_samples_leaf"],
                max_depth=SMAC_DEFAULTS["model"]["max_depth"],
                bootstrapping=SMAC_DEFAULTS["model"]["bootstrapping"],
            )
            # The facade stores prediction parallelism in the random-forest options.
            smac_model_options = getattr(smac_model, "_rf_opts", None)
            if not isinstance(smac_model_options, dict) or "n_jobs" not in smac_model_options:
                raise RuntimeError(
                    "SMAC's random-forest model does not expose its n_jobs option; "
                    "cannot disable parallel prediction workers."
                )
            smac_model_options["n_jobs"] = 1
            smac_incumbents = _ClinicalSMACIncumbentRecorder()
            with _suppress_smac_info_logs():
                smac = HyperparameterOptimizationFacade(
                    scenario,
                    objective_smac,
                    model=smac_model,
                    initial_design=smac_initial_design,
                    callbacks=[smac_incumbents],
                )
                incumbent = smac.optimize()
                smac_best_response = -float(smac.runhistory.get_cost(incumbent))
            smac_elapsed_seconds = time.perf_counter() - smac_started
            smac_best_parameters = {
                "frequency": float(incumbent["frequency"]),
                "pulse_width": float(incumbent["pulse_width"]),
            }
            smac_rows = [
                row
                for row in convergence_rows
                if row["Interpolation Model"] == model_name
                and row["Method"] == method
                and row["Seed"] == run_seed
            ]
            if (
                len(smac_rows) != len(smac_incumbents.records)
                or int(smac.runhistory.finished) != len(smac_rows)
            ):
                raise RuntimeError(
                    "Cannot align SMAC's incumbent trajectory with its completed "
                    "clinical evaluations."
                )
            for query, (row, record) in enumerate(
                zip(smac_rows, smac_incumbents.records), start=1
            ):
                trial_parameters = record["trial_params"]
                recorded_trial = np.asarray(
                    [row["Frequency (Hz)"], row["Pulse Width (us)"]], dtype=float
                )
                callback_trial = np.asarray(
                    [trial_parameters["frequency"], trial_parameters["pulse_width"]],
                    dtype=float,
                )
                if (
                    row["Query"] != query
                    or not np.allclose(recorded_trial, callback_trial, rtol=0.0, atol=1e-12)
                    or not np.isclose(
                        row["Response"], -record["cost"], rtol=1e-12, atol=1e-12
                    )
                ):
                    raise RuntimeError(
                        "SMAC trial records do not match the clinical evaluation records."
                    )
                incumbent_parameters = record["incumbent_params"]
                incumbent_response = float(
                    response_function(
                        incumbent_parameters["frequency"],
                        incumbent_parameters["pulse_width"],
                    )
                )
                if not np.isfinite(incumbent_response):
                    raise ValueError("SMAC selected a non-finite clinical response.")
                row["Selected Incumbent Frequency (Hz)"] = incumbent_parameters[
                    "frequency"
                ]
                row["Selected Incumbent Pulse Width (us)"] = incumbent_parameters[
                    "pulse_width"
                ]
                row["Selected Incumbent Response"] = incumbent_response
            if (
                not np.allclose(
                    [
                        smac_rows[-1]["Selected Incumbent Frequency (Hz)"],
                        smac_rows[-1]["Selected Incumbent Pulse Width (us)"],
                    ],
                    [smac_best_parameters["frequency"], smac_best_parameters["pulse_width"]],
                    rtol=0.0,
                    atol=1e-12,
                )
                or not np.isclose(
                    smac_rows[-1]["Selected Incumbent Response"],
                    smac_best_response,
                    rtol=1e-12,
                    atol=1e-12,
                )
            ):
                raise RuntimeError(
                    "SMAC's final recorded incumbent does not match its returned incumbent."
                )
            append_result(
                method,
                smac_best_response,
                smac_best_parameters,
                smac_elapsed_seconds,
            )

            method = "Random Search"
            print_optimizer_progress(method)
            random_search_started = time.perf_counter()
            random_search_rng = np.random.default_rng(run_seed)
            random_search_best_response = -np.inf
            random_search_best_parameters = None
            for _ in range(num_simulation_trials):
                parameters = {
                    name: random_search_rng.uniform(lower, upper)
                    for name, (lower, upper) in search_boundaries.items()
                }
                response = evaluate_and_record(
                    method,
                    parameters["frequency"],
                    parameters["pulse_width"],
                )
                if response > random_search_best_response:
                    random_search_best_response = response
                    random_search_best_parameters = parameters
            random_search_elapsed_seconds = time.perf_counter() - random_search_started
            append_result(
                method,
                random_search_best_response,
                random_search_best_parameters,
                random_search_elapsed_seconds,
            )

            incomplete_query_counts = {
                method: query_counts[method]
                for method in method_names
                if query_counts[method] != num_simulation_trials
            }
            if incomplete_query_counts:
                raise RuntimeError(
                    f"Optimizer evaluations differ from the requested budget of "
                    f"{num_simulation_trials}: {incomplete_query_counts}."
                )

    comparison_results = pd.DataFrame(result_rows)
    convergence_results = pd.DataFrame(convergence_rows)
    expected_run_count = len(response_models) * len(method_names) * amount_of_seed_repeats
    if len(comparison_results) != expected_run_count:
        raise RuntimeError(
            f"Expected {expected_run_count} optimizer summaries, "
            f"received {len(comparison_results)}."
        )
    expected_evaluation_count = expected_run_count * num_simulation_trials
    if len(convergence_results) != expected_evaluation_count:
        raise RuntimeError(
            f"Expected {expected_evaluation_count} recorded evaluations, "
            f"received {len(convergence_results)}."
        )

    convergence_results["Best Possible Response"] = convergence_results[
        "Interpolation Model"
    ].map(best_possible_responses)
    selected_responses = pd.to_numeric(
        convergence_results["Selected Incumbent Response"], errors="coerce"
    )
    if not np.isfinite(selected_responses).all():
        raise RuntimeError("Clinical incumbent trajectories contain invalid response values.")
    convergence_results["Selected Incumbent Normalized Regret (%)"] = [
        _normalized_regret_percent(best_possible, response)
        for best_possible, response in zip(
            convergence_results["Best Possible Response"], selected_responses
        )
    ]
    if not np.isfinite(
        convergence_results["Selected Incumbent Normalized Regret (%)"]
    ).all():
        raise RuntimeError("Clinical normalized-regret calculations were not finite.")

    query_20_rows = convergence_results.loc[convergence_results["Query"] == 20]
    if len(query_20_rows) != expected_run_count:
        raise RuntimeError(
            f"Expected one Query-20 record for each of {expected_run_count} runs, "
            f"received {len(query_20_rows)}."
        )
    final_incumbent_rows = convergence_results.loc[
        convergence_results["Query"] == num_simulation_trials,
        [
            "Interpolation Model",
            "Method",
            "Seed",
            "Selected Incumbent Response",
        ],
    ].merge(
        comparison_results[
            ["Interpolation Model", "Method", "Seed", "Best Response"]
        ],
        on=["Interpolation Model", "Method", "Seed"],
        how="left",
        validate="one_to_one",
    )
    if (
        len(final_incumbent_rows) != expected_run_count
        or not np.allclose(
            final_incumbent_rows["Selected Incumbent Response"],
            final_incumbent_rows["Best Response"],
            rtol=1e-12,
            atol=1e-12,
        )
    ):
        raise RuntimeError(
            "Final convergence incumbents do not match the reported optimizer results."
        )

    for run_seed in range(
        patient_and_machine_startSeed,
        patient_and_machine_startSeed + amount_of_seed_repeats,
    ):
        for method in method_names:
            initial_queries = []
            for model_name in model_names:
                if method == "SMAC Optimization":
                    model_initial_queries = smac_initial_design_queries[
                        (model_name, run_seed)
                    ]
                else:
                    model_initial_queries = convergence_results.loc[
                        (convergence_results["Interpolation Model"] == model_name)
                        & (convergence_results["Method"] == method)
                        & (convergence_results["Seed"] == run_seed)
                        & (convergence_results["Query"] <= configured_startup_points[method]),
                        ["Frequency (Hz)", "Pulse Width (us)"],
                    ].to_numpy(dtype=float)
                if model_initial_queries.shape != (
                    configured_startup_points[method],
                    2,
                ):
                    raise RuntimeError(
                        f"{method} did not record its configured startup design "
                        f"for {model_name}, seed {run_seed}."
                    )
                initial_queries.append(model_initial_queries)
            if not np.allclose(initial_queries[0], initial_queries[1], rtol=0.0, atol=1e-12):
                raise RuntimeError(
                    f"Paired models received different {method} startup queries "
                    f"for seed {run_seed}."
                )

    return comparison_results, convergence_results

def analyze_clinical_optimizer_statistics(
    results,
    convergence,
    anchor_points,
    response_functions,
    dataset_label,
    output_directory,
):
    """Analyze one dataset's matched simulation runs without pooling datasets."""
    if "OPTIMIZER_STARTUP_POINTS" not in globals():
        raise RuntimeError("Run the optimizer-configuration cell before statistical analysis.")
    if not isinstance(dataset_label, str) or not dataset_label.strip():
        raise ValueError("dataset_label must be a non-empty string.")
    CLINICAL_STATS_METHODS = (
        "Bayesian Optimization",
        "TPE Optimization",
        "SMAC Optimization",
        "Random Search",
    )
    CLINICAL_STATS_MODELS = ("Linear", "PCHIP")
    CLINICAL_STATS_BUDGET = 40
    CLINICAL_STATS_N_SEEDS = 40
    CLINICAL_STATS_FIRST_SEED = 10000
    CLINICAL_STATS_N_PERMUTATIONS = 20000
    CLINICAL_STATS_N_BOOTSTRAP = 10000
    CLINICAL_STATS_ANALYSIS_SEED = 20261103
    CLINICAL_STATS_TIE_TOLERANCE = 1e-12

    if int(CLINICAL_OPTIMIZER_EVALUATIONS_PER_RUN) != CLINICAL_STATS_BUDGET:
        raise RuntimeError("Clinical statistical analysis requires a 40-query budget.")
    if int(CLINICAL_OPTIMIZER_REPEATS) != CLINICAL_STATS_N_SEEDS:
        raise RuntimeError("Clinical statistical analysis requires 40 seed runs.")
    if results is None or convergence is None:
        raise ValueError("Both optimizer result and convergence tables are required.")

    CLINICAL_STATS_RESULTS = pd.DataFrame(results).copy()
    CLINICAL_STATS_CONVERGENCE = pd.DataFrame(convergence).copy()
    CLINICAL_STATS_RUN_KEYS = ["Interpolation Model", "Method", "Seed"]
    CLINICAL_STATS_SEEDS = np.arange(
        CLINICAL_STATS_FIRST_SEED,
        CLINICAL_STATS_FIRST_SEED + CLINICAL_STATS_N_SEEDS,
        dtype=np.int64,
    )
    CLINICAL_STATS_QUERIES = np.arange(1, CLINICAL_STATS_BUDGET + 1, dtype=np.int64)

    CLINICAL_STATS_REQUIRED_RESULT_COLUMNS = set(CLINICAL_STATS_RUN_KEYS) | {
        "Iteration Budget", "Best Response", "Selected Frequency (Hz)",
        "Selected Pulse Width (us)",
    }
    CLINICAL_STATS_REQUIRED_CONVERGENCE_COLUMNS = set(CLINICAL_STATS_RUN_KEYS) | {
        "Query", "Response", "Frequency (Hz)", "Pulse Width (us)",
        "Cumulative Best Response", "Selected Incumbent Response",
        "Selected Incumbent Frequency (Hz)", "Selected Incumbent Pulse Width (us)",
        "Best Possible Response",
        "Selected Incumbent Normalized Regret (%)",
    }
    missing_result_columns = (
        CLINICAL_STATS_REQUIRED_RESULT_COLUMNS - set(CLINICAL_STATS_RESULTS.columns)
    )
    missing_convergence_columns = (
        CLINICAL_STATS_REQUIRED_CONVERGENCE_COLUMNS
        - set(CLINICAL_STATS_CONVERGENCE.columns)
    )
    if missing_result_columns or missing_convergence_columns:
        raise RuntimeError(
            "Clinical result tables are missing required columns: "
            f"results={sorted(missing_result_columns)}, "
            f"convergence={sorted(missing_convergence_columns)}."
        )

    try:
        CLINICAL_STATS_ANCHORS = np.asarray(anchor_points, dtype=float)
    except (TypeError, ValueError) as error:
        raise ValueError(f"{dataset_label} anchors must be finite numeric values.") from error
    if (
        CLINICAL_STATS_ANCHORS.ndim != 2
        or CLINICAL_STATS_ANCHORS.shape[1] != 3
        or not 1 <= CLINICAL_STATS_ANCHORS.shape[0] <= 10
        or not np.isfinite(CLINICAL_STATS_ANCHORS).all()
    ):
        raise ValueError(f"{dataset_label} must have 1 to 10 finite clinical anchor rows.")
    CLINICAL_STATS_FREQUENCY_GRID = np.unique(CLINICAL_STATS_ANCHORS[:, 0])
    CLINICAL_STATS_PULSE_WIDTH_GRID = np.unique(CLINICAL_STATS_ANCHORS[:, 1])
    if (
        len(CLINICAL_STATS_FREQUENCY_GRID) < 2
        or len(CLINICAL_STATS_PULSE_WIDTH_GRID) < 2
        or len(CLINICAL_STATS_FREQUENCY_GRID) * len(CLINICAL_STATS_PULSE_WIDTH_GRID)
        != len(CLINICAL_STATS_ANCHORS)
        or np.unique(CLINICAL_STATS_ANCHORS[:, :2], axis=0).shape[0]
        != len(CLINICAL_STATS_ANCHORS)
    ):
        raise ValueError(f"{dataset_label} PCHIP analysis requires a complete rectangular anchor grid.")
    CLINICAL_STATS_REFERENCE = float(CLINICAL_STATS_ANCHORS[:, 2].max())
    CLINICAL_STATS_REFERENCE_POINT = CLINICAL_STATS_ANCHORS[
        int(np.argmax(CLINICAL_STATS_ANCHORS[:, 2])), :2
    ]

    def _clinical_stats_regret(best_possible_response, response):
        best_possible_response = float(best_possible_response)
        response = np.asarray(response, dtype=float)
        if not np.isfinite(best_possible_response) or best_possible_response <= 0:
            raise ValueError("Normalized regret requires a finite positive reference.")
        if not np.isfinite(response).all():
            raise ValueError("Normalized regret cannot use non-finite responses.")
        return 100.0 * (best_possible_response - response) / best_possible_response

    CLINICAL_STATS_REGRET_TEST = _clinical_stats_regret(
        CLINICAL_STATS_REFERENCE,
        np.asarray([CLINICAL_STATS_REFERENCE, CLINICAL_STATS_REFERENCE / 2.0]),
    )
    assert CLINICAL_STATS_REGRET_TEST[0] == 0.0
    assert np.isclose(CLINICAL_STATS_REGRET_TEST[1], 50.0, rtol=0.0, atol=1e-12)

    CLINICAL_STATS_FREQUENCY_GRID = np.unique(CLINICAL_STATS_ANCHORS[:, 0])
    CLINICAL_STATS_PULSE_WIDTH_GRID = np.unique(CLINICAL_STATS_ANCHORS[:, 1])
    CLINICAL_STATS_FREQUENCIES, CLINICAL_STATS_PULSE_WIDTHS = np.meshgrid(
        np.linspace(CLINICAL_STATS_FREQUENCY_GRID.min(), CLINICAL_STATS_FREQUENCY_GRID.max(), 481),
        np.linspace(CLINICAL_STATS_PULSE_WIDTH_GRID.min(), CLINICAL_STATS_PULSE_WIDTH_GRID.max(), 401),
        indexing="ij",
    )
    if not isinstance(response_functions, dict):
        raise TypeError("response_functions must map 'Linear' and 'PCHIP' to callables.")
    CLINICAL_STATS_RESPONSE_FUNCTIONS = {
        "Linear": response_functions.get("Linear"),
        "PCHIP": response_functions.get("PCHIP"),
    }
    if any(not callable(function) for function in CLINICAL_STATS_RESPONSE_FUNCTIONS.values()):
        raise ValueError("Both Linear and PCHIP response functions are required for analysis.")
    for model_name, response_function in CLINICAL_STATS_RESPONSE_FUNCTIONS.items():
        anchor_responses = np.asarray(
            response_function(CLINICAL_STATS_ANCHORS[:, 0], CLINICAL_STATS_ANCHORS[:, 1]),
            dtype=float,
        )
        if not np.allclose(
            anchor_responses, CLINICAL_STATS_ANCHORS[:, 2], rtol=0.0, atol=1e-12
        ):
            raise RuntimeError(f"{model_name} does not reproduce all anchors exactly.")
        response_grid = np.asarray(
            response_function(CLINICAL_STATS_FREQUENCIES, CLINICAL_STATS_PULSE_WIDTHS),
            dtype=float,
        )
        if not np.isfinite(response_grid).all():
            raise RuntimeError(f"{model_name} returned non-finite domain responses.")
        if float(response_grid.max()) > CLINICAL_STATS_REFERENCE + 1e-12:
            raise RuntimeError(f"{model_name} exceeds the anchor reference maximum.")
        reference_response = float(response_function(*CLINICAL_STATS_REFERENCE_POINT))
        if not np.isclose(
            reference_response, CLINICAL_STATS_REFERENCE, rtol=0.0, atol=1e-12
        ):
            raise RuntimeError(f"{model_name} does not attain the reference optimum.")

    CLINICAL_STATS_RESULTS["Seed"] = pd.to_numeric(
        CLINICAL_STATS_RESULTS["Seed"], errors="raise"
    ).astype("int64")
    CLINICAL_STATS_CONVERGENCE["Seed"] = pd.to_numeric(
        CLINICAL_STATS_CONVERGENCE["Seed"], errors="raise"
    ).astype("int64")
    for frame, column in (
        (CLINICAL_STATS_RESULTS, "Iteration Budget"),
        (CLINICAL_STATS_RESULTS, "Best Response"),
        (CLINICAL_STATS_RESULTS, "Selected Frequency (Hz)"),
        (CLINICAL_STATS_RESULTS, "Selected Pulse Width (us)"),
        (CLINICAL_STATS_CONVERGENCE, "Query"),
        (CLINICAL_STATS_CONVERGENCE, "Response"),
        (CLINICAL_STATS_CONVERGENCE, "Frequency (Hz)"),
        (CLINICAL_STATS_CONVERGENCE, "Pulse Width (us)"),
        (CLINICAL_STATS_CONVERGENCE, "Cumulative Best Response"),
        (CLINICAL_STATS_CONVERGENCE, "Selected Incumbent Response"),
        (CLINICAL_STATS_CONVERGENCE, "Selected Incumbent Frequency (Hz)"),
        (CLINICAL_STATS_CONVERGENCE, "Selected Incumbent Pulse Width (us)"),
        (CLINICAL_STATS_CONVERGENCE, "Best Possible Response"),
        (CLINICAL_STATS_CONVERGENCE, "Selected Incumbent Normalized Regret (%)"),
    ):
        frame[column] = pd.to_numeric(frame[column], errors="raise")

    expected_result_keys = {
        (model_name, method, int(seed))
        for model_name in CLINICAL_STATS_MODELS
        for method in CLINICAL_STATS_METHODS
        for seed in CLINICAL_STATS_SEEDS
    }
    actual_result_keys = set(
        CLINICAL_STATS_RESULTS[CLINICAL_STATS_RUN_KEYS].itertuples(
            index=False, name=None
        )
    )
    if (
        len(CLINICAL_STATS_RESULTS) != len(expected_result_keys)
        or CLINICAL_STATS_RESULTS.duplicated(CLINICAL_STATS_RUN_KEYS).any()
        or actual_result_keys != expected_result_keys
    ):
        raise RuntimeError(
            "Results lack exactly one run per model, optimizer, and expected seed; "
            "matched inference is invalid."
        )
    if set(CLINICAL_STATS_RESULTS["Iteration Budget"].astype(int)) != {
        CLINICAL_STATS_BUDGET
    }:
        raise RuntimeError("The four optimizers do not share the required 40-query budget.")

    expected_convergence_rows = len(expected_result_keys) * CLINICAL_STATS_BUDGET
    if (
        len(CLINICAL_STATS_CONVERGENCE) != expected_convergence_rows
        or CLINICAL_STATS_CONVERGENCE.duplicated(
            CLINICAL_STATS_RUN_KEYS + ["Query"]
        ).any()
    ):
        raise RuntimeError("The convergence table has missing or duplicate queries.")
    for run_key, run_rows in CLINICAL_STATS_CONVERGENCE.groupby(
        CLINICAL_STATS_RUN_KEYS, sort=False
    ):
        query_ids = np.sort(run_rows["Query"].to_numpy(dtype=np.int64))
        if not np.array_equal(query_ids, CLINICAL_STATS_QUERIES):
            raise RuntimeError(f"Run {run_key} does not contain queries 1 through 40.")

    CLINICAL_STATS_CONVERGENCE = CLINICAL_STATS_CONVERGENCE.sort_values(
        CLINICAL_STATS_RUN_KEYS + ["Query"]
    ).reset_index(drop=True)
    for model_name, response_function in CLINICAL_STATS_RESPONSE_FUNCTIONS.items():
        model_rows = CLINICAL_STATS_CONVERGENCE.loc[
            CLINICAL_STATS_CONVERGENCE["Interpolation Model"] == model_name
        ]
        reevaluated_responses = np.asarray(
            response_function(
                model_rows["Frequency (Hz)"].to_numpy(dtype=float),
                model_rows["Pulse Width (us)"].to_numpy(dtype=float),
            ),
            dtype=float,
        )
        if not np.allclose(
            reevaluated_responses,
            model_rows["Response"].to_numpy(dtype=float),
            rtol=0.0,
            atol=1e-12,
        ):
            raise RuntimeError(
                f"{model_name} query responses do not match its deterministic surface."
            )

    CLINICAL_STATS_CONVERGENCE["Clinical Stats Best-So-Far Response"] = (
        CLINICAL_STATS_CONVERGENCE.groupby(
            CLINICAL_STATS_RUN_KEYS, sort=False
        )["Response"].cummax()
    )
    if not np.allclose(
        CLINICAL_STATS_CONVERGENCE["Cumulative Best Response"],
        CLINICAL_STATS_CONVERGENCE["Clinical Stats Best-So-Far Response"],
        rtol=0.0,
        atol=1e-12,
    ):
        raise RuntimeError("Stored cumulative response differs from the query-history maximum.")
    CLINICAL_STATS_CONVERGENCE["Clinical Stats Normalized Regret (%)"] = (
        _clinical_stats_regret(
            CLINICAL_STATS_REFERENCE,
            CLINICAL_STATS_CONVERGENCE[
                "Clinical Stats Best-So-Far Response"
            ].to_numpy(dtype=float),
        )
    )
    stored_incumbent_regret = _clinical_stats_regret(
        CLINICAL_STATS_REFERENCE,
        CLINICAL_STATS_CONVERGENCE["Selected Incumbent Response"].to_numpy(dtype=float),
    )
    if not np.allclose(
        CLINICAL_STATS_CONVERGENCE["Selected Incumbent Normalized Regret (%)"],
        stored_incumbent_regret,
        rtol=0.0,
        atol=1e-12,
    ):
        raise RuntimeError("Stored optimizer-incumbent normalized regret is inconsistent.")

    CLINICAL_STATS_CONVERGENCE[
        "Clinical Stats Best-So-Far Frequency (Hz)"
    ] = np.nan
    CLINICAL_STATS_CONVERGENCE[
        "Clinical Stats Best-So-Far Pulse Width (us)"
    ] = np.nan
    for _, run_rows in CLINICAL_STATS_CONVERGENCE.groupby(
        CLINICAL_STATS_RUN_KEYS, sort=False
    ):
        is_best_so_far = run_rows["Response"].to_numpy(dtype=float) == run_rows[
            "Clinical Stats Best-So-Far Response"
        ].to_numpy(dtype=float)
        best_positions = np.maximum.accumulate(
            np.where(is_best_so_far, np.arange(len(run_rows)), 0)
        )
        CLINICAL_STATS_CONVERGENCE.loc[
            run_rows.index, "Clinical Stats Best-So-Far Frequency (Hz)"
        ] = run_rows["Frequency (Hz)"].to_numpy(dtype=float)[best_positions]
        CLINICAL_STATS_CONVERGENCE.loc[
            run_rows.index, "Clinical Stats Best-So-Far Pulse Width (us)"
        ] = run_rows["Pulse Width (us)"].to_numpy(dtype=float)[best_positions]
    if (
        CLINICAL_STATS_CONVERGENCE["Best Possible Response"]
        - CLINICAL_STATS_REFERENCE
    ).abs().max() > 1e-12:
        raise RuntimeError("Clinical runs used a reference other than the anchor maximum.")
    if (
        CLINICAL_STATS_CONVERGENCE["Clinical Stats Normalized Regret (%)"] < -1e-10
    ).any():
        raise RuntimeError("A query response exceeds the validated global reference.")

    CLINICAL_STATS_QUERY40 = CLINICAL_STATS_CONVERGENCE.loc[
        CLINICAL_STATS_CONVERGENCE["Query"] == CLINICAL_STATS_BUDGET
    ].copy()
    CLINICAL_STATS_QUERY40 = CLINICAL_STATS_QUERY40.rename(
        columns={"Clinical Stats Normalized Regret (%)": "Normalized Regret (%)"}
    )
    CLINICAL_STATS_INCUMBENT_DISAGREE_ROWS = int(
        (
            ~np.isclose(
                CLINICAL_STATS_CONVERGENCE["Selected Incumbent Response"],
                CLINICAL_STATS_CONVERGENCE[
                    "Clinical Stats Best-So-Far Response"
                ],
                rtol=0.0,
                atol=1e-12,
            )
        ).sum()
    )
    CLINICAL_STATS_INCUMBENT_DISAGREE_Q40 = int(
        (
            ~np.isclose(
                CLINICAL_STATS_QUERY40["Selected Incumbent Response"],
                CLINICAL_STATS_QUERY40[
                    "Clinical Stats Best-So-Far Response"
                ],
                rtol=0.0,
                atol=1e-12,
            )
        ).sum()
    )
    CLINICAL_STATS_FINAL_CHECK = CLINICAL_STATS_RESULTS.merge(
        CLINICAL_STATS_QUERY40[
            CLINICAL_STATS_RUN_KEYS + ["Selected Incumbent Response"]
        ],
        on=CLINICAL_STATS_RUN_KEYS,
        how="inner",
        validate="one_to_one",
    )
    if not np.allclose(
        CLINICAL_STATS_FINAL_CHECK["Best Response"],
        CLINICAL_STATS_FINAL_CHECK["Selected Incumbent Response"],
        rtol=0.0,
        atol=1e-12,
    ):
        raise RuntimeError("Query-40 incumbents do not match the optimizer summaries.")

    CLINICAL_STATS_Q40_WIDE = {}
    for model_name in CLINICAL_STATS_MODELS:
        model_rows = CLINICAL_STATS_QUERY40.loc[
            CLINICAL_STATS_QUERY40["Interpolation Model"] == model_name
        ]
        model_wide = model_rows.pivot(
            index="Seed", columns="Method", values="Normalized Regret (%)"
        ).reindex(index=CLINICAL_STATS_SEEDS, columns=CLINICAL_STATS_METHODS)
        if model_wide.shape != (CLINICAL_STATS_N_SEEDS, len(CLINICAL_STATS_METHODS)):
            raise RuntimeError(f"{model_name} lacks 40 complete matched seed blocks.")
        if model_wide.isna().any().any():
            raise RuntimeError(f"{model_name} has an incomplete matched optimizer block.")
        CLINICAL_STATS_Q40_WIDE[model_name] = model_wide.astype(float)
    CLINICAL_STATS_QUERY10 = CLINICAL_STATS_CONVERGENCE.loc[
        CLINICAL_STATS_CONVERGENCE["Query"] == 10
    ].copy()
    CLINICAL_STATS_Q10_WIDE = {}
    for model_name in CLINICAL_STATS_MODELS:
        model_rows = CLINICAL_STATS_QUERY10.loc[
            CLINICAL_STATS_QUERY10["Interpolation Model"] == model_name
        ]
        model_wide = model_rows.pivot(
            index="Seed",
            columns="Method",
            values="Clinical Stats Normalized Regret (%)",
        ).reindex(
            index=CLINICAL_STATS_SEEDS,
            columns=CLINICAL_STATS_METHODS,
        )
        if model_wide.shape != (
            CLINICAL_STATS_N_SEEDS, len(CLINICAL_STATS_METHODS)
        ):
            raise RuntimeError(
                f"{model_name} lacks 40 complete matched query-10 seed blocks."
            )
        if model_wide.isna().any().any():
            raise RuntimeError(
                f"{model_name} has an incomplete query-10 optimizer block."
            )
        if not np.array_equal(
            model_wide.index.to_numpy(dtype=np.int64), CLINICAL_STATS_SEEDS
        ):
            raise RuntimeError(
                f"{model_name} query-10 rows are not aligned by seed."
            )
        CLINICAL_STATS_Q10_WIDE[model_name] = model_wide.astype(float)
    if not CLINICAL_STATS_Q10_WIDE["Linear"].index.equals(
        CLINICAL_STATS_Q10_WIDE["PCHIP"].index
    ):
        raise RuntimeError("Linear and PCHIP query-10 seed blocks do not match.")

    CLINICAL_STATS_GLOBAL_BUDGETS = (40, 10)
    CLINICAL_STATS_GLOBAL_BUDGET_ROLES = {40: "Primary", 10: "Exploratory"}
    CLINICAL_STATS_GLOBAL_WIDES = {}
    for query_budget, model_wides in (
        (40, CLINICAL_STATS_Q40_WIDE),
        (10, CLINICAL_STATS_Q10_WIDE),
    ):
        linear_wide = model_wides["Linear"]
        pchip_wide = model_wides["PCHIP"]
        if not linear_wide.index.equals(pchip_wide.index):
            raise RuntimeError(
                f"Linear and PCHIP query-{query_budget} seed blocks do not match."
            )
        global_wide = ((linear_wide + pchip_wide) / 2.0).reindex(
            index=CLINICAL_STATS_SEEDS, columns=CLINICAL_STATS_METHODS
        )
        if (
            global_wide.shape != (CLINICAL_STATS_N_SEEDS, len(CLINICAL_STATS_METHODS))
            or global_wide.isna().any().any()
            or not np.isfinite(global_wide.to_numpy(dtype=float)).all()
        ):
            raise RuntimeError(
                f"Query-{query_budget} global seed blocks are incomplete or non-finite."
            )
        CLINICAL_STATS_GLOBAL_WIDES[query_budget] = global_wide.astype(float)
    if not np.array_equal(
        CLINICAL_STATS_GLOBAL_WIDES[40].index.to_numpy(dtype=np.int64),
        CLINICAL_STATS_GLOBAL_WIDES[10].index.to_numpy(dtype=np.int64),
    ):
        raise RuntimeError("Query-10 and query-40 global seed blocks do not match.")

    CLINICAL_STATS_HOLM_TEST = multipletests([0.01, 0.04, 0.03], method="holm")[1]
    assert np.allclose(
        CLINICAL_STATS_HOLM_TEST, [0.03, 0.06, 0.06], rtol=0.0, atol=1e-12
    )

    def _clinical_stats_sign_flip(differences, seed):
        differences = np.asarray(differences, dtype=float)
        if differences.ndim != 1 or len(differences) != CLINICAL_STATS_N_SEEDS:
            raise ValueError("Sign-flip tests require one difference per seed block.")
        if not np.isfinite(differences).all():
            raise ValueError("Sign-flip tests cannot use non-finite differences.")
        observed_difference = float(differences.mean())
        if np.all(differences == 0.0):
            return observed_difference, 1.0
        rng = np.random.default_rng(seed)
        signs = rng.choice(
            np.array([-1.0, 1.0]),
            size=(CLINICAL_STATS_N_PERMUTATIONS, len(differences)),
        )
        permuted_means = np.mean(signs * differences[None, :], axis=1)
        extreme_count = np.count_nonzero(
            np.abs(permuted_means) >= abs(observed_difference)
        )
        return observed_difference, float(
            (1 + extreme_count) / (CLINICAL_STATS_N_PERMUTATIONS + 1)
        )

    def _clinical_stats_paired_friedman(wide):
        values = np.asarray(wide, dtype=float)
        if values.shape != (CLINICAL_STATS_N_SEEDS, len(CLINICAL_STATS_METHODS)):
            raise ValueError("Global Friedman tests require a complete seed-by-optimizer matrix.")
        if not np.isfinite(values).all():
            raise ValueError("Global Friedman tests cannot use non-finite regrets.")
        if np.allclose(values, values[:, :1], rtol=1e-12, atol=1e-12):
            return 0.0, 1.0
        result = stats.friedmanchisquare(
            *[values[:, index] for index in range(values.shape[1])]
        )
        statistic, p_value = float(result.statistic), float(result.pvalue)
        if not np.isfinite([statistic, p_value]).all():
            raise RuntimeError("Global Friedman test was undefined.")
        return statistic, p_value

    def _clinical_stats_paired_range_permutation(wide, seed):
        values = np.asarray(wide, dtype=float)
        if values.shape != (CLINICAL_STATS_N_SEEDS, len(CLINICAL_STATS_METHODS)):
            raise ValueError("Range permutations require a complete seed-by-optimizer matrix.")
        if not np.isfinite(values).all():
            raise ValueError("Range permutations cannot use non-finite regrets.")
        observed_range = float(np.ptp(values.mean(axis=0)))
        if np.allclose(values, values[:, :1], rtol=1e-12, atol=1e-12):
            return observed_range, 1.0
        rng = np.random.default_rng(seed)
        permuted_ranges = np.empty(CLINICAL_STATS_N_PERMUTATIONS, dtype=float)
        for permutation_index in range(CLINICAL_STATS_N_PERMUTATIONS):
            permuted_values = np.asarray(
                [rng.permutation(seed_values) for seed_values in values], dtype=float
            )
            permuted_ranges[permutation_index] = np.ptp(
                permuted_values.mean(axis=0)
            )
        extreme_count = np.count_nonzero(permuted_ranges >= observed_range)
        return observed_range, float(
            (1 + extreme_count) / (CLINICAL_STATS_N_PERMUTATIONS + 1)
        )

    def _clinical_stats_bootstrap_paired_difference(
        values_a, values_b, seed, n_resamples=CLINICAL_STATS_N_BOOTSTRAP
    ):
        values_a = np.asarray(values_a, dtype=float)
        values_b = np.asarray(values_b, dtype=float)
        if values_a.shape != values_b.shape or values_a.ndim != 1 or len(values_a) < 2:
            raise ValueError("Paired bootstrap inputs must be aligned one-dimensional runs.")
        if not np.isfinite(values_a).all() or not np.isfinite(values_b).all():
            raise ValueError("Paired bootstrap inputs must be finite.")
        paired_values = np.column_stack((values_a, values_b))
        rng = np.random.default_rng(seed)
        bootstrap_differences = np.empty(int(n_resamples), dtype=float)
        for batch_start in range(0, int(n_resamples), 500):
            batch_stop = min(batch_start + 500, int(n_resamples))
            indices = rng.integers(
                0, len(paired_values), size=(batch_stop - batch_start, len(paired_values))
            )
            sampled_means = paired_values[indices].mean(axis=1)
            bootstrap_differences[batch_start:batch_stop] = (
                sampled_means[:, 0] - sampled_means[:, 1]
            )
        lower, upper = np.quantile(bootstrap_differences, [0.025, 0.975])
        return float(lower), float(upper), bootstrap_differences

    CLINICAL_STATS_BOOTSTRAP_TEST = _clinical_stats_bootstrap_paired_difference(
        [1.0, 2.0, 3.0, 4.0],
        [1.5, 2.5, 3.5, 4.5],
        seed=CLINICAL_STATS_ANALYSIS_SEED,
        n_resamples=128,
    )
    assert np.allclose(
        CLINICAL_STATS_BOOTSTRAP_TEST[2], -0.5, rtol=0.0, atol=1e-12
    )
    CLINICAL_STATS_ZERO_BOOTSTRAP_TEST = _clinical_stats_bootstrap_paired_difference(
        np.ones(4), np.ones(4), seed=CLINICAL_STATS_ANALYSIS_SEED + 1, n_resamples=128
    )
    assert CLINICAL_STATS_ZERO_BOOTSTRAP_TEST[:2] == (0.0, 0.0)

    CLINICAL_STATS_GLOBAL_OMNIBUS_ROWS = []
    CLINICAL_STATS_GLOBAL_PAIRWISE_ROWS = []
    CLINICAL_STATS_ALL_PAIRS = list(itertools.combinations(CLINICAL_STATS_METHODS, 2))
    for budget_index, query_budget in enumerate(CLINICAL_STATS_GLOBAL_BUDGETS):
        global_wide = CLINICAL_STATS_GLOBAL_WIDES[query_budget]
        matrix = global_wide.to_numpy(dtype=float)
        friedman_statistic, friedman_p_value = _clinical_stats_paired_friedman(matrix)
        range_seed = CLINICAL_STATS_ANALYSIS_SEED + 3000 + budget_index
        range_statistic, range_p_value = _clinical_stats_paired_range_permutation(
            matrix, range_seed
        )
        kendalls_w = friedman_statistic / (
            CLINICAL_STATS_N_SEEDS * (len(CLINICAL_STATS_METHODS) - 1)
        )
        if kendalls_w < -1e-12 or kendalls_w > 1.0 + 1e-12:
            raise RuntimeError("Global Kendall's W was outside [0, 1].")
        CLINICAL_STATS_GLOBAL_OMNIBUS_ROWS.append(
            {
                "Iteration Budget": query_budget,
                "Endpoint Role": CLINICAL_STATS_GLOBAL_BUDGET_ROLES[query_budget],
                "Endpoint": f"Query {query_budget} best-so-far normalized regret (%)",
                "N Paired Seed Blocks": CLINICAL_STATS_N_SEEDS,
                "N Interpolation Models": len(CLINICAL_STATS_MODELS),
                "Interpolation Aggregation": "Equal-weight mean of Linear and PCHIP",
                "N Optimizers": len(CLINICAL_STATS_METHODS),
                "Friedman Statistic": friedman_statistic,
                "Friedman P Value": friedman_p_value,
                "Kendall's W": kendalls_w,
                "Paired Permutation Range Statistic": range_statistic,
                "Paired Permutation Range P Value": range_p_value,
                "Paired Permutation Seed": range_seed,
                "Paired Permutations": CLINICAL_STATS_N_PERMUTATIONS,
            }
        )
        for pair_index, (method_a, method_b) in enumerate(CLINICAL_STATS_ALL_PAIRS):
            values_a = global_wide[method_a].to_numpy(dtype=float)
            values_b = global_wide[method_b].to_numpy(dtype=float)
            differences = values_a - values_b
            sign_flip_seed = (
                CLINICAL_STATS_ANALYSIS_SEED + 4000 + budget_index * 100 + pair_index
            )
            mean_difference, raw_p_value = _clinical_stats_sign_flip(
                differences, sign_flip_seed
            )
            bootstrap_seed = (
                CLINICAL_STATS_ANALYSIS_SEED + 5000 + budget_index * 100 + pair_index
            )
            ci_low, ci_high, _ = _clinical_stats_bootstrap_paired_difference(
                values_a, values_b, bootstrap_seed
            )
            difference_sd = float(np.std(differences, ddof=1))
            a_wins = int(np.count_nonzero(differences < -CLINICAL_STATS_TIE_TOLERANCE))
            b_wins = int(np.count_nonzero(differences > CLINICAL_STATS_TIE_TOLERANCE))
            ties = len(differences) - a_wins - b_wins
            if a_wins + b_wins + ties != CLINICAL_STATS_N_SEEDS:
                raise RuntimeError("Global pairwise win/tie/loss counts do not sum to 40.")
            CLINICAL_STATS_GLOBAL_PAIRWISE_ROWS.append(
                {
                    "Iteration Budget": query_budget,
                    "Endpoint Role": CLINICAL_STATS_GLOBAL_BUDGET_ROLES[query_budget],
                    "Endpoint": f"Query {query_budget} best-so-far normalized regret (%)",
                    "Method A": method_a,
                    "Method B": method_b,
                    "N Paired Seed Blocks": len(differences),
                    "N Interpolation Models": len(CLINICAL_STATS_MODELS),
                    "Interpolation Aggregation": "Equal-weight mean of Linear and PCHIP",
                    "Mean Regret Difference (A - B) (%)": mean_difference,
                    "95% Seed-Block CI Low (%)": ci_low,
                    "95% Seed-Block CI High (%)": ci_high,
                    "Paired Effect Size dz": (
                        mean_difference / difference_sd if difference_sd > 0 else np.nan
                    ),
                    "Proportion Method A Lower Regret": np.mean(differences < 0.0),
                    "A Wins (lower regret)": a_wins,
                    "Ties": ties,
                    "B Wins (lower regret)": b_wins,
                    "Paired Sign-Flip P Value": raw_p_value,
                    "Paired Sign-Flip Seed": sign_flip_seed,
                    "Paired Sign-Flip Permutations": CLINICAL_STATS_N_PERMUTATIONS,
                    "Bootstrap Resamples": CLINICAL_STATS_N_BOOTSTRAP,
                    "Bootstrap Seed": bootstrap_seed,
                }
            )

    CLINICAL_STATS_GLOBAL_OMNIBUS = pd.DataFrame(
        CLINICAL_STATS_GLOBAL_OMNIBUS_ROWS
    )
    CLINICAL_STATS_GLOBAL_PAIRWISE = pd.DataFrame(
        CLINICAL_STATS_GLOBAL_PAIRWISE_ROWS
    )
    if len(CLINICAL_STATS_GLOBAL_OMNIBUS) != len(CLINICAL_STATS_GLOBAL_BUDGETS):
        raise RuntimeError("Expected one global omnibus row per query budget.")
    if len(CLINICAL_STATS_GLOBAL_PAIRWISE) != len(CLINICAL_STATS_GLOBAL_BUDGETS) * 6:
        raise RuntimeError("Expected six global pairwise contrasts per query budget.")
    CLINICAL_STATS_GLOBAL_PAIRWISE["Holm Adjusted P Value"] = np.nan
    for query_budget in CLINICAL_STATS_GLOBAL_BUDGETS:
        budget_mask = CLINICAL_STATS_GLOBAL_PAIRWISE["Iteration Budget"] == query_budget
        if int(budget_mask.sum()) != 6:
            raise RuntimeError(
                f"Query-{query_budget} Holm family must contain six pairwise contrasts."
            )
        CLINICAL_STATS_GLOBAL_PAIRWISE.loc[
            budget_mask, "Holm Adjusted P Value"
        ] = multipletests(
            CLINICAL_STATS_GLOBAL_PAIRWISE.loc[
                budget_mask, "Paired Sign-Flip P Value"
            ].to_numpy(dtype=float),
            method="holm",
        )[1]

    CLINICAL_STATS_GLOBAL_BOOTSTRAP = CLINICAL_STATS_GLOBAL_PAIRWISE[
        [
            "Iteration Budget",
            "Endpoint Role",
            "Method A",
            "Method B",
            "N Paired Seed Blocks",
            "Mean Regret Difference (A - B) (%)",
            "95% Seed-Block CI Low (%)",
            "95% Seed-Block CI High (%)",
            "Bootstrap Resamples",
            "Bootstrap Seed",
        ]
    ].copy()
    CLINICAL_STATS_OMNIBUS = CLINICAL_STATS_GLOBAL_OMNIBUS
    CLINICAL_STATS_PAIRWISE = CLINICAL_STATS_GLOBAL_PAIRWISE
    CLINICAL_STATS_BOOTSTRAP = CLINICAL_STATS_GLOBAL_BOOTSTRAP
    CLINICAL_STATS_Q40_OMNIBUS = CLINICAL_STATS_GLOBAL_OMNIBUS.loc[
        CLINICAL_STATS_GLOBAL_OMNIBUS["Iteration Budget"] == 40
    ].copy()
    CLINICAL_STATS_Q10_OMNIBUS = CLINICAL_STATS_GLOBAL_OMNIBUS.loc[
        CLINICAL_STATS_GLOBAL_OMNIBUS["Iteration Budget"] == 10
    ].copy()
    CLINICAL_STATS_Q40_PAIRWISE = CLINICAL_STATS_GLOBAL_PAIRWISE.loc[
        CLINICAL_STATS_GLOBAL_PAIRWISE["Iteration Budget"] == 40
    ].copy()
    CLINICAL_STATS_Q10_PAIRWISE = CLINICAL_STATS_GLOBAL_PAIRWISE.loc[
        CLINICAL_STATS_GLOBAL_PAIRWISE["Iteration Budget"] == 10
    ].copy()
    CLINICAL_STATS_Q40_BOOTSTRAP = CLINICAL_STATS_GLOBAL_BOOTSTRAP.loc[
        CLINICAL_STATS_GLOBAL_BOOTSTRAP["Iteration Budget"] == 40
    ].copy()
    CLINICAL_STATS_Q10_BOOTSTRAP = CLINICAL_STATS_GLOBAL_BOOTSTRAP.loc[
        CLINICAL_STATS_GLOBAL_BOOTSTRAP["Iteration Budget"] == 10
    ].copy()

    CLINICAL_STATS_DESCRIPTIVE_ROWS = []
    for model_name in CLINICAL_STATS_MODELS:
        for method in CLINICAL_STATS_METHODS:
            method_rows = CLINICAL_STATS_CONVERGENCE.loc[
                (CLINICAL_STATS_CONVERGENCE["Interpolation Model"] == model_name)
                & (CLINICAL_STATS_CONVERGENCE["Method"] == method)
            ]
            for query in (10, 20, 40):
                values = method_rows.loc[
                    method_rows["Query"] == query,
                    "Clinical Stats Normalized Regret (%)",
                ].to_numpy(dtype=float)
                if len(values) != CLINICAL_STATS_N_SEEDS:
                    raise RuntimeError(
                        f"{model_name} / {method} / Query {query} does not have 40 runs."
                    )
                q1, median, q3 = np.quantile(values, [0.25, 0.5, 0.75])
                CLINICAL_STATS_DESCRIPTIVE_ROWS.append(
                    {
                        "Interpolation Model": model_name,
                        "Method": method,
                        "Query": query,
                        "N Independent Seed Blocks": len(values),
                        "Mean Normalized Regret (%)": float(values.mean()),
                        "SD Normalized Regret (%)": float(values.std(ddof=1)),
                        "Median Normalized Regret (%)": float(median),
                        "Q1 Normalized Regret (%)": float(q1),
                        "Q3 Normalized Regret (%)": float(q3),
                    }
                )
    CLINICAL_STATS_DESCRIPTIVE = pd.DataFrame(CLINICAL_STATS_DESCRIPTIVE_ROWS)

    def _clinical_stats_bootstrap_mean_band(trajectories, seed):
        trajectories = np.asarray(trajectories, dtype=float)
        if (
            trajectories.ndim != 2
            or trajectories.shape[0] != CLINICAL_STATS_N_SEEDS
            or not np.isfinite(trajectories).all()
        ):
            raise ValueError("Convergence bootstrap requires 40 finite seed trajectories.")
        rng = np.random.default_rng(seed)
        bootstrap_means = np.empty(
            (CLINICAL_STATS_N_BOOTSTRAP, trajectories.shape[1]), dtype=float
        )
        for batch_start in range(0, CLINICAL_STATS_N_BOOTSTRAP, 500):
            batch_stop = min(batch_start + 500, CLINICAL_STATS_N_BOOTSTRAP)
            indices = rng.integers(
                0, trajectories.shape[0],
                size=(batch_stop - batch_start, trajectories.shape[0]),
            )
            bootstrap_means[batch_start:batch_stop] = trajectories[indices].mean(axis=1)
        lower, upper = np.quantile(bootstrap_means, [0.025, 0.975], axis=0)
        return trajectories.mean(axis=0), lower, upper

    CLINICAL_STATS_OUTPUT_DIR = Path(output_directory)
    CLINICAL_STATS_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    CLINICAL_STATS_DATASET_SLUG = "".join(
        character.lower() if character.isalnum() else "_"
        for character in dataset_label.strip()
    ).strip("_")
    if not CLINICAL_STATS_DATASET_SLUG:
        raise ValueError("dataset_label must contain letters or digits for global test filenames.")
    CLINICAL_STATS_GLOBAL_OMNIBUS_PATH = CLINICAL_STATS_OUTPUT_DIR / (
        f"clinical_optimizer_global_paired_{CLINICAL_STATS_DATASET_SLUG}_omnibus.csv"
    )
    CLINICAL_STATS_GLOBAL_POSTHOC_PATH = CLINICAL_STATS_OUTPUT_DIR / (
        f"clinical_optimizer_global_paired_{CLINICAL_STATS_DATASET_SLUG}_posthoc.csv"
    )
    CLINICAL_STATS_CONVERGENCE_ROWS = []
    CLINICAL_STATS_AUC_BY_RUN = {}
    CLINICAL_STATS_FIGURE, CLINICAL_STATS_AXES = plt.subplots(
        1, 2, figsize=(15, 6), sharey=True
    )
    CLINICAL_STATS_COLORS = plt.get_cmap("tab10")
    for model_index, (axis, model_name) in enumerate(
        zip(CLINICAL_STATS_AXES, CLINICAL_STATS_MODELS)
    ):
        for method_index, method in enumerate(CLINICAL_STATS_METHODS):
            run_rows = CLINICAL_STATS_CONVERGENCE.loc[
                (CLINICAL_STATS_CONVERGENCE["Interpolation Model"] == model_name)
                & (CLINICAL_STATS_CONVERGENCE["Method"] == method)
            ]
            trajectory_wide = run_rows.pivot(
                index="Seed",
                columns="Query",
                values="Clinical Stats Normalized Regret (%)",
            ).reindex(index=CLINICAL_STATS_SEEDS, columns=CLINICAL_STATS_QUERIES)
            if trajectory_wide.isna().any().any():
                raise RuntimeError(f"{model_name} / {method} has an incomplete trajectory.")
            trajectories = trajectory_wide.to_numpy(dtype=float)
            mean_curve, ci_low_curve, ci_high_curve = _clinical_stats_bootstrap_mean_band(
                trajectories,
                CLINICAL_STATS_ANALYSIS_SEED + 20000 + model_index * 100 + method_index,
            )
            median_curve = np.median(trajectories, axis=0)
            q1_curve, q3_curve = np.quantile(trajectories, [0.25, 0.75], axis=0)
            for query_index, query in enumerate(CLINICAL_STATS_QUERIES):
                CLINICAL_STATS_CONVERGENCE_ROWS.append(
                    {
                        "Interpolation Model": model_name,
                        "Method": method,
                        "Query": int(query),
                        "N Independent Seed Blocks": CLINICAL_STATS_N_SEEDS,
                        "Mean Normalized Regret (%)": float(mean_curve[query_index]),
                        "Median Normalized Regret (%)": float(median_curve[query_index]),
                        "Q1 Normalized Regret (%)": float(q1_curve[query_index]),
                        "Q3 Normalized Regret (%)": float(q3_curve[query_index]),
                        "95% Pointwise Bootstrap CI Low for Mean": float(ci_low_curve[query_index]),
                        "95% Pointwise Bootstrap CI High for Mean": float(ci_high_curve[query_index]),
                    }
                )
            axis.plot(
                CLINICAL_STATS_QUERIES,
                mean_curve,
                color=CLINICAL_STATS_COLORS(method_index),
                label=method,
            )
            axis.fill_between(
                CLINICAL_STATS_QUERIES,
                ci_low_curve,
                ci_high_curve,
                color=CLINICAL_STATS_COLORS(method_index),
                alpha=0.16,
                linewidth=0,
            )
            CLINICAL_STATS_AUC_BY_RUN[(model_name, method)] = pd.Series(
                np.sum((trajectories[:, 1:] + trajectories[:, :-1]) / 2.0, axis=1),
                index=CLINICAL_STATS_SEEDS,
                dtype=float,
            )
        axis.set_title(f"{model_name} interpolation")
        axis.set_xlabel("Query number")
        axis.set_xlim(1, CLINICAL_STATS_BUDGET)
        axis.set_xticks([1, 10, 20, 30, 40])
        axis.set_ylim(bottom=0)
        axis.grid(True, alpha=0.25)
    CLINICAL_STATS_AXES[0].set_ylabel("Best-so-far normalized regret (%)")
    CLINICAL_STATS_AXES[-1].legend(
        loc="upper left", bbox_to_anchor=(1.02, 1.0), borderaxespad=0.0
    )
    CLINICAL_STATS_FIGURE.suptitle(
        f"{dataset_label} optimizer convergence (mean and pointwise 95% bootstrap CI; "
            f"n={CLINICAL_STATS_N_SEEDS} seed blocks)"
    )
    CLINICAL_STATS_FIGURE.tight_layout(rect=(0.0, 0.0, 0.84, 0.94))
    CLINICAL_STATS_CONVERGENCE_SUMMARY = pd.DataFrame(CLINICAL_STATS_CONVERGENCE_ROWS)
    CLINICAL_STATS_PLOT_PATH = (
        CLINICAL_STATS_OUTPUT_DIR / "clinical_optimizer_convergence.png"
    )
    CLINICAL_STATS_FIGURE.savefig(CLINICAL_STATS_PLOT_PATH, dpi=200, bbox_inches="tight")
    plt.close(CLINICAL_STATS_FIGURE)

    CLINICAL_STATS_Q40_MEANS = {
        model_name: CLINICAL_STATS_Q40_WIDE[model_name].mean(axis=0)
        for model_name in CLINICAL_STATS_MODELS
    }
    CLINICAL_STATS_RANKS = {
        model_name: values.rank(method="min", ascending=True).astype(int)
        for model_name, values in CLINICAL_STATS_Q40_MEANS.items()
    }
    CLINICAL_STATS_LINEAR_ORDER = tuple(
        CLINICAL_STATS_Q40_MEANS["Linear"].sort_values().index
    )
    CLINICAL_STATS_PCHIP_ORDER = tuple(
        CLINICAL_STATS_Q40_MEANS["PCHIP"].sort_values().index
    )
    CLINICAL_STATS_KENDALL_TAU = float(
        stats.kendalltau(
            [CLINICAL_STATS_RANKS["Linear"][method] for method in CLINICAL_STATS_METHODS],
            [CLINICAL_STATS_RANKS["PCHIP"][method] for method in CLINICAL_STATS_METHODS],
        )[0]
    )
    CLINICAL_STATS_SENSITIVITY_ROWS = []
    for method in CLINICAL_STATS_METHODS:
        linear_q40 = CLINICAL_STATS_Q40_WIDE["Linear"][method].to_numpy(dtype=float)
        pchip_q40 = CLINICAL_STATS_Q40_WIDE["PCHIP"][method].to_numpy(dtype=float)
        q40_change = pchip_q40 - linear_q40
        linear_auc = CLINICAL_STATS_AUC_BY_RUN[("Linear", method)].reindex(
            CLINICAL_STATS_SEEDS
        ).to_numpy(dtype=float)
        pchip_auc = CLINICAL_STATS_AUC_BY_RUN[("PCHIP", method)].reindex(
            CLINICAL_STATS_SEEDS
        ).to_numpy(dtype=float)
        auc_change = pchip_auc - linear_auc
        CLINICAL_STATS_SENSITIVITY_ROWS.append(
            {
                "Method": method,
                "N Matched Seed Blocks": CLINICAL_STATS_N_SEEDS,
                "Linear Mean Q40 Regret (%)": float(linear_q40.mean()),
                "PCHIP Mean Q40 Regret (%)": float(pchip_q40.mean()),
                "Mean PCHIP - Linear Q40 Regret (percentage points)": float(q40_change.mean()),
                "SD PCHIP - Linear Q40 Regret": float(q40_change.std(ddof=1)),
                "Median PCHIP - Linear Q40 Regret": float(np.median(q40_change)),
                "Q1 PCHIP - Linear Q40 Regret": float(np.quantile(q40_change, 0.25)),
                "Q3 PCHIP - Linear Q40 Regret": float(np.quantile(q40_change, 0.75)),
                "Linear Q40 Rank (1 is best)": int(CLINICAL_STATS_RANKS["Linear"][method]),
                "PCHIP Q40 Rank (1 is best)": int(CLINICAL_STATS_RANKS["PCHIP"][method]),
                "Linear Mean AUC (regret percentage points x queries)": float(linear_auc.mean()),
                "PCHIP Mean AUC (regret percentage points x queries)": float(pchip_auc.mean()),
                "Mean PCHIP - Linear AUC": float(auc_change.mean()),
            }
        )
    CLINICAL_STATS_SENSITIVITY = pd.DataFrame(CLINICAL_STATS_SENSITIVITY_ROWS)

    CLINICAL_STATS_GLOBAL_OMNIBUS.to_csv(
        CLINICAL_STATS_GLOBAL_OMNIBUS_PATH, index=False
    )
    CLINICAL_STATS_GLOBAL_PAIRWISE.to_csv(
        CLINICAL_STATS_GLOBAL_POSTHOC_PATH, index=False
    )
    CLINICAL_STATS_DESCRIPTIVE.to_csv(
        CLINICAL_STATS_OUTPUT_DIR / "clinical_optimizer_descriptive_checkpoints.csv",
        index=False,
    )
    CLINICAL_STATS_SENSITIVITY.to_csv(
        CLINICAL_STATS_OUTPUT_DIR / "clinical_optimizer_interpolation_sensitivity.csv",
        index=False,
    )
    CLINICAL_STATS_CONVERGENCE_SUMMARY.to_csv(
        CLINICAL_STATS_OUTPUT_DIR / "clinical_optimizer_convergence_summary.csv",
        index=False,
    )

    CLINICAL_STATS_FILES = {
        "Global paired omnibus tests": CLINICAL_STATS_GLOBAL_OMNIBUS_PATH,
        "Global paired post-hoc tests with Holm correction": CLINICAL_STATS_GLOBAL_POSTHOC_PATH,
        "Descriptive checkpoints": CLINICAL_STATS_OUTPUT_DIR / "clinical_optimizer_descriptive_checkpoints.csv",
        "Interpolation sensitivity": CLINICAL_STATS_OUTPUT_DIR / "clinical_optimizer_interpolation_sensitivity.csv",
        "Convergence summary": CLINICAL_STATS_OUTPUT_DIR / "clinical_optimizer_convergence_summary.csv",
        "Convergence plot": CLINICAL_STATS_PLOT_PATH,
    }

    print(
        f"Clinical statistical post-analysis: {CLINICAL_STATS_N_SEEDS} independent "
        "matched simulation seed blocks across both interpolations and all optimizers."
    )
    print(
        "Seed blocks are algorithmic simulation replicates, not clinical patients. "
        "Queries 10 and 20 are checkpoints within each 40-query run, not extra replicates."
    )
    print(
        "Unclipped normalized regret is recomputed from each run's best-so-far "
        "noise-free query response, using the anchor maximum "
        f"{CLINICAL_STATS_REFERENCE:.3f} at ({CLINICAL_STATS_REFERENCE_POINT[0]:g} Hz, "
        f"{CLINICAL_STATS_REFERENCE_POINT[1]:g} us)."
    )
    print(
        f"The stored optimizer incumbent differs from the query-history maximum in "
        f"{CLINICAL_STATS_INCUMBENT_DISAGREE_ROWS}/{len(CLINICAL_STATS_CONVERGENCE)} "
        f"query rows and {CLINICAL_STATS_INCUMBENT_DISAGREE_Q40}/"
        f"{len(CLINICAL_STATS_QUERY40)} Q40 runs; inferential endpoints use the "
        "query-history maximum."
    )
    print(
        "Linear and PCHIP regrets are equally averaged within each seed block; the "
        "interpolations are not counted as independent replicates. Query 40 is primary; "
        "query 10 is exploratory."
    )
    print(
        f"Each budget has Friedman and paired permutation-range omnibus tests, plus six "
        f"sign-flip contrasts with Holm correction per budget ({CLINICAL_STATS_N_PERMUTATIONS:,} "
        f"permutations; {CLINICAL_STATS_N_BOOTSTRAP:,} bootstrap resamples). Sign-flip "
        "inference assumes exchangeable, symmetric paired differences under the null. "
        "Bootstrap intervals resample complete seed blocks; convergence bands are pointwise."
    )
    print("\\nGlobal paired omnibus tests:")
    display(CLINICAL_STATS_GLOBAL_OMNIBUS.round(6))
    print("\\nGlobal paired post-hoc tests with Holm correction (negative A - B favors A):")
    display(CLINICAL_STATS_GLOBAL_PAIRWISE.round(6))
    print("\\nQ10/Q20/Q40 descriptive summaries:")
    display(CLINICAL_STATS_DESCRIPTIVE.round(4))
    print("\\nLinear-to-PCHIP sensitivity (descriptive; same seed blocks):")
    display(CLINICAL_STATS_SENSITIVITY.round(4))

    CLINICAL_STATS_PUBLICATION_LINES = [
        f"Publication-ready statistical summary: {dataset_label} clinically anchored "
            "taVNS optimizer simulations",
        (
            f"Four optimizers were compared on Linear and PCHIP interpolations of the "
            f"{len(CLINICAL_STATS_ANCHORS)} measured {dataset_label} anchors using "
            f"{CLINICAL_STATS_N_SEEDS} matched seed blocks and query budgets 10 and 40. "
            "Linear and PCHIP regrets were averaged within each seed for global tests; "
            "seed blocks, not individual queries, are the independent simulation replicates."
        ),
        (
            f"The validated reference was {CLINICAL_STATS_REFERENCE:.3f} at "
            f"({CLINICAL_STATS_REFERENCE_POINT[0]:g} Hz, "
            f"{CLINICAL_STATS_REFERENCE_POINT[1]:g} us); regret is best-so-far "
            "noise-free regret as a percentage of this response, recomputed from "
            "query-history maxima rather than optimizer-selected incumbents."
        ),
        (
            f"Stored optimizer incumbents differ from query-history maxima in "
            f"{CLINICAL_STATS_INCUMBENT_DISAGREE_ROWS}/"
            f"{len(CLINICAL_STATS_CONVERGENCE)} query rows and "
            f"{CLINICAL_STATS_INCUMBENT_DISAGREE_Q40}/"
            f"{len(CLINICAL_STATS_QUERY40)} query-40 runs; these values were not "
            "used as inferential endpoints."
        ),
    ]
    for _, row in CLINICAL_STATS_GLOBAL_OMNIBUS.iterrows():
        kendalls_w = row["Kendall's W"]
        CLINICAL_STATS_PUBLICATION_LINES.append(
            f"Query {row['Iteration Budget']} ({row['Endpoint Role']}) paired omnibus: "
            f"chi-square={row['Friedman Statistic']:.3f}, "
            f"Friedman p={row['Friedman P Value']:.6g}, Kendall's W={kendalls_w:.3f}; "
            f"permutation-range statistic={row['Paired Permutation Range Statistic']:.3f}, "
            f"p={row['Paired Permutation Range P Value']:.6g}."
        )
    CLINICAL_STATS_SIGNIFICANT = CLINICAL_STATS_GLOBAL_PAIRWISE.loc[
        CLINICAL_STATS_GLOBAL_PAIRWISE["Holm Adjusted P Value"] < 0.05
    ]
    if CLINICAL_STATS_SIGNIFICANT.empty:
        CLINICAL_STATS_PUBLICATION_LINES.append(
            "No optimizer pair remained distinguishable after Holm correction "
            "within either six-comparison query-budget family."
        )
    else:
        CLINICAL_STATS_PUBLICATION_LINES.append(
            "Holm-adjusted pairwise contrasts below 0.05 (A minus B; negative favors A):"
        )
        for _, row in CLINICAL_STATS_SIGNIFICANT.iterrows():
            CLINICAL_STATS_PUBLICATION_LINES.append(
                f"Query {row['Iteration Budget']} ({row['Endpoint Role']}), "
                f"{row['Method A']} minus {row['Method B']}: mean paired difference="
                f"{row['Mean Regret Difference (A - B) (%)']:.3f} percentage points, "
                f"95% seed-block CI [{row['95% Seed-Block CI Low (%)']:.3f}, "
                f"{row['95% Seed-Block CI High (%)']:.3f}], "
                f"raw sign-flip p={row['Paired Sign-Flip P Value']:.6g}, "
                f"Holm-adjusted p={row['Holm Adjusted P Value']:.6g}."
            )
    CLINICAL_STATS_PUBLICATION_LINES.append(
        "Mean-Q40 ranking, Linear: "
        + " < ".join(CLINICAL_STATS_LINEAR_ORDER)
        + "; PCHIP: "
        + " < ".join(CLINICAL_STATS_PCHIP_ORDER)
        + f" (descriptive Kendall rank correlation tau={CLINICAL_STATS_KENDALL_TAU:.3f})."
    )
    CLINICAL_STATS_PUBLICATION_LINES.append(
        "Interpolation sensitivity is descriptive and paired by seed; Linear and PCHIP "
        "are alternative interpolations of the same anchors, not independent clinical datasets."
    )
    CLINICAL_STATS_PUBLICATION_LINES.append(
        "At each budget, regrets were averaged equally across Linear and PCHIP within "
        "each matched seed before the global tests. The omnibus permutation-range test "
        f"and sign-flip contrasts used {CLINICAL_STATS_N_PERMUTATIONS:,} permutations, "
        f"and paired intervals used {CLINICAL_STATS_N_BOOTSTRAP:,} seed-block bootstrap "
        "resamples. Sign-flip p-values assume exchangeable, symmetric paired differences "
        "under the null; confidence intervals are independent of permutation tests."
    )
    CLINICAL_STATS_PUBLICATION_LINES.append(
        "Limitations: this deterministic simulation does not sample patients, biological "
        "response variability, or measurement uncertainty. It cannot establish clinical "
        "efficacy, patient-level generalizability, or treatment recommendations."
    )
    CLINICAL_STATS_PUBLICATION_SUMMARY = "\\n".join(CLINICAL_STATS_PUBLICATION_LINES)
    print("\\n" + CLINICAL_STATS_PUBLICATION_SUMMARY)
    print("\\nSaved statistical-analysis files:")
    for label, file_path in CLINICAL_STATS_FILES.items():
        display(FileLink(str(file_path), result_html_prefix=f"{label}: "))
    return {
        "global_omnibus": CLINICAL_STATS_GLOBAL_OMNIBUS,
        "global_pairwise": CLINICAL_STATS_GLOBAL_PAIRWISE,
        "omnibus": CLINICAL_STATS_OMNIBUS,
        "pairwise": CLINICAL_STATS_PAIRWISE,
        "bootstrap": CLINICAL_STATS_BOOTSTRAP,
        "query10_omnibus": CLINICAL_STATS_Q10_OMNIBUS,
        "query10_pairwise": CLINICAL_STATS_Q10_PAIRWISE,
        "query10_bootstrap": CLINICAL_STATS_Q10_BOOTSTRAP,
        "descriptive": CLINICAL_STATS_DESCRIPTIVE,
        "sensitivity": CLINICAL_STATS_SENSITIVITY,
        "convergence_summary": CLINICAL_STATS_CONVERGENCE_SUMMARY,
        "bogp_zero_regret_audit": None,
        "publication_summary": CLINICAL_STATS_PUBLICATION_SUMMARY,
        "files": CLINICAL_STATS_FILES,
        "reference": CLINICAL_STATS_REFERENCE,
        "reference_point": CLINICAL_STATS_REFERENCE_POINT,
    }



def summarize_clinical_optimizer_checkpoints(results, convergence, expected_seed_repeats):
    results = pd.DataFrame(results).copy()
    convergence = pd.DataFrame(convergence).copy()
    if results.empty or convergence.empty:
        raise ValueError("Optimizer results and convergence data must not be empty.")
    final_query = int(results["Iteration Budget"].max())
    summary_keys = ["Interpolation Model", "Method"]

    def summarize_checkpoint(query, label):
        checkpoint = convergence.loc[convergence["Query"] == query]
        checkpoint_summary = (
            checkpoint.groupby(summary_keys, as_index=False)
            .agg(
                Seed_Runs=("Seed", "nunique"),
                Mean_Regret=("Selected Incumbent Normalized Regret (%)", "mean"),
                SD_Regret=("Selected Incumbent Normalized Regret (%)", "std"),
                Median_Regret=("Selected Incumbent Normalized Regret (%)", "median"),
                Q1_Regret=(
                    "Selected Incumbent Normalized Regret (%)",
                    lambda values: values.quantile(0.25),
                ),
                Q3_Regret=(
                    "Selected Incumbent Normalized Regret (%)",
                    lambda values: values.quantile(0.75),
                ),
            )
        )
        expected_group_count = len(results[summary_keys].drop_duplicates())
        if (
            len(checkpoint_summary) != expected_group_count
            or not checkpoint_summary["Seed_Runs"].eq(expected_seed_repeats).all()
        ):
            raise RuntimeError(f"Query {query} normalized-regret summary is missing optimizer runs.")
        checkpoint_summary["Median [Q1, Q3]"] = checkpoint_summary.apply(
            lambda row: (
                f"{row['Median_Regret']:.2f} "
                f"[{row['Q1_Regret']:.2f}, {row['Q3_Regret']:.2f}]"
            ),
            axis=1,
        )
        return checkpoint_summary.drop(
            columns=["Median_Regret", "Q1_Regret", "Q3_Regret"]
        ).rename(
            columns={
                "Mean_Regret": f"Mean_Normalized_Regret_{label} (%)",
                "SD_Regret": f"SD_Normalized_Regret_{label} (%)",
                "Median [Q1, Q3]": f"Median [Q1, Q3]_{label} (%)",
            }
        )

    query_10_summary = summarize_checkpoint(10, "Q10")
    query_20_summary = summarize_checkpoint(20, "Q20")
    final_label = f"Q{final_query}" if final_query != 20 else "Final"
    final_query_summary = summarize_checkpoint(final_query, final_label)
    checkpoint_run_columns = summary_keys + ["Seed_Runs"]
    if (
        not query_10_summary[checkpoint_run_columns].equals(
            query_20_summary[checkpoint_run_columns]
        )
        or not query_20_summary[checkpoint_run_columns].equals(
            final_query_summary[checkpoint_run_columns]
        )
    ):
        raise RuntimeError("Query-10, Query-20, and final summaries include different seed runs.")
    mean_runtime = results.groupby(summary_keys, as_index=False).agg(
        Mean_Seconds=("Sec To Compute", "mean")
    )
    return (
        query_10_summary.merge(
            query_20_summary.drop(columns="Seed_Runs"),
            on=summary_keys,
            validate="one_to_one",
        )
        .merge(
            final_query_summary.drop(columns="Seed_Runs"),
            on=summary_keys,
            validate="one_to_one",
        )
        .merge(mean_runtime, on=summary_keys, validate="one_to_one")
        .round(2)
    )


def run_clinical_optimizer_experiment(
    *,
    dataset_label,
    anchor_points,
    frequency_bounds_hz,
    pulse_width_bounds_us,
    raw_file_prefix,
    raw_output_directory=".",
    statistics_output_directory="clinical_optimizer_statistics",
    smac_output_directory="smac3_output",
    num_simulation_trials=CLINICAL_OPTIMIZER_EVALUATIONS_PER_RUN,
    amount_of_seed_repeats=CLINICAL_OPTIMIZER_REPEATS,
    patient_and_machine_startSeed=10000,
):
    """Run and export one dataset's paired Linear/PCHIP optimizer comparison."""
    if not isinstance(dataset_label, str) or not dataset_label.strip():
        raise ValueError("dataset_label must be a non-empty string.")
    raw_anchor_values = np.asarray(anchor_points, dtype=object)
    if (
        raw_anchor_values.ndim == 2
        and raw_anchor_values.shape[1] == 3
        and any(value is None for value in raw_anchor_values[:, 2])
    ):
        print(
            f"{dataset_label} PCHIP response model disabled: enter measured response "
            "values for every anchor before running optimizers."
        )
        raise ValueError(f"{dataset_label} response anchors are missing measured values.")
    try:
        anchor_values = np.asarray(anchor_points, dtype=float)
    except (TypeError, ValueError) as error:
        raise ValueError(f"{dataset_label} anchors must be finite numeric values.") from error
    if (
        anchor_values.ndim != 2
        or anchor_values.shape[1] != 3
        or not 1 <= anchor_values.shape[0] <= 10
        or not np.isfinite(anchor_values).all()
    ):
        raise ValueError(f"{dataset_label} must have 1 to 10 finite clinical anchor rows.")
    frequency_bounds = _validate_clinical_map_bounds(
        frequency_bounds_hz, "frequency_bounds_hz"
    )
    pulse_width_bounds = _validate_clinical_map_bounds(
        pulse_width_bounds_us, "pulse_width_bounds_us"
    )
    response_functions = {
        "Linear": build_clinical_response_function(
            anchor_values,
            frequency_bounds_hz=frequency_bounds,
            pulse_width_bounds_us=pulse_width_bounds,
        ),
        "PCHIP": build_clinical_pchip_response_function(
            anchor_values,
        ),
    }
    for model_name, response_function in response_functions.items():
        reproduced = np.asarray(
            response_function(anchor_values[:, 0], anchor_values[:, 1]), dtype=float
        )
        if not np.allclose(reproduced, anchor_values[:, 2], rtol=0.0, atol=1e-12):
            raise RuntimeError(f"{dataset_label} {model_name} model failed to reproduce its anchors.")

    optimizer_results, optimizer_convergence = run_clinical_interpolation_optimizer_comparison(
        anchor_points=anchor_values,
        response_functions=response_functions,
        dataset_label=dataset_label,
        frequency_bounds_hz=frequency_bounds,
        pulse_width_bounds_us=pulse_width_bounds,
        smac_output_directory=smac_output_directory,
        num_simulation_trials=num_simulation_trials,
        amount_of_seed_repeats=amount_of_seed_repeats,
        patient_and_machine_startSeed=patient_and_machine_startSeed,
    )
    optimizer_summary = summarize_clinical_optimizer_checkpoints(
        optimizer_results, optimizer_convergence, int(amount_of_seed_repeats)
    )
    print(
        f"{dataset_label} clinical interpolation optimizer comparison completed "
        "(lower normalized regret is better):"
    )
    summary_formatters = {
        column: "{:.2f}".format
        for column in optimizer_summary.select_dtypes(include=[np.number]).columns
        if column != "Seed_Runs"
    }
    print(optimizer_summary.to_string(index=False, formatters=summary_formatters))

    raw_output_directory = Path(raw_output_directory)
    raw_output_directory.mkdir(parents=True, exist_ok=True)
    raw_csv_frames = {
        "Clinical optimizer summary": ("summary", optimizer_summary),
        "Clinical optimizer run results": ("results", optimizer_results),
        "Clinical optimizer convergence data": ("convergence", optimizer_convergence),
    }
    raw_csv_files = {}
    for label, (suffix, frame) in raw_csv_frames.items():
        csv_path = raw_output_directory / f"{raw_file_prefix}_{suffix}.csv"
        frame.to_csv(csv_path, index=False)
        raw_csv_files[label] = str(csv_path.resolve())
        display(FileLink(str(csv_path), result_html_prefix=f"{dataset_label} {label}: "))
    print(f"Exported {len(raw_csv_files)} {dataset_label} optimizer CSV files.")

    statistics = analyze_clinical_optimizer_statistics(
        optimizer_results,
        optimizer_convergence,
        anchor_values,
        response_functions,
        dataset_label,
        statistics_output_directory,
    )
    return {
        "results": optimizer_results,
        "convergence": optimizer_convergence,
        "summary": optimizer_summary,
        "raw_csv_files": raw_csv_files,
        "response_functions": response_functions,
        "statistics": statistics,
    }


In [7]:
BADRAN_RESPONSE_ANCHORS = [
    (1.0, 100.0, -1.656),
    (1.0, 200.0, -2.080),
    (1.0, 500.0, 1.313),
    (10.0, 100.0, 0.279),
    (10.0, 200.0, -1.320),
    (10.0, 500.0, 1.220),
    (25.0, 100.0, 0.284),
    (25.0, 200.0, 1.760),
    (25.0, 500.0, 3.929),
]
BADRAN_FREQUENCY_BOUNDS_HZ = (1.0, 25.0)
BADRAN_PULSE_WIDTH_BOUNDS_US = (100.0, 500.0)

CLINICAL_RESPONSE_ANCHORS = BADRAN_RESPONSE_ANCHORS
CLINICAL_MAP_FREQUENCY_BOUNDS_HZ = BADRAN_FREQUENCY_BOUNDS_HZ
CLINICAL_MAP_PULSE_WIDTH_BOUNDS_US = BADRAN_PULSE_WIDTH_BOUNDS_US

BADRAN_CLINICAL_EXPERIMENT = run_clinical_optimizer_experiment(
    dataset_label="Badran",
    anchor_points=BADRAN_RESPONSE_ANCHORS,
    frequency_bounds_hz=BADRAN_FREQUENCY_BOUNDS_HZ,
    pulse_width_bounds_us=BADRAN_PULSE_WIDTH_BOUNDS_US,
    raw_file_prefix="clinical_interpolation_optimizer",
    raw_output_directory=".",
    statistics_output_directory="clinical_optimizer_statistics",
    smac_output_directory="smac3_output",
    num_simulation_trials=CLINICAL_OPTIMIZER_EVALUATIONS_PER_RUN,
    amount_of_seed_repeats=CLINICAL_OPTIMIZER_REPEATS,
    patient_and_machine_startSeed=10000,
)
BADRAN_OPTIMIZER_RESULTS = BADRAN_CLINICAL_EXPERIMENT["results"]
BADRAN_OPTIMIZER_CONVERGENCE = BADRAN_CLINICAL_EXPERIMENT["convergence"]
BADRAN_OPTIMIZER_SUMMARY = BADRAN_CLINICAL_EXPERIMENT["summary"]
BADRAN_RESPONSE_FUNCTIONS = BADRAN_CLINICAL_EXPERIMENT["response_functions"]
BADRAN_OPTIMIZER_CSV_FILES = BADRAN_CLINICAL_EXPERIMENT["raw_csv_files"]
BADRAN_CLINICAL_STATISTICS = BADRAN_CLINICAL_EXPERIMENT["statistics"]

BADRAN_LINEAR_RESPONSE_FUNCTION = BADRAN_RESPONSE_FUNCTIONS["Linear"]
BADRAN_PCHIP_RESPONSE_FUNCTION = BADRAN_RESPONSE_FUNCTIONS["PCHIP"]
CLINICAL_RESPONSE_FUNCTION = BADRAN_LINEAR_RESPONSE_FUNCTION
CLINICAL_PCHIP_RESPONSE_FUNCTION = BADRAN_PCHIP_RESPONSE_FUNCTION

CLINICAL_OPTIMIZER_RESULTS = BADRAN_OPTIMIZER_RESULTS
CLINICAL_OPTIMIZER_CONVERGENCE = BADRAN_OPTIMIZER_CONVERGENCE
CLINICAL_OPTIMIZER_SUMMARY = BADRAN_OPTIMIZER_SUMMARY
CLINICAL_OPTIMIZER_CSV_FILES = BADRAN_OPTIMIZER_CSV_FILES
CLINICAL_STATS_OMNIBUS = BADRAN_CLINICAL_STATISTICS["omnibus"]
CLINICAL_STATS_PAIRWISE = BADRAN_CLINICAL_STATISTICS["pairwise"]
CLINICAL_STATS_BOOTSTRAP = BADRAN_CLINICAL_STATISTICS["bootstrap"]
CLINICAL_STATS_DESCRIPTIVE = BADRAN_CLINICAL_STATISTICS["descriptive"]
CLINICAL_STATS_SENSITIVITY = BADRAN_CLINICAL_STATISTICS["sensitivity"]
CLINICAL_STATS_CONVERGENCE_SUMMARY = BADRAN_CLINICAL_STATISTICS["convergence_summary"]
CLINICAL_STATS_BO_DIAGNOSTICS = BADRAN_CLINICAL_STATISTICS["bogp_zero_regret_audit"]
CLINICAL_STATS_PUBLICATION_SUMMARY = BADRAN_CLINICAL_STATISTICS["publication_summary"]
CLINICAL_STATS_FILES = BADRAN_CLINICAL_STATISTICS["files"]


Running Linear | Bayesian Optimization | repeat 1/40 | seed=10000 | evaluations=40
Running Linear | TPE Optimization | repeat 1/40 | seed=10000 | evaluations=40
Running Linear | SMAC Optimization | repeat 1/40 | seed=10000 | evaluations=40
Running Linear | Random Search | repeat 1/40 | seed=10000 | evaluations=40
Running Linear | Bayesian Optimization | repeat 2/40 | seed=10001 | evaluations=40
Running Linear | TPE Optimization | repeat 2/40 | seed=10001 | evaluations=40
Running Linear | SMAC Optimization | repeat 2/40 | seed=10001 | evaluations=40
Running Linear | Random Search | repeat 2/40 | seed=10001 | evaluations=40
Running Linear | Bayesian Optimization | repeat 3/40 | seed=10002 | evaluations=40
Running Linear | TPE Optimization | repeat 3/40 | seed=10002 | evaluations=40
Running Linear | SMAC Optimization | repeat 3/40 | seed=10002 | evaluations=40
Running Linear | Random Search | repeat 3/40 | seed=10002 | evaluations=40
Running Linear | Bayesian Optimization | repeat 4/40 | 

/content/clinical_interpolation_optimizer_summary.csv

/content/clinical_interpolation_optimizer_results.csv

/content/clinical_interpolation_optimizer_convergence.csv

Exported 3 Badran optimizer CSV files.
Clinical statistical post-analysis: 40 independent simulation seed blocks per interpolation; 40 matched optimizer blocks per surface.
Seed blocks are algorithmic simulation replicates, not clinical patients. Queries 10 and 20 are checkpoints within each 40-query run, not extra replicates.
Unclipped normalized regret is recomputed from each run's best-so-far noise-free query response, using the anchor maximum 3.929 at (25 Hz, 500 us).
The stored optimizer incumbent differs from the query-history maximum in 817/12800 query rows and 13/320 Q40 runs; inferential endpoints use the query-history maximum.
Both interpolations are exploratory. Query 40 is the endpoint; Holm adjustment covers the two omnibus tests and, separately, all 12 pairwise contrasts.
Paired sign-flip tests assume exchangeable, symmetric within-seed differences under the null. The 10,000-resample bootstrap retains paired seed blocks; its confidence intervals are independent of permuta

,Interpolation Model,Endpoint,Test,N Independent Seed Blocks,N Paired Blocks,N Optimizers,Friedman Statistic,Raw P Value,Kendall W,Holm Adjusted P Value
0,Linear,Query 40 normalized regret (%),Friedman,40,40,4,95.25,0.0,0.79375,0.0
1,PCHIP,Query 40 normalized regret (%),Friedman,40,40,4,89.37,0.0,0.74475,0.0


... (13 outputs left out)

/content/clinical_optimizer_statistics/clinical_optimizer_interpolation_sensitivity.csv

/content/clinical_optimizer_statistics/clinical_optimizer_convergence_summary.csv

/content/clinical_optimizer_statistics/clinical_optimizer_bogp_zero_regret_audit.csv

/content/clinical_optimizer_statistics/clinical_optimizer_convergence.png

In [ ]:
ATANACKOV_RESPONSE_ANCHORS = [
    (10.0, 100.0, 56.1),
    (10.0, 250.0, 57.7),
    (10.0, 500.0, 58.9),
    (25.0, 100.0, 57.2),
    (25.0, 250.0, 58.3),
    (25.0, 500.0, 55.4),
]
ATANACKOV_FREQUENCY_BOUNDS_HZ = (10.0, 25.0)
ATANACKOV_PULSE_WIDTH_BOUNDS_US = (100.0, 500.0)

ATANACKOV_CLINICAL_EXPERIMENT = run_clinical_optimizer_experiment(
    dataset_label="Atanackov",
    anchor_points=ATANACKOV_RESPONSE_ANCHORS,
    frequency_bounds_hz=ATANACKOV_FREQUENCY_BOUNDS_HZ,
    pulse_width_bounds_us=ATANACKOV_PULSE_WIDTH_BOUNDS_US,
    raw_file_prefix="clinical_interpolation_optimizer_atanackov",
    raw_output_directory=".",
    statistics_output_directory=Path("clinical_optimizer_statistics") / "atanackov",
    smac_output_directory="smac3_output",
    num_simulation_trials=CLINICAL_OPTIMIZER_EVALUATIONS_PER_RUN,
    amount_of_seed_repeats=CLINICAL_OPTIMIZER_REPEATS,
    patient_and_machine_startSeed=10000,
)
ATANACKOV_OPTIMIZER_RESULTS = ATANACKOV_CLINICAL_EXPERIMENT["results"]
ATANACKOV_OPTIMIZER_CONVERGENCE = ATANACKOV_CLINICAL_EXPERIMENT["convergence"]
ATANACKOV_OPTIMIZER_SUMMARY = ATANACKOV_CLINICAL_EXPERIMENT["summary"]
ATANACKOV_RESPONSE_FUNCTIONS = ATANACKOV_CLINICAL_EXPERIMENT["response_functions"]
ATANACKOV_OPTIMIZER_CSV_FILES = ATANACKOV_CLINICAL_EXPERIMENT["raw_csv_files"]
ATANACKOV_CLINICAL_STATISTICS = ATANACKOV_CLINICAL_EXPERIMENT["statistics"]
ATANACKOV_LINEAR_RESPONSE_FUNCTION = ATANACKOV_RESPONSE_FUNCTIONS["Linear"]
ATANACKOV_PCHIP_RESPONSE_FUNCTION = ATANACKOV_RESPONSE_FUNCTIONS["PCHIP"]


**WHERE ALL THE FAMILIES ARE RUN**

In [ ]:
baseline_family_outputs = run_and_export_family(
    family=baseline_family,
    num_simulation_trials=40,
    amount_of_patients=40,
    patient_and_machine_startSeed=10000,
    display_links=True,
)

analyze_family_results(baseline_family_outputs)["summary_df"]

In [ ]:
symmetric_family_outputs = run_and_export_family(
    family=symmetric_family,
    num_simulation_trials=40,
    amount_of_patients=40,
    patient_and_machine_startSeed=10000,
    display_links=True,
)

analyze_family_results(symmetric_family_outputs)["summary_df"]

In [ ]:
slow_rise_quick_fall_family_outputs = run_and_export_family(
    family=slow_rise_quick_fall_family,
    num_simulation_trials=40,
    amount_of_patients=40,
    patient_and_machine_startSeed=10000,
    display_links=True,
)

analyze_family_results(slow_rise_quick_fall_family_outputs)["summary_df"]

In [ ]:
three_optima_family_outputs = run_and_export_family(
    family=three_optima_family,
    num_simulation_trials=40,
    amount_of_patients=40,
    patient_and_machine_startSeed=10000,
    display_links=True,
)

analyze_family_results(three_optima_family_outputs)["summary_df"]

In [ ]:
broad_peaks_family_outputs = run_and_export_family(
    family=broad_peaks_family,
    num_simulation_trials=40,
    amount_of_patients=40,
    patient_and_machine_startSeed=10000,
    display_links=True,
)

analyze_family_results(broad_peaks_family_outputs)["summary_df"]

In [ ]:
sharp_peaks_family_outputs = run_and_export_family(
    family=sharp_peaks_family,
    num_simulation_trials=40,
    amount_of_patients=40,
    patient_and_machine_startSeed=10000,
    display_links=True,
)

analyze_family_results(sharp_peaks_family_outputs)["summary_df"]

In [ ]:
hetero_peaks_family_outputs = run_and_export_family(
    family=heteroscedastic_noise_family,
    num_simulation_trials=40,
    amount_of_patients=40,
    patient_and_machine_startSeed=10000,
    display_links=True,
)

analyze_family_results(hetero_peaks_family_outputs)["summary_df"]

In [ ]:
heavy_tailed_noise_family_outputs = run_and_export_family(
    family=heavy_tailed_noise_family,
    num_simulation_trials=40,
    amount_of_patients=40,
    patient_and_machine_startSeed=10000,
    display_links=True,
)

analyze_family_results(heavy_tailed_noise_family_outputs)["summary_df"]

In [ ]:
unique_parameter_interaction_family_outputs = run_and_export_family(
    family=unique_parameter_interaction_family,
    num_simulation_trials=40,
    amount_of_patients=40,
    patient_and_machine_startSeed=10000,
    display_links=True,
)

analyze_family_results(unique_parameter_interaction_family_outputs)["summary_df"]

In [ ]:

# Optional: skip rerunning the optimizer families by reloading previously exported CSVs.
# Only these two files per family are needed:
#   <family>_family_results.csv and <family>_family_convergence.csv

LOADED_FAMILY_SPECS = [
    {"family_name": "quick_rise_slow_fall", "outputs_var": "baseline_family_outputs"},
    {"family_name": "symmetric", "outputs_var": "symmetric_family_outputs"},
    {"family_name": "slow_rise_quick_fall", "outputs_var": "slow_rise_quick_fall_family_outputs"},
    {"family_name": "3optima", "outputs_var": "three_optima_family_outputs"},
    {"family_name": "broad_optima", "outputs_var": "broad_peaks_family_outputs"},
    {"family_name": "sharp_optima", "outputs_var": "sharp_peaks_family_outputs"},
    {"family_name": "heteroscedastic_noise", "outputs_var": "hetero_peaks_family_outputs"},
    {"family_name": "heavy_tailed_noise", "outputs_var": "heavy_tailed_noise_family_outputs"},
    {"family_name": "unique_parameter_interaction", "outputs_var": "unique_parameter_interaction_family_outputs"},
]

USE_INTERACTIVE_UPLOAD = True
FAMILY_CSV_DIR = "."


def upload_family_csvs(upload_dir="uploaded_family_csvs"):
    """Prompt for CSV uploads in Colab/Jupyter widgets; returns the directory holding them."""
    upload_dir = Path(upload_dir)
    upload_dir.mkdir(parents=True, exist_ok=True)

    try:
        from google.colab import files as colab_files
    except ImportError:
        colab_files = None

    if colab_files is not None:
        print("Select every *_family_results.csv and *_family_convergence.csv file to upload.")
        uploaded = colab_files.upload()
        for filename, contents in uploaded.items():
            # Colab appends " (1)", " (2)", etc. to duplicate filenames already
            # present in the runtime; strip that suffix to restore the canonical
            # "<family>_family_results.csv" / "<family>_family_convergence.csv" name.
            clean_name = re.sub(r" \(\d+\)(?=\.[^.]+$)", "", Path(filename).name)
            (upload_dir / clean_name).write_bytes(contents)
        print(f"Saved {len(uploaded)} uploaded file(s) to {upload_dir.resolve()}")
        return upload_dir

    print(
        "Not running in Colab. Copy the exported CSV files into "
        f"{upload_dir.resolve()} (or pass the folder that already contains them "
        "to load_family_outputs_from_csv)."
    )
    return upload_dir


def load_family_outputs_from_csv(
    input_dir=".",
    family_specs=None,
    iteration_budget=40,
    assign_globals=True,
    require_all=True,
):
    """Rebuild each family's outputs dictionary from previously exported CSV files."""
    input_dir = Path(input_dir)
    if not input_dir.is_dir():
        raise FileNotFoundError(f"{input_dir.resolve()} is not a directory.")

    family_specs = family_specs or LOADED_FAMILY_SPECS
    loaded_outputs = {}
    missing_files = []

    for spec in family_specs:
        family_name = spec["family_name"]
        safe_prefix = re.sub(r"[^A-Za-z0-9_-]+", "_", family_name).strip("_")
        results_path = input_dir / f"{safe_prefix}_family_results.csv"
        convergence_path = input_dir / f"{safe_prefix}_family_convergence.csv"

        absent = [path.name for path in (results_path, convergence_path) if not path.exists()]
        if absent:
            missing_files.extend(absent)
            continue

        results_df = pd.read_csv(results_path)
        convergence_df = pd.read_csv(convergence_path)

        if "Surface Family" not in results_df:
            results_df["Surface Family"] = family_name
        if "Surface Family" not in convergence_df:
            convergence_df["Surface Family"] = family_name
        for frame in (results_df, convergence_df):
            if "Iteration Budget" not in frame:
                frame["Iteration Budget"] = iteration_budget

        outputs = {
            "family_name": family_name,
            "results_df": results_df,
            "convergence_df": convergence_df,
            "source_files": {
                "results": str(results_path),
                "convergence": str(convergence_path),
            },
        }
        loaded_outputs[family_name] = outputs
        if assign_globals:
            globals()[spec["outputs_var"]] = outputs

        print(
            f"Loaded {family_name!r}: {len(results_df):,} profile-method rows, "
            f"{len(convergence_df):,} query rows."
        )

    if missing_files:
        message = (
            "Missing exported CSV files: "
            + ", ".join(missing_files)
            + f" (searched {input_dir.resolve()})."
        )
        if require_all:
            raise FileNotFoundError(message)
        print(message)

    return loaded_outputs


# Set USE_INTERACTIVE_UPLOAD = True in Colab to pick the files with a file dialog.
family_csv_dir = upload_family_csvs() if USE_INTERACTIVE_UPLOAD else FAMILY_CSV_DIR
loaded_family_outputs = load_family_outputs_from_csv(family_csv_dir)


In [ ]:
# Reload and analyze one family without rerunning any optimizer.
# Upload exactly 2 CSVs: one results file and one convergence file, from any family.

import io

SINGLE_FAMILY_USE_INTERACTIVE_UPLOAD = True
SINGLE_FAMILY_RESULTS_CSV = "family_results.csv"
SINGLE_FAMILY_CONVERGENCE_CSV = "family_convergence.csv"


def upload_two_family_csvs():
    """Upload exactly 2 files (any names) and split them into results vs convergence by column contents."""
    try:
        from google.colab import files as colab_files
    except ImportError as error:
        raise RuntimeError(
            "Interactive upload requires Google Colab. Set "
            "SINGLE_FAMILY_USE_INTERACTIVE_UPLOAD = False and point "
            "SINGLE_FAMILY_RESULTS_CSV / SINGLE_FAMILY_CONVERGENCE_CSV at local files."
        ) from error

    print("Select exactly 2 files: one results CSV and one convergence CSV.")
    uploaded = colab_files.upload()
    if len(uploaded) != 2:
        raise ValueError(f"Expected exactly 2 files; got {len(uploaded)}.")

    frames = {
        filename: pd.read_csv(io.BytesIO(contents))
        for filename, contents in uploaded.items()
    }

    results_df = convergence_df = None
    for frame in frames.values():
        if "Best Possible Score" in frame.columns:
            results_df = frame
        elif "Query" in frame.columns:
            convergence_df = frame

    if results_df is None or convergence_df is None:
        raise ValueError(
            "Could not tell which upload was the results file and which was "
            "the convergence file. The results file needs a 'Best Possible "
            "Score' column; the convergence file needs a 'Query' column."
        )

    return results_df, convergence_df


def analyze_uploaded_family(results_df, convergence_df, **analyze_kwargs):
    """Run analyze_family_results directly on two already-loaded DataFrames."""
    family_name = str(results_df["Surface Family"].iloc[0])
    outputs = {
        "family_name": family_name,
        "results_df": results_df,
        "convergence_df": convergence_df,
    }
    return outputs, analyze_family_results(outputs, family_name=family_name, **analyze_kwargs)


if SINGLE_FAMILY_USE_INTERACTIVE_UPLOAD:
    single_family_results_df, single_family_convergence_df = upload_two_family_csvs()
else:
    single_family_results_df = pd.read_csv(SINGLE_FAMILY_RESULTS_CSV)
    single_family_convergence_df = pd.read_csv(SINGLE_FAMILY_CONVERGENCE_CSV)

single_family_outputs, single_family_analysis = analyze_uploaded_family(
    single_family_results_df,
    single_family_convergence_df,
)
single_family_analysis["summary_df"]


In [ ]:
sensitivity_methods = [
    "Bayesian Optimization",
    "TPE Optimization",
    "SMAC Optimization",
    "Random Search",
]
sensitivity_regret_tolerance = 0.05
sensitivity_n_permutations = 20000
sensitivity_n_bootstrap = 10000
sensitivity_random_seed = 10001


sensitivity_family_specs = [
    {
        "family_name": "quick_rise_slow_fall",
        "outputs_var": "baseline_family_outputs",
    },
    {
        "family_name": "symmetric",
        "outputs_var": "symmetric_family_outputs",
    },
    {
        "family_name": "slow_rise_quick_fall",
        "outputs_var": "slow_rise_quick_fall_family_outputs",
    },
    {
        "family_name": "3optima",
        "outputs_var": "three_optima_family_outputs",
    },
    {
        "family_name": "broad_optima",
        "outputs_var": "broad_peaks_family_outputs",
    },
    {
        "family_name": "sharp_optima",
        "outputs_var": "sharp_peaks_family_outputs",
    },
    {
        "family_name": "heteroscedastic_noise",
        "outputs_var": "hetero_peaks_family_outputs",
    },
    {
        "family_name": "heavy_tailed_noise",
        "outputs_var": "heavy_tailed_noise_family_outputs",
    },
    {
        "family_name": "unique_parameter_interaction",
        "outputs_var": "unique_parameter_interaction_family_outputs",
    },
]


required_sensitivity_variables = []
for sensitivity_spec in sensitivity_family_specs:
    required_sensitivity_variables.extend(
        variable_name
        for variable_name in [
            sensitivity_spec.get("results_var"),
            sensitivity_spec.get("convergence_var"),
            sensitivity_spec.get("outputs_var"),
        ]
        if variable_name is not None
    )
missing_sensitivity_variables = [
    variable_name
    for variable_name in required_sensitivity_variables
    if variable_name not in globals()
]
if missing_sensitivity_variables:
    raise RuntimeError(
        "Run the formal family simulation cells before the final sensitivity "
        "analysis. Missing variables: "
        + ", ".join(missing_sensitivity_variables)
    )


def _concat_sensitivity_frames(data, label):
    if isinstance(data, pd.DataFrame):
        return data.copy()
    if isinstance(data, (list, tuple)) and data:
        if not all(isinstance(frame, pd.DataFrame) for frame in data):
            raise TypeError(f"Every item in {label} must be a pandas DataFrame.")
        return pd.concat([frame.copy() for frame in data], ignore_index=True)
    raise TypeError(f"{label} must be a non-empty DataFrame or list of DataFrames.")


def _unique_budget_values(frame):
    if frame.empty or "Iteration Budget" not in frame.columns:
        return []
    values = pd.to_numeric(frame["Iteration Budget"], errors="coerce")
    if values.isna().any():
        raise ValueError("Iteration Budget contains missing or non-numeric values.")
    return sorted(values.astype(int).unique().tolist())


def _get_sensitivity_sources(sensitivity_spec):
    if "outputs_var" in sensitivity_spec:
        outputs = globals()[sensitivity_spec["outputs_var"]]
        if not isinstance(outputs, dict):
            raise TypeError(
                f"{sensitivity_spec['outputs_var']} must be the dictionary "
                "returned by run_and_export_family."
            )
        results_data = outputs.get("results_df")
        if results_data is None:
            results_data = outputs.get("family_test")
        convergence_data = outputs.get("convergence_df")
        if convergence_data is None:
            convergence_data = outputs.get("family_convergence")
    else:
        results_data = globals()[sensitivity_spec["results_var"]]
        convergence_data = globals()[sensitivity_spec["convergence_var"]]
    return results_data, convergence_data


def _normalize_sensitivity_family(family_name, results_data, convergence_data):
    results_df = _concat_sensitivity_frames(results_data, f"{family_name} results")
    convergence_df = _concat_sensitivity_frames(
        convergence_data,
        f"{family_name} convergence",
    )

    for frame, label in [
        (results_df, f"{family_name} results"),
        (convergence_df, f"{family_name} convergence"),
    ]:
        if "Surface Family" not in frame.columns:
            frame["Surface Family"] = family_name
        frame["Surface Family"] = frame["Surface Family"].astype(str)
        observed_families = set(frame["Surface Family"].unique())
        if observed_families != {family_name}:
            raise ValueError(
                f"{label} contains {sorted(observed_families)}, expected "
                f"only {family_name!r}."
            )

    if "Seed" not in results_df.columns:
        raise ValueError(f"{family_name} results must contain a Seed column.")
    results_df["Profile ID"] = (
        family_name + "::" + results_df["Seed"].astype(str)
    )

    profile_seed_column = (
        "Profile Seed" if "Profile Seed" in convergence_df.columns else "Seed"
    )
    if profile_seed_column in convergence_df.columns:
        convergence_df["Profile ID"] = (
            family_name + "::" + convergence_df[profile_seed_column].astype(str)
        )
    elif "Profile ID" in convergence_df.columns:
        convergence_df["Profile ID"] = convergence_df["Profile ID"].astype(str)
    else:
        raise ValueError(
            f"{family_name} convergence must contain Profile ID, Profile Seed, or Seed."
        )

    budget_values = sorted(
        set(_unique_budget_values(results_df))
        | set(_unique_budget_values(convergence_df))
    )
    if len(budget_values) != 1:
        raise ValueError(
            f"{family_name} must have exactly one iteration budget; found "
            f"{budget_values}."
        )
    iteration_budget = budget_values[0]
    results_df["Iteration Budget"] = iteration_budget
    convergence_df["Iteration Budget"] = iteration_budget

    required_result_columns = {
        "Method",
        "Best Possible Score",
        "Selected Score (Noise-Free)",
    }
    missing_columns = required_result_columns.difference(results_df.columns)
    if missing_columns:
        raise ValueError(
            f"{family_name} results are missing {sorted(missing_columns)}."
        )
    results_df["Method"] = results_df["Method"].astype(str)
    observed_methods = set(results_df["Method"].unique())
    if observed_methods != set(sensitivity_methods):
        raise ValueError(
            f"{family_name} must contain exactly the four expected methods; "
            f"found {sorted(observed_methods)}."
        )

    for column in ["Best Possible Score", "Selected Score (Noise-Free)"]:
        results_df[column] = pd.to_numeric(results_df[column], errors="coerce")
    if results_df[["Best Possible Score", "Selected Score (Noise-Free)"]].isna().any().any():
        raise ValueError(f"{family_name} contains missing or non-numeric scores.")
    if not np.isfinite(
        results_df[["Best Possible Score", "Selected Score (Noise-Free)"]]
        .to_numpy(dtype=float)
    ).all():
        raise ValueError(f"{family_name} contains non-finite scores.")
    if (results_df["Best Possible Score"] <= 0).any():
        raise ValueError(f"{family_name} has non-positive Best Possible Score values.")

    duplicate_keys = results_df.duplicated(
        subset=["Profile ID", "Iteration Budget", "Method"],
        keep=False,
    )
    if duplicate_keys.any():
        raise ValueError(
            f"{family_name} has duplicate profile-method-budget result rows."
        )
    profile_method_counts = results_df.groupby(
        ["Profile ID", "Iteration Budget"]
    )["Method"].nunique()
    if (profile_method_counts != len(sensitivity_methods)).any():
        raise ValueError(
            f"{family_name} contains incomplete paired profiles; every profile "
            "must have all four methods."
        )

    results_df["Simple Regret"] = (
        results_df["Best Possible Score"]
        - results_df["Selected Score (Noise-Free)"]
    )
    results_df["Normalized Regret"] = (
        results_df["Simple Regret"] / results_df["Best Possible Score"]
    )
    results_df["Normalized Regret (%)"] = 100 * results_df["Normalized Regret"]

    best_possible_by_profile = (
        results_df.drop_duplicates("Profile ID")
        .set_index("Profile ID")["Best Possible Score"]
    )
    _add_incumbent_regret(convergence_df, best_possible_by_profile)

    return results_df, convergence_df


normalized_family_results = []
normalized_family_convergence = []
for sensitivity_spec in sensitivity_family_specs:
    family_results, family_convergence = _normalize_sensitivity_family(
        sensitivity_spec["family_name"],
        *_get_sensitivity_sources(sensitivity_spec),
    )
    normalized_family_results.append(family_results)
    normalized_family_convergence.append(family_convergence)

sensitivity_results_df = pd.concat(normalized_family_results, ignore_index=True)
sensitivity_convergence_df = pd.concat(
    normalized_family_convergence,
    ignore_index=True,
)
sensitivity_results_df["Profile Rank"] = sensitivity_results_df.groupby(
    ["Profile ID", "Iteration Budget"]
)["Normalized Regret"].rank(method="average", ascending=True)


def _bootstrap_mean_ci(values, seed):
    values = np.asarray(values, dtype=float)
    if len(values) < 2:
        mean_value = float(np.mean(values))
        return mean_value, mean_value
    rng = np.random.default_rng(seed)
    bootstrap_means = rng.choice(
        values,
        size=(sensitivity_n_bootstrap, len(values)),
        replace=True,
    ).mean(axis=1)
    return tuple(np.quantile(bootstrap_means, [0.025, 0.975]))


def _paired_sign_flip(values_a, values_b, seed):
    differences = np.asarray(values_a, dtype=float) - np.asarray(values_b, dtype=float)
    observed_difference = float(np.mean(differences))
    rng = np.random.default_rng(seed)
    signs = rng.choice(
        np.array([-1.0, 1.0]),
        size=(sensitivity_n_permutations, len(differences)),
    )
    permuted_means = np.mean(signs * differences, axis=1)
    p_value = (
        1
        + np.count_nonzero(np.abs(permuted_means) >= abs(observed_difference))
    ) / (sensitivity_n_permutations + 1)
    return differences, observed_difference, p_value


def _paired_range_permutation(wide, seed):
    values = wide.to_numpy(dtype=float)
    observed_statistic = float(np.ptp(values.mean(axis=0)))
    rng = np.random.default_rng(seed)
    permuted_statistics = np.empty(sensitivity_n_permutations)
    for permutation_index in range(sensitivity_n_permutations):
        permuted_values = np.array(
            [rng.permutation(profile_values) for profile_values in values]
        )
        permuted_statistics[permutation_index] = np.ptp(
            permuted_values.mean(axis=0)
        )
    p_value = (
        1
        + np.count_nonzero(permuted_statistics >= observed_statistic)
    ) / (sensitivity_n_permutations + 1)
    return observed_statistic, p_value


def _holm_adjust(p_values):
    p_values = np.asarray(p_values, dtype=float)
    adjusted = np.full(len(p_values), np.nan)
    valid = np.isfinite(p_values)
    if valid.any():
        adjusted[valid] = multipletests(p_values[valid], method="holm")[1]
    return adjusted


family_method_summary_rows = []
for group_index, (group_key, group) in enumerate(
    sensitivity_results_df.groupby(
        ["Iteration Budget", "Surface Family", "Method"],
        sort=True,
    )
):
    iteration_budget, family_name, method = group_key
    regret_values = group["Normalized Regret (%)"].to_numpy(dtype=float)
    ci_low, ci_high = _bootstrap_mean_ci(
        regret_values,
        sensitivity_random_seed + group_index,
    )
    family_method_summary_rows.append(
        {
            "Iteration Budget": iteration_budget,
            "Surface Family": family_name,
            "Method": method,
            "N Profiles": len(group),
            "Mean Normalized Regret (%)": np.mean(regret_values),
            "95% CI Low (%)": ci_low,
            "95% CI High (%)": ci_high,
            "Median Normalized Regret (%)": np.median(regret_values),
            "Q25 Normalized Regret (%)": np.quantile(regret_values, 0.25),
            "Q75 Normalized Regret (%)": np.quantile(regret_values, 0.75),
            "P Regret <= 5%": np.mean(
                group["Normalized Regret"] <= sensitivity_regret_tolerance
            ),
            "Mean Profile Rank": group["Profile Rank"].mean(),
            "P Best Method": np.mean(group["Profile Rank"] == 1),
            "Mean Seconds": (
                group["Sec To Compute"].mean()
                if "Sec To Compute" in group.columns
                else np.nan
            ),
        }
    )

sensitivity_family_method_df = pd.DataFrame(family_method_summary_rows)
sensitivity_family_method_df["Family Mean Regret Rank"] = (
    sensitivity_family_method_df.groupby(
        ["Iteration Budget", "Surface Family"]
    )["Mean Normalized Regret (%)"].rank(method="average", ascending=True)
)
sensitivity_family_method_matrix_df = (
    sensitivity_family_method_df.pivot(
        index=["Iteration Budget", "Surface Family"],
        columns="Method",
        values="Mean Normalized Regret (%)",
    )
    .reindex(columns=sensitivity_methods)
    .reset_index()
    .rename_axis(None, axis=1)
)

family_winner_rows = []
for group_key, group in sensitivity_family_method_df.groupby(
    ["Iteration Budget", "Surface Family"],
    sort=True,
):
    iteration_budget, family_name = group_key
    best_value = group["Mean Normalized Regret (%)"].min()
    winners = group[np.isclose(group["Mean Normalized Regret (%)"], best_value)][
        "Method"
    ].tolist()
    family_winner_rows.append(
        {
            "Iteration Budget": iteration_budget,
            "Surface Family": family_name,
            "Best Method(s)": "; ".join(winners),
            "Best Mean Normalized Regret (%)": best_value,
            "N Methods Tied": len(winners),
        }
    )
sensitivity_family_winners_df = pd.DataFrame(family_winner_rows)

overall_summary_rows = []
for budget_index, (iteration_budget, budget_frame) in enumerate(
    sensitivity_results_df.groupby("Iteration Budget", sort=True)
):
    budget_winner_rows = sensitivity_family_winners_df[
        sensitivity_family_winners_df["Iteration Budget"] == iteration_budget
    ]
    winner_counts = {
        method: int(
            budget_winner_rows["Best Method(s)"].map(
                lambda winner_text: method in {
                    winner.strip()
                    for winner in winner_text.split(";")
                }
            ).sum()
        )
        for method in sensitivity_methods
    }
    for method_index, method in enumerate(sensitivity_methods):
        method_frame = budget_frame[budget_frame["Method"] == method]
        profile_regrets = method_frame["Normalized Regret (%)"].to_numpy(dtype=float)
        profile_ci_low, profile_ci_high = _bootstrap_mean_ci(
            profile_regrets,
            sensitivity_random_seed + 1000 + budget_index * 100 + method_index,
        )
        overall_summary_rows.append(
            {
                "Method": method,
                "N Profiles": len(profile_regrets),
                "Mean Regret (%)": np.mean(profile_regrets),
                "95% CI Low (%)": profile_ci_low,
                "95% CI High (%)": profile_ci_high,
                "Median Regret (%)": np.median(profile_regrets),
                "Success Rate (Regret <= 5%)": np.mean(
                    method_frame["Normalized Regret"] <= sensitivity_regret_tolerance
                ),
                "Mean Rank (1-4)": method_frame["Profile Rank"].mean(),
                "Families Won (of 9)": winner_counts[method],
                "Mean Seconds": (
                    method_frame["Sec To Compute"].mean()
                    if "Sec To Compute" in method_frame.columns
                    else np.nan
                ),
            }
        )
sensitivity_overall_summary_df = pd.DataFrame(overall_summary_rows)

sensitivity_global_omnibus_rows = []
sensitivity_global_pairwise_rows = []
sensitivity_family_interaction_rows = []
for budget_index, (iteration_budget, budget_frame) in enumerate(
    sensitivity_results_df.groupby("Iteration Budget", sort=True)
):
    wide = (
        budget_frame.pivot(
            index="Profile ID",
            columns="Method",
            values="Normalized Regret",
        )
        .reindex(columns=sensitivity_methods)
        .dropna()
    )
    if len(wide) < 2:
        continue
    friedman_statistic, friedman_p_value = stats.friedmanchisquare(
        *[wide[method].to_numpy(dtype=float) for method in sensitivity_methods]
    )
    permutation_statistic, permutation_p_value = _paired_range_permutation(
        wide,
        sensitivity_random_seed + 3000 + budget_index,
    )
    sensitivity_global_omnibus_rows.append(
        {
            "Iteration Budget": iteration_budget,
            "N Paired Profiles": len(wide),
            "N Families": budget_frame["Surface Family"].nunique(),
            "Endpoint": "Selected Score (Noise-Free), normalized regret",
            "Friedman Statistic": friedman_statistic,
            "Friedman P Value": friedman_p_value,
            "Kendall's W": friedman_statistic
            / (len(wide) * (len(sensitivity_methods) - 1)),
            "Paired Permutation Range Statistic": permutation_statistic,
            "Paired Permutation P Value": permutation_p_value,
        }
    )

    for pair_index, (method_a, method_b) in enumerate(
        itertools.combinations(sensitivity_methods, 2)
    ):
        differences, mean_difference, p_value = _paired_sign_flip(
            wide[method_a],
            wide[method_b],
            sensitivity_random_seed
            + 4000
            + budget_index * 100
            + pair_index,
        )
        ci_low, ci_high = _bootstrap_mean_ci(
            100 * differences,
            sensitivity_random_seed
            + 5000
            + budget_index * 100
            + pair_index,
        )
        difference_sd = np.std(differences, ddof=1)
        sensitivity_global_pairwise_rows.append(
            {
                "Iteration Budget": iteration_budget,
                "Method A": method_a,
                "Method B": method_b,
                "N Paired Profiles": len(differences),
                "Mean Regret Difference (A - B) (%)": 100 * mean_difference,
                "95% CI Low (%)": ci_low,
                "95% CI High (%)": ci_high,
                "Paired Effect Size dz": (
                    mean_difference / difference_sd
                    if difference_sd > 0
                    else np.nan
                ),
                "P Method A Lower Regret": np.mean(differences < 0),
                "Paired Sign-Flip P Value": p_value,
            }
        )

    for pair_index, (method_a, method_b) in enumerate(
        itertools.combinations(sensitivity_methods, 2)
    ):
        family_difference_arrays = []
        family_mean_differences = []
        for _, family_frame in budget_frame.groupby("Surface Family"):
            family_wide = (
                family_frame.pivot(
                    index="Profile ID",
                    columns="Method",
                    values="Normalized Regret",
                )
                .reindex(columns=sensitivity_methods)
                .dropna()
            )
            if len(family_wide) > 0:
                family_differences = (
                    family_wide[method_a].to_numpy(dtype=float)
                    - family_wide[method_b].to_numpy(dtype=float)
                )
                family_difference_arrays.append(family_differences)
                family_mean_differences.append(np.mean(family_differences))
        if len(family_difference_arrays) >= 2:
            try:
                heterogeneity_statistic, heterogeneity_p_value = stats.kruskal(
                    *family_difference_arrays
                )
            except ValueError:
                heterogeneity_statistic, heterogeneity_p_value = np.nan, np.nan
        else:
            heterogeneity_statistic, heterogeneity_p_value = np.nan, np.nan
        sensitivity_family_interaction_rows.append(
            {
                "Iteration Budget": iteration_budget,
                "Method A": method_a,
                "Method B": method_b,
                "N Families": len(family_difference_arrays),
                "Mean Family Contrast (A - B) (%)": 100
                * np.mean(family_mean_differences),
                "SD Family Contrast (%)": (
                    100 * np.std(family_mean_differences, ddof=1)
                    if len(family_mean_differences) > 1
                    else np.nan
                ),
                "Min Family Contrast (%)": 100 * np.min(family_mean_differences),
                "Max Family Contrast (%)": 100 * np.max(family_mean_differences),
                "Kruskal-Wallis H": heterogeneity_statistic,
                "Family Contrast Heterogeneity P Value": heterogeneity_p_value,
            }
        )

sensitivity_global_omnibus_df = pd.DataFrame(sensitivity_global_omnibus_rows)
sensitivity_global_pairwise_df = pd.DataFrame(sensitivity_global_pairwise_rows)
if not sensitivity_global_pairwise_df.empty:
    sensitivity_global_pairwise_df["Holm Adjusted P Value"] = np.nan
    for iteration_budget in sensitivity_global_pairwise_df["Iteration Budget"].unique():
        budget_mask = (
            sensitivity_global_pairwise_df["Iteration Budget"] == iteration_budget
        )
        sensitivity_global_pairwise_df.loc[budget_mask, "Holm Adjusted P Value"] = _holm_adjust(
            sensitivity_global_pairwise_df.loc[
                budget_mask,
                "Paired Sign-Flip P Value",
            ]
        )
sensitivity_family_interaction_df = pd.DataFrame(
    sensitivity_family_interaction_rows
)
if not sensitivity_family_interaction_df.empty:
    sensitivity_family_interaction_df["Holm Adjusted P Value"] = np.nan
    for iteration_budget in sensitivity_family_interaction_df["Iteration Budget"].unique():
        budget_mask = sensitivity_family_interaction_df["Iteration Budget"] == iteration_budget
        sensitivity_family_interaction_df.loc[
            budget_mask,
            "Holm Adjusted P Value",
        ] = _holm_adjust(
            sensitivity_family_interaction_df.loc[
                budget_mask,
                "Family Contrast Heterogeneity P Value",
            ]
        )

sensitivity_convergence_summary_df = pd.DataFrame()
sensitivity_convergence_compact_df = pd.DataFrame()
trajectory_score_column = INCUMBENT_REGRET_COLUMN
if trajectory_score_column in sensitivity_convergence_df.columns and "Query" in sensitivity_convergence_df.columns:
    trajectory_group_columns = [
        "Surface Family",
        "Method",
        "Iteration Budget",
        "Query",
    ]
    sensitivity_convergence_summary_df = (
        sensitivity_convergence_df.groupby(trajectory_group_columns)[
            trajectory_score_column
        ]
        .agg(
            Median="median",
            Q25=lambda values: values.quantile(0.25),
            Q75=lambda values: values.quantile(0.75),
        )
        .reset_index()
    )
    profile_counts = (
        sensitivity_convergence_df.groupby(trajectory_group_columns)["Profile ID"]
        .nunique()
        .reset_index(name="N Profiles")
    )
    sensitivity_convergence_summary_df = sensitivity_convergence_summary_df.merge(
        profile_counts,
        on=trajectory_group_columns,
        how="left",
    )
    expected_profiles_by_family = sensitivity_convergence_df.groupby(
        "Surface Family"
    )["Profile ID"].nunique()
    sensitivity_convergence_summary_df["N Profiles Expected"] = (
        sensitivity_convergence_summary_df["Surface Family"].map(
            expected_profiles_by_family
        )
    )
    sensitivity_convergence_summary_df["Complete Coverage"] = (
        sensitivity_convergence_summary_df["N Profiles"]
        == sensitivity_convergence_summary_df["N Profiles Expected"]
    )
    compact_rows = []
    compact_group_columns = [
        "Surface Family",
        "Method",
        "Iteration Budget",
    ]
    for group_key, group in sensitivity_convergence_summary_df.groupby(
        compact_group_columns,
        sort=True,
    ):
        ordered_group = group.sort_values("Query")
        ordered_group = ordered_group.dropna(subset=["Median"])
        if ordered_group.empty:
            continue
        first_row = ordered_group.iloc[0]
        final_row = ordered_group.iloc[-1]
        compact_rows.append(
            {
                **dict(zip(compact_group_columns, group_key)),
                "N Profiles": int(final_row["N Profiles"]),
                "N Queries": len(ordered_group),
                "All Queries Fully Covered": bool(
                    ordered_group["Complete Coverage"].all()
                ),
                "First Query": first_row["Query"],
                "Final Query": final_row["Query"],
                "First Median Normalized Regret (%)": first_row["Median"],
                "Final Median Normalized Regret (%)": final_row["Median"],
                "Final Q25 (%)": final_row["Q25"],
                "Final Q75 (%)": final_row["Q75"],
                "Median Regret Improvement (%)": (
                    first_row["Median"] - final_row["Median"]
                ),
            }
        )
    sensitivity_convergence_compact_df = pd.DataFrame(compact_rows)

sensitivity_crn_audit_df = pd.DataFrame()
sensitivity_crn_summary_df = pd.DataFrame()
crn_columns = {
    "Profile ID",
    "Iteration Budget",
    "Query",
    "Method",
    "Evaluation Seed",
}
if crn_columns.issubset(sensitivity_convergence_df.columns):
    sensitivity_crn_audit_df = (
        sensitivity_convergence_df.groupby(
            ["Surface Family", "Profile ID", "Iteration Budget", "Query"]
        )
        .agg(
            N_Methods=("Method", "nunique"),
            N_Evaluation_Seeds=("Evaluation Seed", "nunique"),
        )
        .reset_index()
    )
    sensitivity_crn_audit_df["Uses Common Random Number"] = (
        (sensitivity_crn_audit_df["N_Methods"] == len(sensitivity_methods))
        & (sensitivity_crn_audit_df["N_Evaluation_Seeds"] == 1)
    )
    sensitivity_crn_summary_df = (
        sensitivity_crn_audit_df.groupby(["Surface Family", "Iteration Budget"])[
            "Uses Common Random Number"
        ]
        .agg(
            **{
                "N Tested Groups": "size",
                "N Passed Groups": "sum",
                "N Failed Groups": lambda values: (~values).sum(),
            }
        )
        .reset_index()
    )
    sensitivity_crn_summary_df["Overall Status"] = np.where(
        sensitivity_crn_summary_df["N Failed Groups"] == 0,
        "PASS",
        "FAIL",
    )

sensitivity_validation_rows = []
for family_name, family_frame in sensitivity_results_df.groupby(
    "Surface Family",
    sort=True,
):
    family_budgets = sorted(family_frame["Iteration Budget"].unique().tolist())
    sensitivity_validation_rows.append(
        {
            "Surface Family": family_name,
            "N Profiles": family_frame["Profile ID"].nunique(),
            "Iteration Budgets": ", ".join(str(value) for value in family_budgets),
            "N Methods": family_frame["Method"].nunique(),
            "Complete Paired Profiles": (
                family_frame.groupby(["Profile ID", "Iteration Budget"])["Method"]
                .nunique()
                .eq(len(sensitivity_methods))
                .all()
            ),
            "CRN Audit Status": (
                ", ".join(
                    sensitivity_crn_summary_df.loc[
                        sensitivity_crn_summary_df["Surface Family"] == family_name,
                        "Overall Status",
                    ].astype(str)
                )
                if not sensitivity_crn_summary_df.empty
                else "NOT AVAILABLE"
            ),
        }
    )
sensitivity_validation_df = pd.DataFrame(sensitivity_validation_rows)

sensitivity_export_frames = {
    "combined_sensitivity_profile_results.csv": sensitivity_results_df,
    "combined_sensitivity_convergence.csv": sensitivity_convergence_df,
    "sensitivity_family_method_summary.csv": sensitivity_family_method_df,
    "sensitivity_family_method_matrix.csv": sensitivity_family_method_matrix_df,
    "sensitivity_family_winners.csv": sensitivity_family_winners_df,
    "sensitivity_overall_summary.csv": sensitivity_overall_summary_df,
    "sensitivity_global_omnibus.csv": sensitivity_global_omnibus_df,
    "sensitivity_global_pairwise.csv": sensitivity_global_pairwise_df,
    "sensitivity_family_interaction.csv": sensitivity_family_interaction_df,
    "sensitivity_convergence_summary.csv": sensitivity_convergence_summary_df,
    "sensitivity_convergence_compact.csv": sensitivity_convergence_compact_df,
    "sensitivity_crn_audit.csv": sensitivity_crn_audit_df,
    "sensitivity_crn_summary.csv": sensitivity_crn_summary_df,
}
sensitivity_output_files = {}
for filename, frame in sensitivity_export_frames.items():
    output_path = Path(filename)
    frame.to_csv(output_path, index=False)
    sensitivity_output_files[filename] = str(output_path)

print(
    "Primary endpoint: Selected Score (Noise-Free), expressed as normalized "
    "regret; lower regret is better."
)
print(
    "Convergence endpoint: noise-free regret of the incumbent, i.e. the "
    "parameters the optimizer would recommend if stopped at that query."
)
if sensitivity_results_df["Iteration Budget"].nunique() != 1:
    print(
        "The formal families use different iteration budgets. Results are "
        "reported separately by budget; rerun all formal families at one common "
        "budget before reporting a single pooled conclusion."
    )

print("\n--- Sensitivity input validation ---")
display(sensitivity_validation_df)
print("\n--- Family-by-method effectiveness table ---")
display(sensitivity_family_method_df)
print("\n--- Mean normalized regret matrix (%) ---")
display(sensitivity_family_method_matrix_df)
print("\n--- Best method by family ---")
display(sensitivity_family_winners_df)
print("\n--- Overall optimizer performance (all 360 profiles) ---")
display(sensitivity_overall_summary_df)
print("\n--- Global paired omnibus tests, stratified by iteration budget ---")
display(sensitivity_global_omnibus_df)
print("\n--- Global paired post-hoc tests with Holm correction ---")
display(sensitivity_global_pairwise_df)
print("\n--- Family-by-method contrast heterogeneity ---")
display(sensitivity_family_interaction_df)
if not sensitivity_convergence_compact_df.empty:
    print("\n--- Compact cross-family convergence summary ---")
    display(sensitivity_convergence_compact_df)
if not sensitivity_crn_summary_df.empty:
    print("\n--- Compact cross-family common-random-number audit ---")
    display(sensitivity_crn_summary_df)
print(f"\nExported {len(sensitivity_output_files)} sensitivity-analysis CSV files.")

sensitivity_analysis_outputs = {
    "profile_results_df": sensitivity_results_df,
    "convergence_df": sensitivity_convergence_df,
    "family_method_summary_df": sensitivity_family_method_df,
    "family_method_matrix_df": sensitivity_family_method_matrix_df,
    "family_winners_df": sensitivity_family_winners_df,
    "overall_summary_df": sensitivity_overall_summary_df,
    "global_omnibus_df": sensitivity_global_omnibus_df,
    "global_pairwise_df": sensitivity_global_pairwise_df,
    "family_interaction_df": sensitivity_family_interaction_df,
    "convergence_summary_df": sensitivity_convergence_summary_df,
    "convergence_compact_df": sensitivity_convergence_compact_df,
    "crn_audit_df": sensitivity_crn_audit_df,
    "crn_summary_df": sensitivity_crn_summary_df,
    "validation_df": sensitivity_validation_df,
    "trajectory_score_column": trajectory_score_column,
    "files": sensitivity_output_files,
}

checkpoint_queries = [20, 40]
checkpoint_columns = [
    "Iteration Budget",
    "Surface Family",
    "Query",
    "N Profiles",
    "BOGP",
    "TPE",
    "SMAC",
    "Random Search",
]

if sensitivity_convergence_summary_df.empty:
    sensitivity_convergence_checkpoint_df = pd.DataFrame(
        columns=checkpoint_columns
    )
else:
    checkpoint_rows = sensitivity_convergence_summary_df.loc[
        sensitivity_convergence_summary_df["Query"].isin(checkpoint_queries)
        & sensitivity_convergence_summary_df["Complete Coverage"]
    ].copy()

    if checkpoint_rows.empty:
        sensitivity_convergence_checkpoint_df = pd.DataFrame(
            columns=checkpoint_columns
        )
    else:
        checkpoint_rows["Median [Q1, Q3] (%)"] = checkpoint_rows.apply(
            lambda row: (
                f"{row['Median']:.1f} "
                f"[{row['Q25']:.1f}, {row['Q75']:.1f}]"
            ),
            axis=1,
        )
        sensitivity_convergence_checkpoint_df = (
            checkpoint_rows.pivot(
                index=[
                    "Iteration Budget",
                    "Surface Family",
                    "Query",
                    "N Profiles",
                ],
                columns="Method",
                values="Median [Q1, Q3] (%)",
            )
            .reindex(columns=sensitivity_methods)
            .rename(
                columns={
                    "Bayesian Optimization": "BOGP",
                    "TPE Optimization": "TPE",
                    "SMAC Optimization": "SMAC",
                }
            )
            .reset_index()
            .reindex(columns=checkpoint_columns)
            .sort_values(
                ["Iteration Budget", "Surface Family", "Query"]
            )
            .reset_index(drop=True)
        )

print(
    f"Checkpoint metric: {trajectory_score_column}; values are median [Q1, Q3] "
    "normalized regret (%). Only fully covered family-method-query groups are included."
)
display(sensitivity_convergence_checkpoint_df)

checkpoint_output_path = Path("sensitivity_convergence_checkpoints.csv")
sensitivity_convergence_checkpoint_df.to_csv(checkpoint_output_path, index=False)
sensitivity_output_files[checkpoint_output_path.name] = str(checkpoint_output_path)
sensitivity_analysis_outputs["convergence_checkpoints_df"] = (
    sensitivity_convergence_checkpoint_df
)
sensitivity_analysis_outputs["files"] = sensitivity_output_files
print(f"Exported checkpoint table to {checkpoint_output_path.resolve()}")


In [ ]:
seed_block_required_inputs = [
    "sensitivity_results_df",
    "sensitivity_methods",
    "sensitivity_n_permutations",
    "sensitivity_n_bootstrap",
    "sensitivity_random_seed",
    "sensitivity_regret_tolerance",
    "_paired_range_permutation",
    "_paired_sign_flip",
    "_bootstrap_mean_ci",
    "_holm_adjust",
]
missing_seed_block_inputs = [
    name for name in seed_block_required_inputs if name not in globals()
]
if missing_seed_block_inputs:
    raise RuntimeError(
        "Run the preceding sensitivity-analysis cell after loading all nine family "
        "result sets. Missing seed-block inputs: "
        + ", ".join(missing_seed_block_inputs)
    )

print("SEED-BLOCK CORRECTION: The row-wise pooled omnibus/pairwise tests and pooled CIs printed above treat repeated family seeds as independent. They are superseded by the seed-block results below.")


def _paired_friedman_with_complete_tie_fallback(wide):
    values = wide.to_numpy(dtype=float)
    if np.allclose(values, values[:, :1], rtol=1e-12, atol=1e-12):
        return 0.0, 1.0
    result = stats.friedmanchisquare(
        *[wide[column].to_numpy(dtype=float) for column in wide.columns]
    )
    return float(result.statistic), float(result.pvalue)


seed_block_results = sensitivity_results_df.copy()
seed_block_seed = pd.to_numeric(seed_block_results["Seed"], errors="coerce")
if seed_block_seed.isna().any() or not np.equal(seed_block_seed, np.floor(seed_block_seed)).all():
    raise ValueError("Seed-block analysis requires integer-convertible Seed values.")
seed_block_results["Seed"] = seed_block_seed.astype("int64")

seed_block_duplicate_rows = seed_block_results.duplicated(
    subset=["Iteration Budget", "Surface Family", "Seed", "Method"],
    keep=False,
)
if seed_block_duplicate_rows.any():
    raise ValueError("Seed-block analysis found duplicate family-seed-method rows.")

seed_block_pairwise_rows = []
seed_block_omnibus_rows = []
seed_block_overall_rows = []
seed_block_family_contrast_rows = []
seed_block_result_frames = []

for budget_index, (iteration_budget, budget_data) in enumerate(
    seed_block_results.groupby("Iteration Budget", sort=True)
):
    family_names = sorted(budget_data["Surface Family"].astype(str).unique())
    n_families = len(family_names)
    if n_families < 1:
        continue

    family_profile_wide = budget_data.pivot(
        index=["Seed", "Surface Family"],
        columns="Method",
        values="Normalized Regret",
    ).reindex(columns=sensitivity_methods)

    complete_family_profiles = family_profile_wide.dropna()
    family_counts_by_seed = complete_family_profiles.groupby(level="Seed").size()
    complete_seed_ids = family_counts_by_seed[
        family_counts_by_seed == n_families
    ].index
    complete_family_profiles = complete_family_profiles.loc[
        complete_family_profiles.index.get_level_values("Seed").isin(complete_seed_ids)
    ]
    seed_block_wide = complete_family_profiles.groupby(level="Seed", sort=True).mean()
    seed_block_wide = seed_block_wide.reindex(columns=sensitivity_methods)

    if len(seed_block_wide) < 2:
        raise ValueError(
            f"Budget {iteration_budget} has fewer than two complete seed blocks "
            f"across its {n_families} families."
        )

    n_seed_blocks = len(seed_block_wide)
    n_candidate_seeds = budget_data["Seed"].nunique()
    n_excluded_seed_blocks = n_candidate_seeds - n_seed_blocks
    complete_budget_data = budget_data[
        budget_data["Seed"].isin(complete_seed_ids)
    ].copy()

    seed_block_long = seed_block_wide.rename_axis("Seed").reset_index()
    seed_block_long.insert(0, "N Families", n_families)
    seed_block_long.insert(0, "Iteration Budget", iteration_budget)
    seed_block_result_frames.append(seed_block_long)

    friedman_statistic, friedman_p_value = _paired_friedman_with_complete_tie_fallback(
        seed_block_wide
    )
    permutation_statistic, permutation_p_value = _paired_range_permutation(
        seed_block_wide,
        sensitivity_random_seed + 3000 + budget_index,
    )
    seed_block_omnibus_rows.append(
        {
            "Iteration Budget": iteration_budget,
            "N Paired Seed Blocks": n_seed_blocks,
            "N Families": n_families,
            "N Excluded Seed Blocks": n_excluded_seed_blocks,
            "Family Weighting": "Equal",
            "Endpoint": "Mean normalized noise-free regret across families",
            "Friedman Statistic": friedman_statistic,
            "Friedman P Value": friedman_p_value,
            "Kendall's W": friedman_statistic
            / (n_seed_blocks * (len(sensitivity_methods) - 1)),
            "Paired Permutation Range Statistic": permutation_statistic,
            "Paired Permutation P Value": permutation_p_value,
        }
    )

    method_family_means = complete_budget_data.groupby(
        ["Surface Family", "Method"]
    )["Normalized Regret (%)"].mean()
    winner_counts = {method: 0 for method in sensitivity_methods}
    for family_name in family_names:
        family_means = method_family_means.loc[family_name]
        best_mean = family_means.min()
        for method in sensitivity_methods:
            if np.isclose(family_means[method], best_mean):
                winner_counts[method] += 1

    for method_index, method in enumerate(sensitivity_methods):
        block_regrets = 100 * seed_block_wide[method].to_numpy(dtype=float)
        ci_low, ci_high = _bootstrap_mean_ci(
            block_regrets,
            sensitivity_random_seed + 1000 + budget_index * 100 + method_index,
        )
        method_data = complete_budget_data[complete_budget_data["Method"] == method]
        seed_block_overall_rows.append(
            {
                "Iteration Budget": iteration_budget,
                "Method": method,
                "N Profiles": len(method_data),
                "N Seed Blocks": n_seed_blocks,
                "N Families": n_families,
                "Mean Regret (%)": np.mean(block_regrets),
                "95% Seed-Block CI Low (%)": ci_low,
                "95% Seed-Block CI High (%)": ci_high,
                "Median Family-Profile Regret (%)": method_data[
                    "Normalized Regret (%)"
                ].median(),
                "Success Rate (Regret <= 5%)": np.mean(
                    method_data["Normalized Regret"] <= sensitivity_regret_tolerance
                ),
                "Mean Profile Rank (descriptive)": method_data["Profile Rank"].mean(),
                "Families Won (of N Families)": winner_counts[method],
                "Mean Seconds": (
                    method_data["Sec To Compute"].mean()
                    if "Sec To Compute" in method_data.columns
                    else np.nan
                ),
            }
        )

    for pair_index, (method_a, method_b) in enumerate(
        itertools.combinations(sensitivity_methods, 2)
    ):
        differences, mean_difference, p_value = _paired_sign_flip(
            seed_block_wide[method_a],
            seed_block_wide[method_b],
            sensitivity_random_seed + 4000 + budget_index * 100 + pair_index,
        )
        ci_low, ci_high = _bootstrap_mean_ci(
            100 * differences,
            sensitivity_random_seed + 5000 + budget_index * 100 + pair_index,
        )
        difference_sd = np.std(differences, ddof=1)
        seed_block_pairwise_rows.append(
            {
                "Iteration Budget": iteration_budget,
                "Method A": method_a,
                "Method B": method_b,
                "N Paired Seed Blocks": n_seed_blocks,
                "N Families": n_families,
                "N Excluded Seed Blocks": n_excluded_seed_blocks,
                "Mean Regret Difference (A - B) (%)": 100 * mean_difference,
                "95% Seed-Block CI Low (%)": ci_low,
                "95% Seed-Block CI High (%)": ci_high,
                "Paired Effect Size dz": (
                    mean_difference / difference_sd if difference_sd > 0 else np.nan
                ),
                "Proportion Method A Lower Regret": np.mean(differences < 0),
                "Paired Sign-Flip P Value": p_value,
            }
        )

        family_contrast_wide = (
            complete_family_profiles[method_a] - complete_family_profiles[method_b]
        ).unstack(level="Surface Family").reindex(columns=family_names).dropna()
        if len(family_names) >= 3 and len(family_contrast_wide) >= 2:
            family_friedman_statistic, family_friedman_p_value = (
                _paired_friedman_with_complete_tie_fallback(family_contrast_wide)
            )
        else:
            family_friedman_statistic, family_friedman_p_value = np.nan, np.nan
        family_mean_contrasts = family_contrast_wide.mean(axis=0)
        seed_block_family_contrast_rows.append(
            {
                "Iteration Budget": iteration_budget,
                "Method A": method_a,
                "Method B": method_b,
                "N Paired Seed Blocks": len(family_contrast_wide),
                "N Families": len(family_names),
                "Mean Family Contrast (A - B) (%)": 100 * family_mean_contrasts.mean(),
                "SD Family Contrast (%)": 100 * family_mean_contrasts.std(ddof=1),
                "Min Family Contrast (%)": 100 * family_mean_contrasts.min(),
                "Max Family Contrast (%)": 100 * family_mean_contrasts.max(),
                "Repeated-Measures Friedman Statistic": family_friedman_statistic,
                "Family Contrast Heterogeneity P Value": family_friedman_p_value,
            }
        )

sensitivity_seed_block_results_df = pd.concat(seed_block_result_frames, ignore_index=True)
sensitivity_seed_block_overall_summary_df = pd.DataFrame(seed_block_overall_rows)
sensitivity_seed_block_omnibus_df = pd.DataFrame(seed_block_omnibus_rows)
sensitivity_seed_block_pairwise_df = pd.DataFrame(seed_block_pairwise_rows)
sensitivity_seed_block_family_contrast_df = pd.DataFrame(seed_block_family_contrast_rows)

for test_df, raw_column in [
    (sensitivity_seed_block_pairwise_df, "Paired Sign-Flip P Value"),
    (sensitivity_seed_block_family_contrast_df, "Family Contrast Heterogeneity P Value"),
]:
    if not test_df.empty:
        test_df["Holm Adjusted P Value"] = np.nan
        for iteration_budget in test_df["Iteration Budget"].unique():
            budget_mask = test_df["Iteration Budget"] == iteration_budget
            test_df.loc[budget_mask, "Holm Adjusted P Value"] = _holm_adjust(
                test_df.loc[budget_mask, raw_column]
            )

# Replace row-wise pooled inferential outputs so later notebook code uses the corrected results.
sensitivity_overall_summary_df = sensitivity_seed_block_overall_summary_df
sensitivity_global_omnibus_df = sensitivity_seed_block_omnibus_df
sensitivity_global_pairwise_df = sensitivity_seed_block_pairwise_df
sensitivity_family_interaction_df = sensitivity_seed_block_family_contrast_df

print("\n--- Seed-block overall optimizer performance ---")
print("Means and confidence intervals use equal family weighting and resample seed blocks.")
display(sensitivity_seed_block_overall_summary_df)
print("\n--- Seed-block paired omnibus tests by iteration budget ---")
display(sensitivity_seed_block_omnibus_df)
print("\n--- Seed-block paired optimizer comparisons with Holm correction ---")
display(sensitivity_seed_block_pairwise_df)
print("\n--- Family contrasts tested as repeated measures across seed blocks ---")
display(sensitivity_seed_block_family_contrast_df)

seed_block_export_frames = {
    "sensitivity_seed_block_results.csv": sensitivity_seed_block_results_df,
    "sensitivity_seed_block_overall_summary.csv": sensitivity_seed_block_overall_summary_df,
    "sensitivity_seed_block_omnibus.csv": sensitivity_seed_block_omnibus_df,
    "sensitivity_seed_block_pairwise.csv": sensitivity_seed_block_pairwise_df,
    "sensitivity_seed_block_family_contrasts.csv": sensitivity_seed_block_family_contrast_df,
    "sensitivity_overall_summary.csv": sensitivity_seed_block_overall_summary_df,
    "sensitivity_global_omnibus.csv": sensitivity_seed_block_omnibus_df,
    "sensitivity_global_pairwise.csv": sensitivity_seed_block_pairwise_df,
    "sensitivity_family_interaction.csv": sensitivity_seed_block_family_contrast_df,
}
for filename, frame in seed_block_export_frames.items():
    frame.to_csv(Path(filename), index=False)

if "sensitivity_output_files" in globals():
    sensitivity_output_files.update(
        {filename: str(Path(filename)) for filename in seed_block_export_frames}
    )
if "sensitivity_export_frames" in globals():
    sensitivity_export_frames.update(seed_block_export_frames)
if "sensitivity_analysis_outputs" in globals():
    sensitivity_analysis_outputs.update(
        {
            "overall_summary_df": sensitivity_seed_block_overall_summary_df,
            "global_omnibus_df": sensitivity_seed_block_omnibus_df,
            "global_pairwise_df": sensitivity_seed_block_pairwise_df,
            "family_interaction_df": sensitivity_seed_block_family_contrast_df,
            "seed_block_results_df": sensitivity_seed_block_results_df,
            "seed_block_overall_summary_df": sensitivity_seed_block_overall_summary_df,
            "seed_block_omnibus_df": sensitivity_seed_block_omnibus_df,
            "seed_block_pairwise_df": sensitivity_seed_block_pairwise_df,
            "seed_block_family_contrast_df": sensitivity_seed_block_family_contrast_df,
            "files": sensitivity_output_files,
        }
    )

print("\nSeed-block CSV outputs written. The earlier row-wise pooled inferential tables are superseded.")